# Playground Series S6E11 -- Credit Card Fraud | Unified Shallow-Tree Stack

One notebook, one pipeline, rebuilt from scratch around the LB 0.9539 shallow-tree stack
(`credit-card-fraud-lb-0-9539-shallow-tree-stack.ipynb`), with the feature-engineering pipeline and
pseudo-labeling idea of `credit-card-fraud-detection-version-22.ipynb` integrated as first-class members.

## What is inside

* **EDA + metric**: fraud rate, single-feature AUC ranking, train-vs-test adversarial check, ROC AUC intuition.
* **Leakage-free features**: in-fold `TargetEncoder` encodings, digit keys, original-data lookups, count encodings,
  triple crosses, and the v22 `FeatureEngineer` (group statistics, frequency/count maps, smoothed target + value
  encodings, pair and three-way interactions) feeding three extra stack members.
* **Member zoo**: shallow LightGBM/XGBoost/CatBoost, tuned variants, the goodpjw2008 Part 3/5 m-estimate recipe,
  full-data refits, joint GLM + GLM-boosted trees, RealMLP and TabM (pytabkit), spline GLM, GLM on encodings,
  seed bags, kNN (GPU), plus the three v22 engineered-feature GBMs.
* **Public prediction banks**: every published OOF/test bank the loaders can find, plus a generic `/kaggle/input`
  sweep for newer `oof/test` pairs and `submission*.csv` files.
* **Blending**: rank-logit hill climb vs logistic stacker, the wide pool at C=0.01, the two-stack mix, the public
  ladder rebuilds (0.95370 / 0.95388 / 0.95417) when mounted, and a final consensus blend putting the strongest
  public file beside the CV-backed wide stack.
* **Optional**: conservative pseudo-label refinement (off by default; cannot be validated on the leaderboard).

All OOF predictions are aligned on `StratifiedKFold(5, shuffle=True, random_state=42)`; every encoding is fitted
inside the fold's training rows only.

In [ ]:
%pip install -q pytabkit

In [ ]:
import os, time, warnings, torch
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.linear_model import LogisticRegression
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=0.9)

KAGGLE = Path('/kaggle/input').exists()
if KAGGLE:
    DATA = next(Path('/kaggle/input').rglob('train.csv')).parent
else:
    DATA = next((p for p in (Path('.'), Path('..')) if (p / 'train.csv').exists()), Path('.'))
OUT = Path('/kaggle/working') if KAGGLE else Path('.')
OUT.mkdir(exist_ok=True)
THREADS = os.cpu_count() or 4
GPU = torch.cuda.is_available()
TARGET = 'is_fraud'
CAT_ITERS, MLP_EPOCHS = 3000, 16
VERSION = 'unified-v1'
T_START = time.time()
BUDGET_H = 9.0

train = pd.read_csv(DATA / 'train.csv'); test = pd.read_csv(DATA / 'test.csv')
print(f'train {train.shape}, test {test.shape}, cores {THREADS}, GPU {GPU}, OUT {OUT}')

## 0. The problem and the data

One row is one card transaction. `is_fraud` is 1 when it was fraudulent.
The data is synthetic, generated from the [Credit Card Fraud Detection 2026](https://www.kaggle.com/datasets/uditjain13/credit-card-fraud-detection-2026) dataset;
distributions are close to, but not the same as, the original.


In [ ]:

desc = {
 'amount_usd': 'transaction amount, USD', 'hours_since_last_txn': 'hours since the card was last used',
 'txn_count_last_24h': 'transactions on the card in the last 24 h', 'distance_from_home_km': 'distance from the cardholder\'s home',
 'card_age_months': 'age of the card', 'customer_age': 'age of the cardholder', 'account_balance_usd': 'account balance, USD',
 'cvv_retry_count': 'failed CVV attempts', 'velocity_score': 'spending-velocity score', 'time_of_day_hour': 'hour, 0-23',
 'day_of_week': 'day, 0-6', 'merchant_risk_score': 'merchant risk score, 0-100', 'prior_disputes': 'earlier disputes on the account',
 'merchant_category': 'merchant type', 'card_type': 'card network', 'auth_method': 'how the payment was authenticated',
 'channel': 'POS, online, in-app, contactless, ATM', 'device_type': 'device used', 'is_foreign_transaction': 'foreign transaction',
 'is_new_merchant': 'first time at this merchant', 'used_vpn': 'VPN detected', 'ip_country_mismatch': 'IP country differs from card country',
 'billing_shipping_mismatch': 'billing and shipping address differ', 'is_ai_generated_scam_attempt': 'flagged as an AI-generated scam',
}
cols = pd.DataFrame([dict(column=c, dtype=str(train[c].dtype), unique=train[c].nunique(), example=train[c].iloc[0], meaning=desc[c])
                     for c in train.columns if c in desc])
display(cols)
display(train.head(5))


## 1. The metric: ROC AUC

The score is the area under the ROC curve between your probabilities and the true labels.
The useful way to read it: **AUC is the chance that a random fraud gets a higher score than a random legitimate transaction.**
Only the *order* of your predictions matters. Three consequences, each shown below with a computed example:


In [ ]:

rng = np.random.default_rng(0)
y_toy = (rng.random(10_000) < 0.0167).astype(int)                 # same fraud rate as train
good = y_toy * 1.0 + rng.normal(0, 0.6, y_toy.size)             # a decent ranking
print(f"1. all-zero predictions: accuracy {np.mean(y_toy == 0):.3f}, AUC {roc_auc_score(y_toy, np.zeros(y_toy.size)):.3f}  (98% accurate, worthless)")
print(f"2. any monotone transform keeps AUC: raw {roc_auc_score(y_toy, good):.4f}, exp() {roc_auc_score(y_toy, np.exp(good)):.4f}, "
      f"rank {roc_auc_score(y_toy, pd.Series(good).rank()):.4f}, 0.8 + x/1e6 {roc_auc_score(y_toy, 0.8 + good / 1e6):.4f}")
print(f"3. calibration is free: scores squeezed into [0.80, 0.81] (useless as probabilities) score the same AUC")
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for s, lab in [(good, 'decent model'), (rng.normal(size=y_toy.size), 'random')]:
    f, t, _ = roc_curve(y_toy, s); ax[0].plot(f, t, label=f'{lab} (AUC {roc_auc_score(y_toy, s):.3f})')
ax[0].plot([0, 1], [0, 1], 'k:', lw=1); ax[0].set_xlabel('false positive rate'); ax[0].set_ylabel('true positive rate'); ax[0].legend(); ax[0].set_title('ROC curve')
ax[1].hist(good[y_toy == 0], bins=60, alpha=.6, density=True, label='legit'); ax[1].hist(good[y_toy == 1], bins=60, alpha=.6, density=True, label='fraud')
ax[1].set_title('AUC = P(fraud score > legit score)'); ax[1].legend(); plt.tight_layout(); plt.show()


What this means for the competition:

- Class imbalance (1.7% fraud) does not need resampling or class weights for AUC; a model that ranks well is enough.
- Blending is easy: average **ranks** (or probabilities) of several models; scale differences between models do not matter.
- Small AUC gains are real gains here: with 300k test rows, a difference of 0.0005 is many correctly re-ordered pairs, but check it in CV, not on the public board alone.


## 2. EDA

### Class balance and the numeric features by class

In [ ]:

y = train[TARGET].astype(int)
print(f"fraud rate {y.mean():.4f} ({y.sum():,} of {len(y):,}); missing values: {int(train.isna().sum().sum())} in train, {int(test.isna().sum().sum())} in test")
num = [c for c in train.columns if train[c].dtype.kind in 'if' and c not in ('id', TARGET)]
cat = train.select_dtypes('object').columns.tolist()
boo = train.select_dtypes('bool').columns.tolist()
auc1 = {c: roc_auc_score(y, train[c]) for c in num}
auc1 = pd.Series({c: max(a, 1 - a) for c, a in auc1.items()}).sort_values(ascending=False)
print('single-feature AUC (how well each column alone orders fraud):'); print(auc1.round(3).to_string())
fig, axs = plt.subplots(3, 5, figsize=(16, 8)); axs = axs.ravel()
for ax, c in zip(axs, auc1.index):
    lo, hi = train[c].quantile([0.005, 0.995])
    for lab, g in train.groupby(TARGET):
        ax.hist(g[c].clip(lo, hi), bins=40, density=True, alpha=.55, label='fraud' if lab else 'legit')
    ax.set_title(f'{c}  (AUC {auc1[c]:.3f})', fontsize=8)
for ax in axs[len(auc1):]: ax.axis('off')
axs[0].legend(); plt.tight_layout(); plt.show()


### Fraud rate by category and by flag

In [ ]:

fig, axs = plt.subplots(2, 3, figsize=(16, 7)); axs = axs.ravel()
for ax, c in zip(axs, cat):
    r = train.groupby(c)[TARGET].agg(['mean', 'size']).sort_values('mean')
    ax.barh(r.index.astype(str), r['mean']); ax.axvline(y.mean(), color='k', ls=':', lw=1); ax.set_title(f'fraud rate by {c}', fontsize=9)
r = pd.DataFrame({c: [train.loc[train[c], TARGET].mean(), train.loc[~train[c], TARGET].mean(), train[c].mean()] for c in boo},
                 index=['rate if True', 'rate if False', 'share True']).T.sort_values('rate if True')
axs[-1].barh(r.index, r['rate if True'], label='flag True'); axs[-1].barh(r.index, r['rate if False'], label='flag False', alpha=.6)
axs[-1].axvline(y.mean(), color='k', ls=':', lw=1); axs[-1].set_title('fraud rate by flag', fontsize=9); axs[-1].legend(fontsize=7)
plt.tight_layout(); plt.show()
display(r.round(4))


### Train vs test: are they the same distribution?

An adversarial check: train a quick model to tell train rows from test rows.
AUC near 0.5 means the two sets look alike, so CV on train should predict the leaderboard.


In [ ]:

import lightgbm as lgb
feats = [c for c in train.columns if c not in ('id', TARGET)]
def prep(df):
    d = df[feats].copy()
    d[boo] = d[boo].astype(int)
    for c in cat: d[c] = pd.Categorical(d[c], categories=sorted(train[c].unique()))
    return d
X, XT = prep(train), prep(test)
adv = pd.concat([X, XT]); adv_y = np.r_[np.zeros(len(X)), np.ones(len(XT))]
a_oof = np.zeros(len(adv))
for tr_i, va_i in StratifiedKFold(3, shuffle=True, random_state=0).split(adv, adv_y):
    m = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.1, num_leaves=31, n_jobs=THREADS, verbose=-1).fit(adv.iloc[tr_i], adv_y[tr_i])
    a_oof[va_i] = m.predict_proba(adv.iloc[va_i])[:, 1]
print(f'adversarial AUC train vs test: {roc_auc_score(adv_y, a_oof):.4f}')


## 3. The ensemble

Gradient-boosted trees (LightGBM, XGBoost, CatBoost), a spline GLM and two neural nets (RealMLP, TabM), all trained on the same 5 stratified folds,
so every model's out-of-fold predictions line up row for row and can be blended.

- **LightGBM / XGBoost / CatBoost**: gradient-boosted trees, native categorical handling, early stopping on the fold's validation AUC.
- **RealMLP** ([pytabkit](https://github.com/dholzmueller/pytabkit), Holzmüller et al., NeurIPS 2024): an MLP with tuned defaults for tables
  (robust scaling, periodic numeric embeddings, a learned per-feature scale). It makes different errors from trees, which is what a blend needs.
- **Target encoding inside each fold** (sklearn `TargetEncoder`, cross-fitted, so a row never sees its own label): every column with >= 100 rows per value
  (numeric ones per exact value), integer-rounded keys for the four continuous columns, and pairs among the ten columns with the strongest per-value signal.
  Not all 276 pairs: on three earlier playgrounds all-pair crosses measured -0.0036, -0.0004 and +0.0002.
- **Shallow trees** (4-leaf LightGBM twice, depth-1 and depth-2 XGBoost, depth-3 CatBoost, on the encoded and the raw inputs): the signal is close to additive; the top public notebooks measured +0.0019 over 31 leaves, and in v5 the blend kept only the shallow members.
- **TabM** (a batch ensemble of 32 MLPs, GPU) next to RealMLP.
- **A spline GLM** (logistic regression on splines and one-hots; design from sidhaarthshree and goodpjw2008).
- **RealMLP with tuned settings** (yekenot's S6E10 parameters, used by ravi20076 here), on the encoded inputs.
- **Decimal digits**: the first decimal digit of five numeric columns; the synthesiser leaves a pattern there that range splits cannot express.
- **The original 20k rows as a lookup**: per-value fraud rates in the original data and a small model trained only on it, as features (not as extra rows).
- LightGBM runs twice (raw / engineered); the other models use the engineered set only if it gains at least 0.0002 OOF AUC, so the notebook decides.


### The pipeline in one picture

Red marks the one place a label can leak: anything fitted on labels (the target encoder) must be fitted inside the fold's training rows.


In [ ]:

fig, ax = plt.subplots(figsize=(13, 4.2)); ax.axis('off'); ax.set_xlim(0, 13); ax.set_ylim(0, 4.2)
def box(x, y, w, h, text, fc='#eef3fb', ec='#4a6fa5'):
    ax.add_patch(plt.Rectangle((x, y), w, h, fc=fc, ec=ec, lw=1.5)); ax.text(x + w / 2, y + h / 2, text.replace('|', chr(10)), ha='center', va='center', fontsize=9)
def arrow(x0, y0, x1, y1): ax.annotate('', (x1, y1), (x0, y0), arrowprops=dict(arrowstyle='->', lw=1.3, color='#555'))
box(0.1, 1.6, 1.7, 1.0, 'train 700k|+ test 300k')
box(2.3, 2.7, 2.0, 1.0, 'fold k: training|rows (4/5)')
box(2.3, 0.5, 2.0, 1.0, 'fold k: validation|rows (1/5)')
box(4.9, 2.7, 2.2, 1.0, 'target encoder|FITTED HERE ONLY', fc='#fdecea', ec='#c0392b')
box(4.9, 0.5, 2.2, 1.0, 'encoder applied|(no labels used)')
box(7.7, 1.6, 2.0, 1.0, 'every model|(trees, GLM, MLPs)')
box(10.2, 2.7, 2.6, 1.0, 'out-of-fold predictions|-> blend weights')
box(10.2, 0.5, 2.6, 1.0, 'test predictions|(mean of 5 folds)')
for a in [(1.8, 2.3, 2.3, 3.2), (1.8, 1.9, 2.3, 1.0), (4.3, 3.2, 4.9, 3.2), (4.3, 1.0, 4.9, 1.0), (6.0, 2.7, 6.0, 1.5),
          (7.1, 3.2, 7.7, 2.4), (7.1, 1.0, 7.7, 1.8), (9.7, 2.3, 10.2, 3.2), (9.7, 1.9, 10.2, 1.0)]:
    arrow(*a)
ax.set_title('Every model sees the same 5 folds; the encoder never sees a validation label', fontsize=11); plt.show()


### Features: targeted encodings, decimal digits, the original data as a lookup

In [ ]:
from itertools import combinations
from sklearn.preprocessing import TargetEncoder
T_START = time.time()
both = pd.concat([train[feats], test[feats]], ignore_index=True)
cont = ['amount_usd', 'hours_since_last_txn', 'distance_from_home_km', 'account_balance_usd']
P = {}
for c in feats:                                      # every column, each exact value a category; the encoder's smoothing handles rare values
    codes, uniq = pd.factorize(both[c].astype(str))
    if len(uniq) <= len(both) // 2:
        P[c] = codes.astype('int32')
for c in cont:                                       # the continuous ones pooled on rounded keys instead, two levels
    P[f'{c}_int'] = pd.factorize(np.round(both[c]).astype(int))[0].astype('int32')
    P[f'{c}_r10'] = pd.factorize(np.round(both[c] / 10).astype(int))[0].astype('int32')
for c in ['merchant_risk_score', 'velocity_score']:  # exact value is a single key above; the integer level pools neighbours
    P[f'{c}_int'] = pd.factorize(np.round(both[c]).astype(int))[0].astype('int32')
SINGLES = list(P)
# pairs only among the ten columns with the strongest per-level signal (the cron EDA notebook's signal ratios)
TOP = ['cvv_retry_count', 'ip_country_mismatch', 'is_foreign_transaction', 'is_new_merchant', 'billing_shipping_mismatch',
       'auth_method', 'used_vpn', 'is_ai_generated_scam_attempt', 'merchant_category', 'txn_count_last_24h']
for a, b in combinations(TOP, 2):
    P[f'{a}__{b}'] = pd.factorize(both[a].astype(str) + '_' + both[b].astype(str))[0].astype('int32')
P = pd.DataFrame(P); ENC = list(P.columns)
PT, P = P.iloc[len(train):].reset_index(drop=True), P.iloc[:len(train)].reset_index(drop=True)
print(f'target-encoded: {len(SINGLES)} single keys + {len(ENC) - len(SINGLES)} pairs among the top 10')

# first decimal digit: the synthesiser leaves a pattern there that no range split can express (a modulo condition)
DIGIT_COLS = ['merchant_risk_score', 'velocity_score', 'hours_since_last_txn', 'amount_usd', 'distance_from_home_km']
digit = lambda d: pd.DataFrame({f'd1_{c}': (np.floor(np.abs(d[c].values) * 10 + 1e-6) % 10).astype('int8') for c in DIGIT_COLS})
D, DT = digit(train), digit(test)
display(pd.DataFrame({c: train.groupby(D[f'd1_{c}'].values)[TARGET].mean() for c in DIGIT_COLS}).round(4).T)

# the original 20k rows as a lookup, not as extra training rows: per-value fraud rates there, and a model trained only on them
orig_path = next(iter(Path('/kaggle/input').rglob('credit_card_fraud_2026.csv')), None) if KAGGLE else Path('../../data/orig/credit_card_fraud_2026.csv')
G, GT = pd.DataFrame(index=train.index), pd.DataFrame(index=test.index)
if orig_path is not None and Path(orig_path).exists():
    try:
        orig = pd.read_csv(orig_path)
        need = set(feats + [TARGET])
        if len(orig) == 0 or not need.issubset(orig.columns):
            raise ValueError(f'rows={len(orig)}, missing={sorted(need - set(orig.columns))[:5]}')
        oy = orig[TARGET].astype(int); prior = float(oy.mean())
        if not np.isfinite(prior):
            raise ValueError('no finite fraud rate in original labels')
        print(f'original data: {len(orig):,} rows, fraud rate {prior:.4f}')
        for c in ['merchant_risk_score', 'velocity_score', 'customer_age'] + cat + boo:
            m = orig.groupby(orig[c].astype(str))[TARGET].agg(['sum', 'size'])
            rate = (m['sum'] + 20 * prior) / (m['size'] + 20)            # smoothed toward the overall rate
            G[f'orig_{c}'] = train[c].astype(str).map(rate).fillna(prior).astype('float32').values
            GT[f'orig_{c}'] = test[c].astype(str).map(rate).fillna(prior).astype('float32').values
        om = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, n_jobs=THREADS, verbose=-1).fit(prep(orig), oy)
        G['orig_model'] = om.predict_proba(X)[:, 1].astype('float32'); GT['orig_model'] = om.predict_proba(XT)[:, 1].astype('float32')
    except Exception as e:
        print(f'original data attached but unusable: {type(e).__name__}: {str(e)[:120]}')
        G, GT = pd.DataFrame(index=train.index), pd.DataFrame(index=test.index)
else:
    print('original data not attached: no lookup features')
print(f'lookup features from the original data: {G.shape[1]}')

FOLDS = list(StratifiedKFold(5, shuffle=True, random_state=42).split(X, y))
CACHE = {}
def frames(k, mode):
    """(train, valid, test) inputs for fold k. The encodings are fitted on the fold's training rows only (cross-fitted 5 ways
    inside them), so no row sees its own label; computed once per fold and reused by every model."""
    tr_i, va_i = FOLDS[k]
    Xa, Xb, Xt = X.iloc[tr_i].reset_index(drop=True), X.iloc[va_i].reset_index(drop=True), XT.reset_index(drop=True)
    if mode == 'raw':
        return Xa, Xb, Xt
    if k not in CACHE:
        enc = TargetEncoder(target_type='binary', cv=5, random_state=0)
        te = lambda arr: pd.DataFrame(arr, columns=['te_' + c for c in ENC]).astype('float32')
        a = te(enc.fit_transform(P.iloc[tr_i], y.iloc[tr_i])); b = te(enc.transform(P.iloc[va_i])); t = te(enc.transform(PT))
        g = lambda df, idx: df.iloc[idx].reset_index(drop=True) if idx is not None else df.reset_index(drop=True)
        CACHE[k] = (pd.concat([g(D, tr_i), g(G, tr_i), a], axis=1), pd.concat([g(D, va_i), g(G, va_i), b], axis=1),
                    pd.concat([g(DT, None), g(GT, None), t], axis=1))
    a, b, t = CACHE[k]
    return pd.concat([Xa, a], axis=1), pd.concat([Xb, b], axis=1), pd.concat([Xt, t], axis=1)

### Why the encoder must be fitted inside the fold: a leak you can measure

Target encoding replaces a value by the fraud rate of the rows that share it. If those rates are computed on **all** rows before the CV split,
a validation row's own label is inside its feature. The CV score then measures memory, not prediction.

Below, 40k rows are put aside as a stand-in for the test set and never used to fit anything. On the other 200k rows the same encodings
are built two ways, and each is scored twice: by 3-fold CV, and on the untouched 40k.
If CV and hold-out disagree, the CV is lying.


In [ ]:

rs = np.random.RandomState(0); idx = rs.permutation(len(X))[:240_000]
hold, dev = idx[:len(idx) // 6], idx[len(idx) // 6:]          # 40k / 200k
cols = SINGLES
lgb4 = lambda: lgb.LGBMClassifier(n_estimators=600, learning_rate=0.1, num_leaves=4, n_jobs=THREADS, verbose=-1)
leak = {}
for how in ['fitted on all rows first (leaky)', 'fitted inside each fold (honest)']:
    leaky = how.endswith('(leaky)'); cv = []
    for tr_i, va_i in StratifiedKFold(3, shuffle=True, random_state=0).split(dev, y.iloc[dev]):
        a_i, b_i = dev[tr_i], dev[va_i]
        enc = TargetEncoder(target_type='binary', random_state=0)
        if leaky:                                          # fit + transform on the same rows: no cross-fitting, validation labels included
            enc.fit(P.iloc[dev][cols], y.iloc[dev]); A, B = enc.transform(P.iloc[a_i][cols]), enc.transform(P.iloc[b_i][cols])
        else:                                              # fit_transform cross-fits inside the training rows; validation only transformed
            A = enc.fit_transform(P.iloc[a_i][cols], y.iloc[a_i]); B = enc.transform(P.iloc[b_i][cols])
        cv.append(roc_auc_score(y.iloc[b_i], lgb4().fit(A, y.iloc[a_i]).predict_proba(B)[:, 1]))
    enc = TargetEncoder(target_type='binary', random_state=0)
    A = (enc.fit(P.iloc[dev][cols], y.iloc[dev]).transform(P.iloc[dev][cols]) if leaky else enc.fit_transform(P.iloc[dev][cols], y.iloc[dev]))
    hold_auc = roc_auc_score(y.iloc[hold], lgb4().fit(A, y.iloc[dev]).predict_proba(enc.transform(P.iloc[hold][cols]))[:, 1])
    leak[how] = (np.mean(cv), hold_auc)
L = pd.DataFrame(leak, index=['3-fold CV AUC', 'hold-out AUC']).T; display(L.round(5))
ax = L.plot.bar(figsize=(8, 3.5), rot=0, color=['#9bb7d4', '#4a6fa5'], ylim=(L.values.min() - 0.01, min(1.0, L.values.max() + 0.01)))
ax.set_title('Leaky encoding: CV promises more than the hold-out delivers'); ax.set_ylabel('AUC'); plt.show()


In [ ]:

oof, pred, secs = {}, {}, {}
BUDGET_H = 9.0                                         # a model that would start after this is skipped (Kaggle stops at 12 h)

# Every member's out-of-fold and test predictions are published as the dataset alexchilton/s6e11-oof-predictions, keyed by a
# fingerprint of (member name, input set, the fitting code and the settings it reads, a checksum of fold 0's exact inputs, the folds).
# A member whose fingerprint is already there is loaded instead of trained; change its code, its settings or the features
# and the fingerprint changes, so it retrains. Attach the dataset to your own notebook to blend these members without training them.
import hashlib, json, types
PRED_CACHE = next((f.parent for f in Path('/kaggle/input').rglob('oof_manifest.json')), None) if KAGGLE else None
MANIFEST = json.loads((PRED_CACHE / 'oof_manifest.json').read_text()) if PRED_CACHE else {}
print('prediction cache:', f'{PRED_CACHE} ({len(MANIFEST)} members)' if PRED_CACHE else 'none attached, every member trains')
# Member fingerprints for the prediction cache. Standard library only: the same text runs inside the notebook (make_nb.py
# pastes it into the cache cell) and offline in tools/rekey_oof.py, so a fingerprint can be computed from a notebook's source
# without running it. A fingerprint changes when the member's code (syntax tree, so not comments or spacing), a notebook
# function it calls, a simple setting it reads (CAT_ITERS, BEST, ...), or the feature cell changes. Machine facts
# (THREADS, GPU) do not count; only UPPERCASE notebook names count as settings. ast.dump differs between Python minor versions, so offline use runs the notebook's Python.
import ast, hashlib, textwrap

FP_SKIP = {'THREADS', 'GPU'}


def _tree(src):
    return ast.dump(ast.parse(textwrap.dedent(src)))


def _names(src):
    t = ast.parse(textwrap.dedent(src))
    return {n.id for n in ast.walk(t) if isinstance(n, ast.Name)} | {n.attr for n in ast.walk(t) if isinstance(n, ast.Attribute)}


def fp_compute(name, mode, fn_name, funcs, settings, feature_src):
    """funcs: notebook function name -> its def source; settings: notebook name -> simple value; feature_src: the cell that defines frames()."""
    order, todo = [], [fn_name]
    while todo:
        f = todo.pop()
        if f in order:
            continue
        order.append(f); todo += sorted((_names(funcs[f]) & set(funcs)) - set(order))
    used = set().union(*(_names(funcs[f]) for f in order))
    sets = sorted((k, repr(settings[k])) for k in used & set(settings) if k.isupper() and k not in FP_SKIP)   # constants only, not loop leftovers
    parts = [name, mode, _tree(feature_src)] + [_tree(funcs[f]) for f in sorted(order)] + [repr(sets)]
    return hashlib.sha1('\n'.join(parts).encode()).hexdigest()[:16]


def fp_feature_cell(cells):
    """The cell that builds the features and defines frames() (not the cache cell, which mentions it too)."""
    return next(c for c in cells if 'def frames(' in c and 'fp_compute' not in c)
import inspect
def fingerprint(name, fit_predict, mode):
    g = globals()
    funcs = {k: inspect.getsource(v) for k, v in g.items()
             if isinstance(v, types.FunctionType) and v.__module__ == '__main__' and v.__name__ != '<lambda>' and not k.startswith('_')}
    settings = {k: v for k, v in g.items() if type(v) in (int, float, str, bool, dict) and not k.startswith('_')}
    try:
        feature_src = fp_feature_cell(get_ipython().user_ns['In'])          # the cell as it was run
    except Exception:                                                     # outside IPython: no cache, a name-based key
        return 'nofp_' + ''.join(ch if ch.isalnum() else '_' for ch in f'{name}_{mode}')
    return fp_compute(name, mode, fit_predict.__name__, funcs, settings, feature_src)

def run(name, fit_predict, mode=None):
    mode = mode or MODE
    if (time.time() - T_START) / 3600 > BUDGET_H:
        print(f'{name}: skipped, {BUDGET_H} h budget used'); return
    fp = fingerprint(name, fit_predict, mode)
    if fp in MANIFEST and (PRED_CACHE / f'oof_{fp}.npy').exists():
        oof[name], pred[name] = np.load(PRED_CACHE / f'oof_{fp}.npy'), np.load(PRED_CACHE / f'test_{fp}.npy'); secs[name] = MANIFEST[fp]['minutes'] * 60
        _publish(fp, name, mode)
        print(f'{name}: OOF AUC {roc_auc_score(y, oof[name]):.5f}  (loaded from the dataset, trained in {MANIFEST[fp]["version"]})'); return
    t0 = time.time(); o = np.zeros(len(X)); p = np.zeros(len(XT))
    for k, (tr_i, va_i) in enumerate(FOLDS):
        Xa, Xb, Xt = frames(k, mode)
        vo, tp = fit_predict(Xa, y.iloc[tr_i].values, Xb, y.iloc[va_i].values, Xt)
        o[va_i] = vo; p += tp / len(FOLDS)
        print(f'  {name} fold {k}: AUC {roc_auc_score(y.iloc[va_i], vo):.5f}  ({Xa.shape[1]} inputs)', flush=True)
    oof[name], pred[name], secs[name] = o, p, time.time() - t0
    _publish(fp, name, mode, trained=True)
    print(f'{name}: OOF AUC {roc_auc_score(y, o):.5f}  ({secs[name] / 60:.1f} min)')

OUT_MANIFEST = {}
def _publish(fp, name, mode, trained=False):         # this run's output holds every member, trained or loaded, for the next dataset version
    np.save(OUT / f'oof_{fp}.npy', oof[name]); np.save(OUT / f'test_{fp}.npy', pred[name])
    OUT_MANIFEST[fp] = dict(name=name, inputs=mode, oof_auc=round(float(roc_auc_score(y, oof[name])), 5), minutes=round(secs[name] / 60, 1),
                            version=VERSION if trained else MANIFEST[fp]['version'])
    (OUT / 'oof_manifest.json').write_text(json.dumps(OUT_MANIFEST, indent=1))


In [ ]:

def lgbm(Xa, ya, Xb, yb, Xt):
    m = lgb.LGBMClassifier(n_estimators=4000, learning_rate=0.03, num_leaves=63, min_child_samples=50, subsample=0.8, subsample_freq=1,
                           colsample_bytree=0.5, reg_lambda=1.0, n_jobs=THREADS, verbose=-1)
    m.fit(Xa, ya, eval_set=[(Xb, yb)], eval_metric='auc', callbacks=[lgb.early_stopping(200, verbose=False)])
    return m.predict_proba(Xb)[:, 1], m.predict_proba(Xt)[:, 1]
run('LightGBM raw', lgbm, 'raw')
run('LightGBM +features', lgbm, 'fe')
# the other three models use the engineered set only if it beat raw by more than run-to-run noise
gain = roc_auc_score(y, oof['LightGBM +features']) - roc_auc_score(y, oof['LightGBM raw'])
MODE = 'fe' if gain >= 0.0002 else 'raw'
print(f'engineered features: {gain:+.5f} OOF AUC for LightGBM -> the other models use {MODE!r}')

def lgbm_shallow(Xa, ya, Xb, yb, Xt):                 # 4 leaves: the signal is close to additive, so tiny trees generalise better
    m = lgb.LGBMClassifier(n_estimators=20000, learning_rate=0.05, num_leaves=4, min_child_samples=100, subsample=0.8, subsample_freq=1,
                           colsample_bytree=0.5, reg_lambda=1.0, n_jobs=THREADS, verbose=-1)
    m.fit(Xa, ya, eval_set=[(Xb, yb)], eval_metric='auc', callbacks=[lgb.early_stopping(300, verbose=False)])
    return m.predict_proba(Xb)[:, 1], m.predict_proba(Xt)[:, 1]
run('LightGBM 4 leaves', lgbm_shallow, 'fe')
run('LightGBM 4 leaves raw', lgbm_shallow, 'raw')
def lgbm_shallow2(Xa, ya, Xb, yb, Xt):                # the same tree size, another seed and a sparser column sample
    m = lgb.LGBMClassifier(n_estimators=20000, learning_rate=0.05, num_leaves=4, min_child_samples=200, subsample=0.7, subsample_freq=1,
                           colsample_bytree=0.3, reg_lambda=5.0, random_state=1, n_jobs=THREADS, verbose=-1)
    m.fit(Xa, ya, eval_set=[(Xb, yb)], eval_metric='auc', callbacks=[lgb.early_stopping(300, verbose=False)])
    return m.predict_proba(Xb)[:, 1], m.predict_proba(Xt)[:, 1]
run('LightGBM 4 leaves b', lgbm_shallow2, 'fe')


### How big should a tree be? Measure it

One fold, the same inputs, only `num_leaves` changes. A deep tree can express interactions between many columns;
if the score falls as trees grow, those interactions are mostly noise here, and the target is close to a sum of per-column effects.


In [ ]:

Xa0, Xb0, _ = frames(0, MODE); ya0, yb0 = y.iloc[FOLDS[0][0]].values, y.iloc[FOLDS[0][1]].values
sizes, curve = [2, 4, 8, 16, 32, 64], []
for L in sizes:
    m = lgb.LGBMClassifier(n_estimators=20000, learning_rate=0.1, num_leaves=L, min_child_samples=100, subsample=0.8, subsample_freq=1,
                           colsample_bytree=0.5, n_jobs=THREADS, verbose=-1)
    m.fit(Xa0, ya0, eval_set=[(Xb0, yb0)], eval_metric='auc', callbacks=[lgb.early_stopping(200, verbose=False)])
    curve.append(m.best_score_['valid_0']['auc'])
fig, ax = plt.subplots(figsize=(7, 3.5)); ax.plot(sizes, curve, 'o-', color='#4a6fa5'); ax.set_xscale('log', base=2); ax.set_xticks(sizes, sizes)
ax.set_xlabel('leaves per tree'); ax.set_ylabel('validation AUC (fold 0)'); ax.set_title('Validation AUC against tree size'); plt.show()
print(dict(zip(sizes, np.round(curve, 5))))


### What the model learned, one column at a time

Trees of depth 1 use one column per tree, so the model is a sum of one curve per column (a *shape function*).
XGBoost's per-feature contributions (`pred_contribs`) then are exactly those curves. Steps and bumps are what the encodings and digits try to capture.


In [ ]:

import xgboost as xgb
Xa1, Xb1, _ = frames(0, 'raw')
m1 = xgb.XGBClassifier(n_estimators=5000, learning_rate=0.1, max_depth=1, tree_method='hist', device='cuda' if GPU else 'cpu',
                       enable_categorical=True, eval_metric='auc', early_stopping_rounds=200, n_jobs=THREADS)
m1.fit(Xa1, ya0, eval_set=[(Xb1, yb0)], verbose=False); m1.set_params(device='cpu')
smp = Xb1.sample(min(20000, len(Xb1)), random_state=0)
contrib = pd.DataFrame(m1.get_booster().predict(xgb.DMatrix(smp, enable_categorical=True), pred_contribs=True)[:, :-1], columns=smp.columns)
top = contrib.abs().mean().sort_values(ascending=False).index[:12]
fig, axes = plt.subplots(3, 4, figsize=(15, 9))
for ax, c in zip(axes.ravel(), top):
    if smp[c].dtype.name == 'category' or smp[c].nunique() <= 12:
        g = contrib[c].groupby(smp[c].astype(str).values).mean().sort_values(); ax.barh(g.index, g.values, color='#4a6fa5')
    else:
        ax.scatter(smp[c], contrib[c], s=2, alpha=0.3, color='#4a6fa5')
    ax.set_title(c, fontsize=9); ax.axhline(0, color='#999', lw=0.8)
fig.suptitle('Shape functions of a depth-1 XGBoost (log-odds contribution per column)', y=1.0); plt.tight_layout(); plt.show()
print(f'depth-1 XGBoost: {m1.best_iteration + 1} trees, fold-0 validation AUC {roc_auc_score(yb0, m1.predict_proba(Xb1)[:, 1]):.5f}')


### Tuning without fooling yourself: small Optuna searches

Each search uses one fold and a 250k-row sample of its training rows, with a fast learning rate, so 30 trials cost minutes, not hours.
The found settings then train a new member at the usual learning rate on all five folds.
Tuning on fold 0's validation rows makes that fold slightly optimistic; the other four folds are untouched, which keeps the OOF score honest enough.
The plots show what a search is for: whether more trials still help (history), and which settings matter at all (importance).

A search is cached in the dataset like the members, keyed by its objective's code, so the next version reloads its trials instead of searching again.

**LightGBM was searched in v8** (30 trials, 28 minutes on 4 CPU cores): the best setting is num_leaves = 2, two-leaf trees, even smaller than our hand-picked 4.
At the full learning rate that member scores OOF 0.94839, the same as the hand-picked 4-leaf LightGBM (0.94838), and took 62 minutes.
The hill climb gives it weight 0 and the stacker 0.142; the leaderboard moved from 0.94982 (v6) to 0.94981 (v8).
So the search confirmed the shallow-tree finding and found nothing beyond it. Its settings are pinned below so it is not searched again.


In [ ]:

import optuna
from optuna.visualization.matplotlib import plot_optimization_history, plot_param_importances
optuna.logging.set_verbosity(optuna.logging.WARNING)
sub_i = np.random.RandomState(0).permutation(len(Xa0))[:250_000]
def cached_search(tag, objective, n_trials):
    """An Optuna study for one member, loaded from the dataset when the same objective was searched before, else run and saved."""
    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=0))
    key = hashlib.sha1((tag + _tree(inspect.getsource(objective)) + repr(len(sub_i))).encode()).hexdigest()[:12]
    saved = PRED_CACHE / f'optuna_{key}.json' if PRED_CACHE else None
    if saved is not None and saved.exists():
        D = optuna.distributions
        for t in json.loads(saved.read_text()):
            study.add_trial(optuna.trial.create_trial(params=t['params'], value=t['value'],
                                                      distributions={k: D.json_to_distribution(v) for k, v in t['dists'].items()}))
        print(f'{tag}: search loaded from the dataset, {len(study.trials)} trials')
    else:
        log_trial = lambda s, t: print(f'  {tag} trial {t.number:2d}: AUC {t.value:.5f}  ({t.duration.total_seconds():.0f} s)', flush=True)
        study.optimize(objective, n_trials=n_trials, callbacks=[log_trial])   # a fixed trial count, so the result does not depend on the machine
        (OUT / f'optuna_{key}.json').write_text(json.dumps([dict(params=t.params, value=t.value, seconds=t.duration.total_seconds(),
            dists={k: optuna.distributions.distribution_to_json(d) for k, d in t.distributions.items()}) for t in study.trials]))
    print(f'{tag}: best fold-0 AUC {study.best_value:.5f}', study.best_params)
    plot_optimization_history(study, target_name='fold-0 AUC'); plt.title(f'{tag}: search history'); plt.show()
    plot_param_importances(study); plt.title(f'{tag}: which settings matter'); plt.show()
    return study

# LightGBM: searched in v8 (30 trials, best fold-0 AUC 0.94568); the result is pinned, so the member below loads from the dataset
BEST = {'num_leaves': 2, 'min_child_samples': 455, 'colsample_bytree': 0.48621093026380613, 'subsample': 0.979109441929957, 'reg_lambda': 22.84848036668211}
def lgbm_tuned(Xa, ya, Xb, yb, Xt):
    m = lgb.LGBMClassifier(n_estimators=30000, learning_rate=0.03, subsample_freq=1, n_jobs=THREADS, verbose=-1, **BEST)
    m.fit(Xa, ya, eval_set=[(Xb, yb)], eval_metric='auc', callbacks=[lgb.early_stopping(400, verbose=False)])
    return m.predict_proba(Xb)[:, 1], m.predict_proba(Xt)[:, 1]
run('LightGBM tuned', lgbm_tuned)


In [ ]:


def lgbm_orig(Xa, ya, Xb, yb, Xt):                    # starts from the original-data model's logit and learns only the correction
    if 'orig_model' not in Xa:
        return lgbm(Xa, ya, Xb, yb, Xt)
    lg = lambda d: np.log(np.clip(d['orig_model'].values, 1e-6, 1 - 1e-6) / (1 - np.clip(d['orig_model'].values, 1e-6, 1 - 1e-6)))
    m = lgb.LGBMClassifier(n_estimators=4000, learning_rate=0.03, num_leaves=63, min_child_samples=50, subsample=0.8, subsample_freq=1,
                           colsample_bytree=0.5, reg_lambda=1.0, n_jobs=THREADS, verbose=-1)
    m.fit(Xa, ya, init_score=lg(Xa), eval_set=[(Xb, yb)], eval_init_score=[lg(Xb)], eval_metric='auc',
          callbacks=[lgb.early_stopping(200, verbose=False)])
    return m.predict(Xb, raw_score=True) + lg(Xb), m.predict(Xt, raw_score=True) + lg(Xt)
run('LightGBM from original', lgbm_orig, 'fe')

# a spline GLM (design from sidhaarthshree's and goodpjw2008's notebooks): the original data looks like a logistic formula,
# so a linear model on splines + one-hots sees it differently from the trees, which is what a stack wants
from scipy import sparse
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, SplineTransformer, StandardScaler
full = pd.concat([train[feats], test[feats]], ignore_index=True)
nums = [c for c in num if c in feats]
zz = full[nums].astype(float).copy()
for c in ('amount_usd', 'account_balance_usd', 'hours_since_last_txn'):
    zz[c] = np.log1p(zz[c].clip(lower=0))
zz = StandardScaler().fit_transform(zz)
small = [c for c in nums if full[c].nunique() <= 100]
M = sparse.hstack([sparse.csr_matrix(zz), sparse.csr_matrix(full[boo].values.astype(float)),
                   OneHotEncoder(handle_unknown='ignore').fit_transform(full[cat].astype(str)),
                   sparse.csr_matrix(SplineTransformer(n_knots=8, degree=3, knots='quantile').fit_transform(zz)),
                   OneHotEncoder(handle_unknown='ignore', min_frequency=50).fit_transform(full[small])]).tocsr()
MA, MT = M[:len(train)], M[len(train):]
glm_key = 'spline_glm_' + hashlib.sha1(M.data[:200000].tobytes() + repr(M.shape).encode()).hexdigest()[:10]   # keyed by the design matrix
if glm_key in MANIFEST and (PRED_CACHE / f'oof_{glm_key}.npy').exists():
    oof['Spline GLM'], pred['Spline GLM'] = np.load(PRED_CACHE / f'oof_{glm_key}.npy'), np.load(PRED_CACHE / f'test_{glm_key}.npy')
    secs['Spline GLM'] = MANIFEST[glm_key]['minutes'] * 60; _publish(glm_key, 'Spline GLM', 'design')
    print(f"Spline GLM: OOF AUC {roc_auc_score(y, oof['Spline GLM']):.5f}  (loaded from the dataset, trained in {MANIFEST[glm_key]['version']})")
else:
    t0 = time.time(); o = np.zeros(len(X)); p = np.zeros(len(XT))
    for k, (tr_i, va_i) in enumerate(FOLDS):
        m = LogisticRegression(C=1.0, max_iter=4000, tol=1e-6).fit(MA[tr_i], y.iloc[tr_i])
        o[va_i] = m.predict_proba(MA[va_i])[:, 1]; p += m.predict_proba(MT)[:, 1] / len(FOLDS)
    oof['Spline GLM'], pred['Spline GLM'], secs['Spline GLM'] = o, p, time.time() - t0
    _publish(glm_key, 'Spline GLM', 'design', trained=True)
    print(f"Spline GLM: OOF AUC {roc_auc_score(y, o):.5f}  ({secs['Spline GLM'] / 60:.1f} min, {M.shape[1]} design columns)")


In [ ]:

import xgboost as xgb
def xgbm(Xa, ya, Xb, yb, Xt):
    m = xgb.XGBClassifier(n_estimators=4000, learning_rate=0.03, max_depth=6, min_child_weight=5, subsample=0.8, colsample_bytree=0.5,
                          tree_method='hist', device='cuda' if GPU else 'cpu', enable_categorical=True, eval_metric='auc', early_stopping_rounds=200, n_jobs=THREADS)
    m.fit(Xa, ya, eval_set=[(Xb, yb)], verbose=False)
    return m.predict_proba(Xb)[:, 1], m.predict_proba(Xt)[:, 1]
run('XGBoost', xgbm)
def xgbm_deep(Xa, ya, Xb, yb, Xt):                    # a second XGBoost: deeper, slower learning, different column sampling
    m = xgb.XGBClassifier(n_estimators=6000, learning_rate=0.02, max_depth=9, min_child_weight=20, subsample=0.7, colsample_bytree=0.35,
                          reg_lambda=5.0, tree_method='hist', device='cuda' if GPU else 'cpu', enable_categorical=True, eval_metric='auc',
                          early_stopping_rounds=300, n_jobs=THREADS)
    m.fit(Xa, ya, eval_set=[(Xb, yb)], verbose=False)
    return m.predict_proba(Xb)[:, 1], m.predict_proba(Xt)[:, 1]
run('XGBoost deep', xgbm_deep)
def xgbm_d2(Xa, ya, Xb, yb, Xt):                      # depth 2: the XGBoost version of the 4-leaf trees
    m = xgb.XGBClassifier(n_estimators=20000, learning_rate=0.05, max_depth=2, min_child_weight=20, subsample=0.8, colsample_bytree=0.5,
                          tree_method='hist', device='cuda' if GPU else 'cpu', enable_categorical=True, eval_metric='auc',
                          early_stopping_rounds=300, n_jobs=THREADS)
    m.fit(Xa, ya, eval_set=[(Xb, yb)], verbose=False)
    return m.predict_proba(Xb)[:, 1], m.predict_proba(Xt)[:, 1]
run('XGBoost depth 2', xgbm_d2, 'fe')
run('XGBoost depth 2 raw', xgbm_d2, 'raw')
def xgbm_d1(Xa, ya, Xb, yb, Xt):                      # depth 1: a purely additive model, one shape function per input
    m = xgb.XGBClassifier(n_estimators=30000, learning_rate=0.1, max_depth=1, min_child_weight=20, subsample=0.8, colsample_bytree=0.5,
                          tree_method='hist', device='cuda' if GPU else 'cpu', enable_categorical=True, eval_metric='auc',
                          early_stopping_rounds=500, n_jobs=THREADS)
    m.fit(Xa, ya, eval_set=[(Xb, yb)], verbose=False)
    return m.predict_proba(Xb)[:, 1], m.predict_proba(Xt)[:, 1]
run('XGBoost depth 1', xgbm_d1, 'fe')


In [ ]:

from catboost import CatBoostClassifier
def catb(Xa, ya, Xb, yb, Xt):
    s = lambda d: d.assign(**{c: d[c].astype(str) for c in cat})
    m = CatBoostClassifier(iterations=CAT_ITERS, learning_rate=0.06, depth=6, eval_metric='AUC', od_type='Iter', od_wait=200,
                           thread_count=THREADS, verbose=0, cat_features=cat, random_seed=0, train_dir=str(OUT / 'catboost_info'),
                           task_type='GPU' if GPU else 'CPU')
    m.fit(s(Xa), ya, eval_set=(s(Xb), yb))
    return m.predict_proba(s(Xb))[:, 1], m.predict_proba(s(Xt))[:, 1]
run('CatBoost', catb)
def catb3(Xa, ya, Xb, yb, Xt):                         # shallow CatBoost: symmetric depth-3 trees
    s = lambda d: d.assign(**{c: d[c].astype(str) for c in cat})
    m = CatBoostClassifier(iterations=3 * CAT_ITERS, learning_rate=0.08, depth=3, eval_metric='AUC', od_type='Iter', od_wait=300,
                           thread_count=THREADS, verbose=0, cat_features=cat, random_seed=0, train_dir=str(OUT / 'catboost_info'),
                           task_type='GPU' if GPU else 'CPU')
    m.fit(s(Xa), ya, eval_set=(s(Xb), yb))
    return m.predict_proba(s(Xb))[:, 1], m.predict_proba(s(Xt))[:, 1]
run('CatBoost depth 3', catb3)


In [ ]:

import torch
from pytabkit import RealMLP_TD_Classifier
torch.set_num_threads(THREADS)
print('RealMLP on', 'GPU' if GPU else 'CPU')
def realmlp(Xa, ya, Xb, yb, Xt):
    m = RealMLP_TD_Classifier(device='cuda' if GPU else 'cpu', random_state=0, verbosity=0, val_metric_name='1-auc_ovr',
                              n_ens=8, n_epochs=MLP_EPOCHS, batch_size=512, use_early_stopping=False, lr=0.04, wd=0.01, sq_mom=0.988,
                              lr_sched='flat_anneal', wd_sched='cos_log_15', first_layer_lr_factor=0.25, embedding_size=5,
                              max_one_hot_cat_size=2, hidden_sizes=[512, 256, 128], act='mish', p_drop=0.05, p_drop_sched='pow5',
                              plr_hidden_1=16, plr_hidden_2=8, plr_act_name='gelu', plr_lr_factor=0.1151, plr_sigma=0.75,
                              ls_eps=0.01, ls_eps_sched='sqrt_cos', add_front_scale=False, bias_init_mode='neg-uniform-dynamic-2',
                              tfms=['one_hot', 'median_center', 'robust_scale', 'smooth_clip', 'embedding', 'l2_normalize'])
    m.fit(Xa, ya, X_val=Xb, y_val=yb)
    return m.predict_proba(Xb)[:, 1], m.predict_proba(Xt)[:, 1]
run('RealMLP', realmlp, 'fe')


In [ ]:

import torch
# TabM (Gorishniy et al., ICLR 2025): k MLPs sharing most weights, trained together as one batch ensemble;
# piecewise-linear embeddings of the numeric inputs. Early stopping on the fold's validation rows, as the trees do.
from pytabkit import TabM_D_Classifier
def tabm(Xa, ya, Xb, yb, Xt):
    m = TabM_D_Classifier(device='cuda' if GPU else 'cpu', random_state=0, verbosity=0, val_metric_name='1-auc_ovr',
                          arch_type='tabm-mini', tabm_k=32, num_emb_type='pwl', d_block=512, n_blocks=3, dropout=0.1,
                          batch_size=1024, lr=2e-3, weight_decay=3e-4, n_epochs=40, patience=6)
    m.fit(Xa, ya, X_val=Xb, y_val=yb)
    return m.predict_proba(Xb)[:, 1], m.predict_proba(Xt)[:, 1]
run('TabM', tabm, 'fe')


### v9: the four "what next" items from v8

1. **Tuning the other shallow members.** The same cached search, run on XGBoost depth 2 and CatBoost depth 3.
2. **More features**: count encodings (how common a value is across train and test; no labels used), three products and ratios of the strongest numeric columns,
   and target encodings of triple crosses among the six strongest flags, fitted inside the fold like every other encoding. They feed two new members only,
   so every member trained before keeps its cached predictions.
3. **A bigger member library**: a k-nearest-neighbour model (the fraud rate among a row's 100 nearest training rows, GPU), and seed bags
   (the same depth-1 / depth-2 XGBoost trained with three seeds and averaged). Each is judged by what it adds to the stack, not by its own score.
4. **More seeds for the blend**: the stacker's held-out AUC is measured over three different outer splits, so the hill climb vs stacker choice comes with its noise.
   More folds for the members would change every out-of-fold file and retrain everything, so the 5 folds stay.


In [ ]:

# 1. tuned XGBoost depth <= 3 and CatBoost depth <= 4, searched like LightGBM (fold 0, 250k-row sample, fast learning rate)
def xgb_objective(trial):
    prm = dict(max_depth=trial.suggest_int('max_depth', 1, 3), min_child_weight=trial.suggest_float('min_child_weight', 1, 300, log=True),
               subsample=trial.suggest_float('subsample', 0.5, 1.0), colsample_bytree=trial.suggest_float('colsample_bytree', 0.1, 0.9),
               reg_lambda=trial.suggest_float('reg_lambda', 1e-3, 100, log=True))
    m = xgb.XGBClassifier(n_estimators=5000, learning_rate=0.1, tree_method='hist', device='cuda' if GPU else 'cpu', enable_categorical=True,
                          eval_metric='auc', early_stopping_rounds=200, n_jobs=THREADS, **prm)
    m.fit(Xa0.iloc[sub_i], ya0[sub_i], eval_set=[(Xb0, yb0)], verbose=False)
    return m.best_score
XGB_BEST = cached_search('XGBoost', xgb_objective, 30).best_params
def xgbm_tuned(Xa, ya, Xb, yb, Xt):
    m = xgb.XGBClassifier(n_estimators=30000, learning_rate=0.05, tree_method='hist', device='cuda' if GPU else 'cpu', enable_categorical=True,
                          eval_metric='auc', early_stopping_rounds=400, n_jobs=THREADS, **XGB_BEST)
    m.fit(Xa, ya, eval_set=[(Xb, yb)], verbose=False)
    return m.predict_proba(Xb)[:, 1], m.predict_proba(Xt)[:, 1]
run('XGBoost tuned', xgbm_tuned, 'fe')

def cat_objective(trial):
    s = lambda d: d.assign(**{c: d[c].astype(str) for c in cat})
    prm = dict(depth=trial.suggest_int('depth', 2, 4), l2_leaf_reg=trial.suggest_float('l2_leaf_reg', 0.1, 30, log=True),
               random_strength=trial.suggest_float('random_strength', 0.1, 10, log=True),
               bagging_temperature=trial.suggest_float('bagging_temperature', 0.0, 1.0))
    m = CatBoostClassifier(iterations=3000, learning_rate=0.1, eval_metric='AUC', od_type='Iter', od_wait=200, thread_count=THREADS, verbose=0,
                           cat_features=cat, random_seed=0, train_dir=str(OUT / 'catboost_info'), task_type='GPU' if GPU else 'CPU', **prm)
    m.fit(s(Xa0.iloc[sub_i]), ya0[sub_i], eval_set=(s(Xb0), yb0))
    return roc_auc_score(yb0, m.predict_proba(s(Xb0))[:, 1])
CAT_BEST = cached_search('CatBoost', cat_objective, 20).best_params
def catb_tuned(Xa, ya, Xb, yb, Xt):
    s = lambda d: d.assign(**{c: d[c].astype(str) for c in cat})
    m = CatBoostClassifier(iterations=3 * CAT_ITERS, learning_rate=0.08, eval_metric='AUC', od_type='Iter', od_wait=300, thread_count=THREADS,
                           verbose=0, cat_features=cat, random_seed=0, train_dir=str(OUT / 'catboost_info'), task_type='GPU' if GPU else 'CPU', **CAT_BEST)
    m.fit(s(Xa), ya, eval_set=(s(Xb), yb))
    return m.predict_proba(s(Xb))[:, 1], m.predict_proba(s(Xt))[:, 1]
run('CatBoost tuned', catb_tuned, 'fe')


In [ ]:

# 2. more features, for two new members only (the feature cell above is unchanged, so no earlier member retrains)
from itertools import combinations
def plus_features(Xa, ya, Xb, Xt):
    """Count encodings and numeric products (no labels), and in-fold target encodings of triple crosses (fitted on Xa only)."""
    out = [Xa.copy(), Xb.copy(), Xt.copy()]
    for c in ['amount_usd', 'account_balance_usd', 'hours_since_last_txn', 'distance_from_home_km', 'merchant_risk_score',
              'velocity_score', 'customer_age', 'card_age_months']:
        vc = both[c].value_counts()                                   # how common the exact value is in train + test
        for d in out:
            d['cnt_' + c] = d[c].map(vc).fillna(0).astype('float32').values
    for d in out:
        d['amount_over_balance'] = (d['amount_usd'] / (d['account_balance_usd'].abs() + 1)).astype('float32')
        d['amount_x_risk'] = (d['amount_usd'] * d['merchant_risk_score']).astype('float32')
        d['velocity_x_risk'] = (d['velocity_score'] * d['merchant_risk_score']).astype('float32')
    top6 = ['cvv_retry_count', 'ip_country_mismatch', 'is_foreign_transaction', 'is_new_merchant', 'billing_shipping_mismatch', 'auth_method']
    keys = lambda d: pd.DataFrame({f'{a}__{b}__{c}': d[a].astype(str) + '_' + d[b].astype(str) + '_' + d[c].astype(str)
                                   for a, b, c in combinations(top6, 3)})
    enc = TargetEncoder(target_type='binary', cv=5, random_state=0)
    te = [enc.fit_transform(keys(Xa), ya), enc.transform(keys(Xb)), enc.transform(keys(Xt))]
    cols = ['te3_' + c for c in keys(Xa.head(1)).columns]
    return [pd.concat([d.reset_index(drop=True), pd.DataFrame(t, columns=cols).astype('float32')], axis=1) for d, t in zip(out, te)]
def lgbm_plus(Xa, ya, Xb, yb, Xt):
    Xa, Xb, Xt = plus_features(Xa, ya, Xb, Xt)
    m = lgb.LGBMClassifier(n_estimators=20000, learning_rate=0.05, num_leaves=4, min_child_samples=100, subsample=0.8, subsample_freq=1,
                           colsample_bytree=0.5, reg_lambda=1.0, n_jobs=THREADS, verbose=-1)
    m.fit(Xa, ya, eval_set=[(Xb, yb)], eval_metric='auc', callbacks=[lgb.early_stopping(300, verbose=False)])
    return m.predict_proba(Xb)[:, 1], m.predict_proba(Xt)[:, 1]
run('LightGBM 4 leaves plus', lgbm_plus, 'fe')
def xgbm_d2_plus(Xa, ya, Xb, yb, Xt):
    Xa, Xb, Xt = plus_features(Xa, ya, Xb, Xt)
    m = xgb.XGBClassifier(n_estimators=20000, learning_rate=0.05, max_depth=2, min_child_weight=20, subsample=0.8, colsample_bytree=0.5,
                          tree_method='hist', device='cuda' if GPU else 'cpu', enable_categorical=True, eval_metric='auc',
                          early_stopping_rounds=300, n_jobs=THREADS)
    m.fit(Xa, ya, eval_set=[(Xb, yb)], verbose=False)
    return m.predict_proba(Xb)[:, 1], m.predict_proba(Xt)[:, 1]
run('XGBoost depth 2 plus', xgbm_d2_plus, 'fe')


In [ ]:

# 3. a bigger library: k nearest neighbours (GPU) and seed bags of the two shallow XGBoost members
def knn_rate(Xa, ya, Xb, yb, Xt):
    """Smoothed fraud rate among each row's 100 nearest training rows, on standardised numeric columns and the strongest encodings."""
    cols = [c for c in ['merchant_risk_score', 'velocity_score', 'cvv_retry_count', 'customer_age', 'txn_count_last_24h',
                        'hours_since_last_txn', 'amount_usd', 'prior_disputes', 'te_cvv_retry_count__ip_country_mismatch',
                        'te_merchant_category', 'te_auth_method', 'te_is_new_merchant__used_vpn'] if c in Xa]
    mu, sd = Xa[cols].astype(float).mean(), Xa[cols].astype(float).std() + 1e-9
    dev = 'cuda' if GPU else 'cpu'
    ref = torch.tensor(((Xa[cols].astype(float) - mu) / sd).values, dtype=torch.float32, device=dev)
    lab = torch.tensor(ya, dtype=torch.float32, device=dev); prior = float(ya.mean())
    def rate(D):
        q = torch.tensor(((D[cols].astype(float) - mu) / sd).values, dtype=torch.float32, device=dev); res = []
        for i in range(0, len(q), 1024):
            idx = torch.cdist(q[i:i + 1024], ref).topk(100, largest=False).indices
            res.append(((lab[idx].sum(1) + 10 * prior) / 110).cpu())
        return torch.cat(res).numpy()
    return rate(Xb), rate(Xt)
if GPU:                                                   # 440k queries against 560k rows per fold: minutes on a GPU, hours on 4 CPU cores
    run('kNN rate', knn_rate, 'fe')
def xgb_bag(depth, Xa, ya, Xb, yb, Xt):
    vo, tp = 0, 0
    for seed in (1, 2, 3):
        m = xgb.XGBClassifier(n_estimators=30000, learning_rate=0.05 if depth == 2 else 0.1, max_depth=depth, min_child_weight=20, subsample=0.8,
                              colsample_bytree=0.5, random_state=seed, tree_method='hist', device='cuda' if GPU else 'cpu', enable_categorical=True,
                              eval_metric='auc', early_stopping_rounds=300 if depth == 2 else 500, n_jobs=THREADS)
        m.fit(Xa, ya, eval_set=[(Xb, yb)], verbose=False)
        vo, tp = vo + m.predict_proba(Xb)[:, 1] / 3, tp + m.predict_proba(Xt)[:, 1] / 3
    return vo, tp
def xgb_d2_bag3(Xa, ya, Xb, yb, Xt): return xgb_bag(2, Xa, ya, Xb, yb, Xt)
def xgb_d1_bag3(Xa, ya, Xb, yb, Xt): return xgb_bag(1, Xa, ya, Xb, yb, Xt)
run('XGBoost depth 2 bag3', xgb_d2_bag3, 'fe')
run('XGBoost depth 1 bag3', xgb_d1_bag3, 'fe')


### v10: the digits carry the label

The synthetic rows come from a generator that writes numbers digit by digit, so how a number is *written* carries signal that its size does not.
goodpjw2008 measured it ([S6E11 Part 3](https://www.kaggle.com/code/goodpjw2008/s6e11-part-3-digits-leak-the-label-lb-0-95113), LB 0.95113)
and ektarr's [RealMLP Stacking](https://www.kaggle.com/code/ektarr/lb-0-95121-realmlp-stacking) (LB 0.95121) uses the same keys.
For each of the six float columns, these keys are target-encoded inside the fold, exactly like the per-value encodings above:

| key | example for 1371.37 | what it groups |
|---|---|---|
| cents | 37 | the two decimals (also kept as a plain number) |
| int(x) | 1371 | the integer part (truncated, not rounded) |
| int(x/10) | 137 | the integer part without its last digit |
| int(10x) | 13713 | the value cut to one decimal |
| first digit, first two digits, number of digits | 1, 13, 4 | the leading digits |

Our v2-v9 features had only a part of this: the first decimal as a raw number, and *rounded* integer keys for four columns.
Rounding mixes two written forms into one key (13.6 and 14.4 both become 14), which is exactly the information the generator left; truncating keeps them apart.
Their measurement on one 4-leaf LightGBM: +0.0015 CV. The keys feed new members only, so nothing cached retrains.

The four new members, and why each:

- **LightGBM 4 leaves digits**: the best member so far (4-leaf LightGBM on the engineered set) with the digit encodings added. The direct test of the idea.
- **LightGBM 4 leaves digits s100**: the same keys with heavier smoothing (100 instead of 20), so rare keys count for less. Its errors differ slightly, which a blend can use; goodpjw2008's blend kept both.
- **XGBoost depth 2 digits**: the same inputs in another tree library; depth-2 XGBoost was the second-strongest shallow member.
- **RealMLP digits**: the neural net with the digit encodings as extra inputs. Nets make different mistakes from trees, so it is the member most likely to add to the stack.


In [ ]:

FLOATS6 = ['amount_usd', 'hours_since_last_txn', 'distance_from_home_km', 'account_balance_usd', 'velocity_score', 'merchant_risk_score']
def digit_features(Xa, ya, Xb, Xt, smooth=20.0):
    """The digit keys of the six float columns, target-encoded on Xa's labels only (5-way cross-fitted inside it); cents kept as a number."""
    def keys(d):
        k = {}
        for c in FLOATS6:
            x = np.nan_to_num(d[c].astype(float).values, nan=0.0, posinf=0.0, neginf=0.0); i = np.floor(x + 1e-9).astype(np.int64); s = np.abs(i).astype(str)
            k[f'cents_{c}'] = np.round(x * 100).astype(np.int64) % 100
            k[f'int_{c}'], k[f'ten_{c}'], k[f'dec_{c}'] = i, np.floor(x / 10 + 1e-9).astype(np.int64), np.floor(x * 10 + 1e-6).astype(np.int64)
            k[f'lead1_{c}'] = np.array([v[:1] for v in s]); k[f'lead2_{c}'] = np.array([v[:2] for v in s]); k[f'len_{c}'] = np.array([len(v) for v in s])
        return pd.DataFrame(k).astype(str)
    ka, kb, kt = keys(Xa), keys(Xb), keys(Xt)
    enc = TargetEncoder(target_type='binary', cv=5, smooth=smooth, random_state=0)
    cols = ['dte_' + c for c in ka.columns]
    te = [enc.fit_transform(ka, ya), enc.transform(kb), enc.transform(kt)]
    cents = lambda d: pd.DataFrame({f'cents_{c}': (np.round(np.nan_to_num(d[c].astype(float).values, nan=0.0, posinf=0.0, neginf=0.0) * 100) % 100).astype('float32') for c in FLOATS6})
    return [pd.concat([d.reset_index(drop=True), cents(d), pd.DataFrame(t, columns=cols).astype('float32')], axis=1)
            for d, t in zip((Xa, Xb, Xt), te)]
def lgbm_digits(Xa, ya, Xb, yb, Xt):
    Xa, Xb, Xt = digit_features(Xa, ya, Xb, Xt)
    m = lgb.LGBMClassifier(n_estimators=20000, learning_rate=0.03, num_leaves=4, min_child_samples=100, subsample=0.8, subsample_freq=1,
                           colsample_bytree=0.5, reg_lambda=1.0, n_jobs=THREADS, verbose=-1)
    m.fit(Xa, ya, eval_set=[(Xb, yb)], eval_metric='auc', callbacks=[lgb.early_stopping(300, verbose=False)])
    return m.predict_proba(Xb)[:, 1], m.predict_proba(Xt)[:, 1]
run('LightGBM 4 leaves digits', lgbm_digits, 'fe')
def lgbm_digits_s100(Xa, ya, Xb, yb, Xt):                # the same with heavier smoothing: a second view the blend can use
    Xa, Xb, Xt = digit_features(Xa, ya, Xb, Xt, smooth=100.0)
    m = lgb.LGBMClassifier(n_estimators=20000, learning_rate=0.03, num_leaves=4, min_child_samples=100, subsample=0.8, subsample_freq=1,
                           colsample_bytree=0.5, reg_lambda=1.0, random_state=1, n_jobs=THREADS, verbose=-1)
    m.fit(Xa, ya, eval_set=[(Xb, yb)], eval_metric='auc', callbacks=[lgb.early_stopping(300, verbose=False)])
    return m.predict_proba(Xb)[:, 1], m.predict_proba(Xt)[:, 1]
run('LightGBM 4 leaves digits s100', lgbm_digits_s100, 'fe')
def xgbm_d2_digits(Xa, ya, Xb, yb, Xt):
    Xa, Xb, Xt = digit_features(Xa, ya, Xb, Xt)
    m = xgb.XGBClassifier(n_estimators=20000, learning_rate=0.03, max_depth=2, min_child_weight=5, subsample=0.8, colsample_bytree=0.5,
                          tree_method='hist', device='cuda' if GPU else 'cpu', enable_categorical=True, eval_metric='auc',
                          early_stopping_rounds=300, n_jobs=THREADS)
    m.fit(Xa, ya, eval_set=[(Xb, yb)], verbose=False)
    return m.predict_proba(Xb)[:, 1], m.predict_proba(Xt)[:, 1]
run('XGBoost depth 2 digits', xgbm_d2_digits, 'fe')
def realmlp_digits(Xa, ya, Xb, yb, Xt):
    Xa, Xb, Xt = digit_features(Xa, ya, Xb, Xt)
    return realmlp(Xa, ya, Xb, yb, Xt)
run('RealMLP digits', realmlp_digits, 'fe')


### v12: three cheap members suggested by the public notebooks

Our digit members score OOF 0.9495; the public notebooks' 4-leaf members with the same keys score 0.9504-0.9505 (goodpjw2008 Part 4, megayak, bulsahkecici).
One difference we can name: every `fe` member here also carries the encodings of the 45 column pairs among the top 10.
Two public notebooks measured those pairs as useless or harmful (areebbb -0.0007 CV on 1 of 5 folds; aphoristicartist 0.95024, no gain).

- **LightGBM 4 leaves digits, no pairs**: `LightGBM 4 leaves digits` with the 45 pair encodings dropped. If the pairs only add noise, this member beats its parent; that is the whole test.
- **XGBoost depth 2 digits, no pairs**: the same test in the other tree library.
- **GLM on the encodings**: a logistic regression on the logits of every value and digit encoding. Alone it is the weakest model,
  but it fits all encodings jointly and has no trees, so its errors differ; aphoristicartist's stack gives this kind of member its largest weight (0.30), megayak's scores 0.9501 alone.


In [ ]:

NO_PAIR = lambda d: d.drop(columns=[c for c in d.columns if c.startswith('te_') and '__' in c])
def lgbm_digits_nopair(Xa, ya, Xb, yb, Xt):
    Xa, Xb, Xt = digit_features(NO_PAIR(Xa), ya, NO_PAIR(Xb), NO_PAIR(Xt))
    m = lgb.LGBMClassifier(n_estimators=20000, learning_rate=0.03, num_leaves=4, min_child_samples=100, subsample=0.8, subsample_freq=1,
                           colsample_bytree=0.5, reg_lambda=1.0, n_jobs=THREADS, verbose=-1)
    m.fit(Xa, ya, eval_set=[(Xb, yb)], eval_metric='auc', callbacks=[lgb.early_stopping(300, verbose=False)])
    return m.predict_proba(Xb)[:, 1], m.predict_proba(Xt)[:, 1]
run('LightGBM 4 leaves digits, no pairs', lgbm_digits_nopair, 'fe')
def xgbm_d2_digits_nopair(Xa, ya, Xb, yb, Xt):
    Xa, Xb, Xt = digit_features(NO_PAIR(Xa), ya, NO_PAIR(Xb), NO_PAIR(Xt))
    m = xgb.XGBClassifier(n_estimators=20000, learning_rate=0.03, max_depth=2, min_child_weight=5, subsample=0.8, colsample_bytree=0.5,
                          tree_method='hist', device='cuda' if GPU else 'cpu', enable_categorical=True, eval_metric='auc',
                          early_stopping_rounds=300, n_jobs=THREADS)
    m.fit(Xa, ya, eval_set=[(Xb, yb)], verbose=False)
    return m.predict_proba(Xb)[:, 1], m.predict_proba(Xt)[:, 1]
run('XGBoost depth 2 digits, no pairs', xgbm_d2_digits_nopair, 'fe')
def glm_encodings(Xa, ya, Xb, yb, Xt):                   # logistic regression on the logits of every value and digit encoding
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    Xa, Xb, Xt = digit_features(Xa, ya, Xb, Xt)
    cols = [c for c in Xa.columns if c.startswith(('te_', 'dte_'))]
    lg = lambda d: np.log(np.clip(d[cols].values, 1e-5, 1 - 1e-5) / (1 - np.clip(d[cols].values, 1e-5, 1 - 1e-5)))
    sc = StandardScaler().fit(lg(Xa))
    m = LogisticRegression(C=0.05, max_iter=2000).fit(sc.transform(lg(Xa)), ya)
    return m.predict_proba(sc.transform(lg(Xb)))[:, 1], m.predict_proba(sc.transform(lg(Xt)))[:, 1]
run('GLM on the encodings', glm_encodings, 'fe')


### v13: the public recipe, reproduced exactly

v12's test said the pair encodings are not why our digit members (OOF 0.9495) trail the public ones (0.9504).
Reading goodpjw2008's Part 3 and Part 5 line by line against ours shows three other differences:

| | ours (v10-v12) | goodpjw2008 Part 3 / 5 |
|---|---|---|
| encoder | sklearn `TargetEncoder`, `smooth='auto'` for the value keys | an m-estimate: (fraud count + 20 x prior) / (rows + 20), cross-fitted 5 ways inside the fold |
| what is encoded | every column, rounded keys, 45 pairs, then the digit keys | every column, the cents and every digit key, all with the same encoder |
| other inputs | first-decimal digits, original-data lookups and model, rounded keys | the raw columns and the cents only |

`smooth='auto'` picks the shrinkage from each key's variance; for a column like `account_balance_usd` (43k distinct values, a few rows each)
that is a very different amount of shrinkage than a fixed 20. The extra inputs matter too: with half the columns sampled per tree, every extra
column takes turns from the strong ones.

The three members below are that recipe, copied (credit: goodpjw2008), on our folds, starting from the raw columns:

- **LightGBM 4 leaves, Part 3 recipe**: `lgb_diglead_l4`, the public notebooks' best single model (their CV 0.95040).
- **LightGBM 3 leaves, Part 5 recipe**: the same with 3 leaves (Part 5: 3 leaves beat 4 on every fold; their 3-seed bag 0.95063).
- **XGBoost depth 2, Part 3 recipe**: `xgb_dig_d2` with the same encodings.

If the first one reaches about 0.9504 here, the encoder and the extra inputs explain our gap, and the lesson is worth more than the member.


In [ ]:
def p3_te(Xa, ya, others, cols, smooth=20.0, inner=5, seed=0):
    """goodpjw2008's out-of-fold m-estimate encoding: each value of each column -> (frauds + smooth x prior) / (rows + smooth)."""
    prior = float(ya.mean())
    Xa, others = Xa.copy(), [o.copy() for o in others]
    key = lambda X, c: X[c].astype(str).values if str(X[c].dtype) == 'category' else X[c].values
    rate = lambda k, t: (lambda s: (s['sum'] + prior * smooth) / (s['count'] + smooth))(
        pd.DataFrame({'k': k, 'y': t}).groupby('k')['y'].agg(['sum', 'count']))
    for c in cols:
        ka, enc = key(Xa, c), np.zeros(len(Xa), dtype=np.float32)
        for i, j in StratifiedKFold(inner, shuffle=True, random_state=seed).split(ka, ya):
            enc[j] = pd.Series(ka[j]).map(rate(ka[i], ya[i])).fillna(prior).values
        Xa[f'te_{c}'] = enc
        r = rate(ka, ya)
        for o in others:
            o[f'te_{c}'] = pd.Series(key(o, c)).map(r).fillna(prior).values.astype(np.float32)
    return Xa, others
def p3_frames(Xa, ya, Xb, Xt, seed=0):
    """Part 3's digit and leading-digit keys of the six floats, every column and key encoded with p3_te, the keys then dropped."""
    def add_keys(d):
        d = d.reset_index(drop=True).copy()
        for c in FLOATS6:
            x = np.nan_to_num(np.asarray(d[c], dtype=float), nan=0.0, posinf=0.0, neginf=0.0)
            d[f'cents_{c}'] = (np.round(x * 100) % 100).astype('int16')
            d[f'key_int_{c}'] = np.floor(x).astype('int64'); d[f'key_ten_{c}'] = np.floor(x / 10).astype('int64')
            d[f'key_dec_{c}'] = np.floor(x * 10 + 1e-6).astype('int64')
            i = np.floor(np.abs(x)).astype('int64').astype(str)
            d[f'key_l1_{c}'] = (i.str.len() * 10 + i.str[0].astype(int)).values
            d[f'key_l2_{c}'] = (i.str.len() * 100 + i.str[:2].astype(int)).values
        return d
    Xa, Xb, Xt = add_keys(Xa), add_keys(Xb), add_keys(Xt)
    cols = feats + [c for c in Xa.columns if c.startswith(('cents_', 'key_'))]
    Xa, (Xb, Xt) = p3_te(Xa, ya, [Xb, Xt], cols, seed=seed)
    drop = lambda d: d[[c for c in d.columns if not c.startswith('key_')]]
    return drop(Xa), drop(Xb), drop(Xt)
P3_LGB = dict(n_estimators=20000, learning_rate=0.03, num_leaves=4, min_child_samples=100, subsample=0.8, subsample_freq=1,
              colsample_bytree=0.5, reg_lambda=1.0, max_bin=255, verbose=-1)
def lgbm_p3_l4(Xa, ya, Xb, yb, Xt):
    Xa, Xb, Xt = p3_frames(Xa, ya, Xb, Xt)
    m = lgb.LGBMClassifier(**P3_LGB, random_state=42, n_jobs=THREADS)
    m.fit(Xa, ya, eval_set=[(Xb, yb)], eval_metric='auc', callbacks=[lgb.early_stopping(300, verbose=False)])
    return m.predict_proba(Xb)[:, 1], m.predict_proba(Xt)[:, 1]
run('LightGBM 4 leaves, Part 3 recipe', lgbm_p3_l4, 'raw')
def lgbm_p3_l3(Xa, ya, Xb, yb, Xt):
    Xa, Xb, Xt = p3_frames(Xa, ya, Xb, Xt)
    m = lgb.LGBMClassifier(**{**P3_LGB, 'num_leaves': 3}, random_state=42, n_jobs=THREADS)
    m.fit(Xa, ya, eval_set=[(Xb, yb)], eval_metric='auc', callbacks=[lgb.early_stopping(300, verbose=False)])
    return m.predict_proba(Xb)[:, 1], m.predict_proba(Xt)[:, 1]
run('LightGBM 3 leaves, Part 5 recipe', lgbm_p3_l3, 'raw')
def lgbm_p3_l3_bag(Xa, ya, Xb, yb, Xt):
    """The Part 5 member refitted with five encoder seeds (the inner split p3_te encodes the training rows with), averaged."""
    vo, tp = np.zeros(len(Xb)), np.zeros(len(Xt))
    for s in range(5):
        a, b, t = p3_frames(Xa, ya, Xb, Xt, seed=s)
        m = lgb.LGBMClassifier(**{**P3_LGB, 'num_leaves': 3}, random_state=42 + s, n_jobs=THREADS)
        m.fit(a, ya, eval_set=[(b, yb)], eval_metric='auc', callbacks=[lgb.early_stopping(300, verbose=False)])
        vo += m.predict_proba(b)[:, 1] / 5; tp += m.predict_proba(t)[:, 1] / 5
    return vo, tp
run('LightGBM 3 leaves, Part 5 recipe, 5 encoder seeds', lgbm_p3_l3_bag, 'raw')
def xgbm_p3_d2(Xa, ya, Xb, yb, Xt):
    Xa, Xb, Xt = p3_frames(Xa, ya, Xb, Xt)
    m = xgb.XGBClassifier(n_estimators=20000, learning_rate=0.03, max_depth=2, subsample=0.8, colsample_bytree=0.5, min_child_weight=5,
                          reg_lambda=1.0, max_bin=256, tree_method='hist', device='cuda' if GPU else 'cpu', eval_metric='auc',
                          early_stopping_rounds=300, enable_categorical=True, max_cat_to_onehot=1, random_state=42, n_jobs=THREADS)
    m.fit(Xa, ya, eval_set=[(Xb, yb)], verbose=False)
    return m.predict_proba(Xb)[:, 1], m.predict_proba(Xt)[:, 1]
run('XGBoost depth 2, Part 3 recipe', xgbm_p3_d2, 'raw')

### v15: test predictions from one fit on all rows, a joint GLM, and trees that start from it

From v10 to v14 the stacker's held-out AUC rose 0.95019 -> 0.95091, and the board moved only 0.95095 -> 0.95101.
The OOF side is getting better; the test side is not following.
Every member so far predicts the test set as **the mean of its 5 fold models**, each trained on 80% of the rows,
with target encodings computed from those 80%.
Four public notebooks replaced that with **one more fit on all 700k rows** (encodings from all rows too, 1.25 x the folds' mean number of trees,
since there are 25% more rows), and each measured a leaderboard gain with the OOF unchanged:

| who | change | public LB |
|---|---|---|
| [goodpjw2008, Part 4](https://www.kaggle.com/code/goodpjw2008/s6e11-part-4-refit-bag-joint-glm-lb-0-95154) | same blend, refit test predictions | 0.95113 -> 0.95132 |
| [areebbb, Pooled Stack](https://www.kaggle.com/code/areebbb/s6e11-pooled-stack-fusion-lb-0-95196) | same blend, refit test predictions | 0.95135 -> 0.95142 |
| [megayak](https://www.kaggle.com/code/megayak/s6e11-per-value-digit-te-shallow-tree-stack) | refit of every member, plus joint keys | 0.95104 -> 0.95145 |

Why it can help here more than usual: the median `account_balance_usd` value occurs in 3 rows, so an encoding computed from 80% of the rows
is noticeably noisier than one from 100%. The OOF cannot show this gain, by construction: only the test predictions change.
The leaderboard is the only judge, which is why it is worth a version of its own.

The three new members, all refitted this way (`run_refit` below):

- **XGBoost depth 2, Part 3 recipe, 5 encoder seeds, full refit**: v13's depth-2 member, bagged over five encoder seeds like v14's LightGBM
  (which moved that member 0.95052 -> 0.95066). XGBoost runs on the GPU, so the bag costs minutes, not the 72 of the LightGBM bag.
- **Joint GLM, full refit**: one L2 logistic regression on the numbers, flags, categories and splines, **a one-hot of every exact value and every digit key
  seen at least 5 times**, and the logits of every encoding. The one-hots let the GLM correct each encoding for the others,
  and a rare value falls back to its coarser digit keys. Our v12 GLM had only the encodings.
  Design: goodpjw2008 (Part 4, `glm_joint`, CV 0.94991, two of nine blend weights) and lazyyeou (`glm_joint_96key`, 0.9501, second-largest blend weight).
- **XGBoost depth 2 boosted from the joint GLM, full refit**: the trees start from the GLM's logit (`base_margin`) instead of from the base rate,
  so they fit only what the additive model misses. Idea and measurement: megayak, v5, best single model of that notebook (0.95104 OOF), blend +0.00016 CV, LB 0.95145 -> 0.95162.


In [ ]:

REFIT_MULT = 1.25                                        # trees in the full refit: 1.25 x the folds' mean best iteration (25% more rows)
def run_refit(name, fit_predict, mode='raw'):
    """run() with one change: the test prediction comes from one more fit on all training rows (encodings fitted on all of them too),
    not from the mean of the 5 fold models. fit_predict(Xa, ya, Xb, yb, Xt, n_iter=None) returns (valid prediction, test prediction, best
    iteration); called with Xb=None it fits n_iter rounds on Xa without early stopping and returns only the test prediction."""
    assert mode == 'raw'
    if (time.time() - T_START) / 3600 > BUDGET_H:
        print(f'{name}: skipped, {BUDGET_H} h budget used'); return
    fp = fingerprint(name, fit_predict, mode)
    if fp in MANIFEST and (PRED_CACHE / f'oof_{fp}.npy').exists():
        oof[name], pred[name] = np.load(PRED_CACHE / f'oof_{fp}.npy'), np.load(PRED_CACHE / f'test_{fp}.npy'); secs[name] = MANIFEST[fp]['minutes'] * 60
        _publish(fp, name, mode)
        print(f'{name}: OOF AUC {roc_auc_score(y, oof[name]):.5f}  (loaded from the dataset, trained in {MANIFEST[fp]["version"]})'); return
    t0 = time.time(); o, its = np.zeros(len(X)), []
    for k, (tr_i, va_i) in enumerate(FOLDS):
        Xa, Xb, Xt = frames(k, mode)
        vo, _, it = fit_predict(Xa, y.iloc[tr_i].values, Xb, y.iloc[va_i].values, Xt)
        o[va_i] = vo; its.append(it)
        print(f'  {name} fold {k}: AUC {roc_auc_score(y.iloc[va_i], vo):.5f}  ({it:.0f} rounds)', flush=True)
    n = int(round(np.mean(its) * REFIT_MULT))
    _, p, _ = fit_predict(X.reset_index(drop=True), y.values, None, None, XT.reset_index(drop=True), n_iter=n)
    oof[name], pred[name], secs[name] = o, p, time.time() - t0
    _publish(fp, name, mode, trained=True)
    print(f'{name}: OOF AUC {roc_auc_score(y, o):.5f}  ({secs[name] / 60:.1f} min; test from one fit on all {len(X):,} rows, {n} rounds)')

def xgbm_p3_d2_bag_refit(Xa, ya, Xb, yb, Xt, n_iter=None):
    """xgbm_p3_d2 averaged over five encoder seeds (and five model seeds); with Xb None, the five fits on all rows."""
    vo, tp, its = 0, 0, []
    for s in range(5):
        a, b, t = p3_frames(Xa, ya, Xb if Xb is not None else Xt, Xt, seed=s)
        prm = dict(learning_rate=0.03, max_depth=2, subsample=0.8, colsample_bytree=0.5, min_child_weight=5, reg_lambda=1.0, max_bin=256,
                   tree_method='hist', device='cuda' if GPU else 'cpu', eval_metric='auc', enable_categorical=True, max_cat_to_onehot=1,
                   random_state=42 + s, n_jobs=THREADS)
        if Xb is None:
            m = xgb.XGBClassifier(n_estimators=n_iter, **prm).fit(a, ya, verbose=False)
            tp = tp + m.predict_proba(t)[:, 1] / 5
        else:
            m = xgb.XGBClassifier(n_estimators=20000, early_stopping_rounds=300, **prm).fit(a, ya, eval_set=[(b, yb)], verbose=False)
            vo = vo + m.predict_proba(b)[:, 1] / 5; its.append(m.best_iteration + 1)
    return (None, tp, n_iter) if Xb is None else (vo, None, float(np.mean(its)))
run_refit('XGBoost depth 2, Part 3 recipe, 5 encoder seeds, full refit', xgbm_p3_d2_bag_refit)

def v15_keys(d):
    """The digit keys p3_frames encodes (cents, int, int/10, int(10x), leading one and two digits with the length), as numbers."""
    k = {}
    for c in FLOATS6:
        x = np.nan_to_num(d[c].astype(float).values, nan=0.0, posinf=0.0, neginf=0.0); i = np.floor(x).astype(np.int64); s = pd.Series(np.floor(np.abs(x)).astype(np.int64).astype(str))
        k[f'cents_{c}'] = np.round(x * 100).astype(np.int64) % 100
        k[f'int_{c}'], k[f'ten_{c}'], k[f'dec_{c}'] = i, np.floor(x / 10).astype(np.int64), np.floor(x * 10 + 1e-6).astype(np.int64)
        k[f'l1_{c}'] = (s.str.len() * 10 + s.str[0].astype(int)).values
        k[f'l2_{c}'] = (s.str.len() * 100 + s.str[:2].astype(int)).values
    return pd.DataFrame(k)
_JOINT = {}                                              # the boosted member reuses the GLM member's fits: same rows, same logits
def joint_glm_logits(Xa, ya, Xb, Xt, C=0.01):
    """The joint GLM fitted on Xa: standardised (log) numbers, flags, one-hot categories, splines, a one-hot of every exact numeric value
    and every digit key seen >= 5 times in Xa, and the standardised logits of the Part 3 encodings. Returns its logit on Xa, Xb, Xt."""
    key = (len(Xa), len(Xb), int(ya.sum()), float(Xa['amount_usd'].sum()), float(Xb['amount_usd'].sum()))
    if key in _JOINT:
        return _JOINT[key]
    a, b, t = p3_frames(Xa, ya, Xb, Xt)
    tecols = [c for c in a.columns if c.startswith('te_')]
    lg = lambda e: (lambda q: np.log(q / (1 - q)))(np.clip(e[tecols].values, 1e-6, 1 - 1e-6))
    def dense(d):
        z = d[nums].astype(float).reset_index(drop=True)
        for c in ('amount_usd', 'account_balance_usd', 'hours_since_last_txn'):
            z[c] = np.log1p(z[c].clip(lower=0))
        return z.values
    vals = lambda d: pd.concat([d[nums].astype(float).reset_index(drop=True), v15_keys(d).astype(float)], axis=1)
    sc_n, sc_e = StandardScaler().fit(dense(Xa)), StandardScaler().fit(lg(a))
    spl = SplineTransformer(n_knots=8, degree=3, knots='quantile').fit(sc_n.transform(dense(Xa)))
    oh_c = OneHotEncoder(handle_unknown='ignore').fit(Xa[cat].astype(str))
    oh_v = OneHotEncoder(handle_unknown='ignore', min_frequency=5, dtype=np.float32).fit(vals(Xa))
    def design(d, e):
        zn = sc_n.transform(dense(d))
        return sparse.hstack([sparse.csr_matrix(zn), sparse.csr_matrix(d[boo].values.astype(float)), oh_c.transform(d[cat].astype(str)),
                              sparse.csr_matrix(spl.transform(zn)), oh_v.transform(vals(d)), sparse.csr_matrix(sc_e.transform(lg(e)))]).tocsr()
    Da = design(Xa, a)
    m = LogisticRegression(C=C, max_iter=3000, tol=1e-6).fit(Da, ya)
    print(f'    joint GLM: {Da.shape[1]:,} design columns, {m.n_iter_[0]} iterations', flush=True)
    _JOINT[key] = [m.decision_function(Da), m.decision_function(design(Xb, b)), m.decision_function(design(Xt, t))]
    return _JOINT[key]
def glm_joint(Xa, ya, Xb, yb, Xt, n_iter=None):
    za, zb, zt = joint_glm_logits(Xa, ya, Xb if Xb is not None else Xt, Xt)
    return (None, zt, 0) if Xb is None else (zb, None, 0)
run_refit('Joint GLM, full refit', glm_joint)

def xgbm_glm_boost(Xa, ya, Xb, yb, Xt, n_iter=None):
    """megayak's booster: depth-2 XGBoost on the Part 3 inputs whose starting score (base_margin) is the joint GLM's logit,
    so its trees fit only what the GLM misses. Predictions are margins (logits); AUC and the rank blend only see their order."""
    Xv = Xb if Xb is not None else Xt
    za, zb, zt = joint_glm_logits(Xa, ya, Xv, Xt)
    a, b, t = p3_frames(Xa, ya, Xv, Xt)
    prm = dict(learning_rate=0.02, max_depth=2, subsample=0.8, colsample_bytree=0.5, min_child_weight=5, reg_lambda=1.0, max_bin=256,
               tree_method='hist', device='cuda' if GPU else 'cpu', eval_metric='auc', enable_categorical=True, max_cat_to_onehot=1,
               random_state=42, n_jobs=THREADS)
    if Xb is None:
        m = xgb.XGBClassifier(n_estimators=n_iter, **prm).fit(a, ya, base_margin=za, verbose=False)
        return None, m.predict(t, base_margin=zt, output_margin=True), n_iter
    m = xgb.XGBClassifier(n_estimators=20000, early_stopping_rounds=300, **prm)
    m.fit(a, ya, base_margin=za, eval_set=[(b, yb)], base_margin_eval_set=[zb], verbose=False)
    it = m.best_iteration + 1
    print(f'    fold GLM alone {roc_auc_score(yb, zb):.5f}', flush=True)
    return m.predict(b, base_margin=zb, output_margin=True, iteration_range=(0, it)), None, it
run_refit('XGBoost depth 2 boosted from the joint GLM, full refit', xgbm_glm_boost)


### v22 members: the alternate feature pipeline as stack members

`FeatureEngineer` builds a different view of the same rows: median/mode imputation, group statistics over
`merchant_category` x `card_type`, frequency and count maps, smoothed target encodings of categoricals *and*
numeric values, pair and three-way interactions, and row-wise log/ratio/cyclical features -- all fitted inside
each fold. Three GBMs consume it; they enter the same `oof`/`pred` pool, so the stacker decides what they are worth.

In [ ]:
from itertools import combinations

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold


class _V22CFG:
    GROUPBY_COLS = ["merchant_category", "card_type"]
    GROUP_STAT_COLS = ["amount_usd", "velocity_score", "distance_from_home_km"]
    GROUP_STATS = ["mean", "std", "max", "median"]
    TE_SMOOTHING = 50.0
    TE_MIN_SAMPLES = 20
    USE_TARGET_ENCODING = True
    TE_N_FOLDS = 5
    RANDOM_SEED = 42
    USE_VALUE_ENCODING = True
    VALUE_TE_SMOOTHING = 50.0
    VALUE_TE_MIN_SAMPLES = 20
    VALUE_TE_MAX_UNIQUE = 50000


ID = 'id'
CFG = _V22CFG()

NUMERIC_COLS = [
    "amount_usd", "hours_since_last_txn", "txn_count_last_24h", "distance_from_home_km",
    "card_age_months", "customer_age", "account_balance_usd", "cvv_retry_count",
    "velocity_score", "time_of_day_hour", "day_of_week", "merchant_risk_score", "prior_disputes",
]
CATEGORICAL_COLS = ["merchant_category", "card_type", "auth_method", "channel", "device_type"]
BOOL_COLS = [
    "is_foreign_transaction", "is_new_merchant", "used_vpn", "ip_country_mismatch",
    "billing_shipping_mismatch", "is_ai_generated_scam_attempt",
]
VALUE_ENCODE_COLS = ["amount_usd", "hours_since_last_txn", "distance_from_home_km", "velocity_score", "merchant_risk_score"]
INTERACTION_PAIRS = [
    ("merchant_category", "channel"), ("auth_method", "channel"), ("device_type", "card_type"),
    ("merchant_category", "auth_method"), ("channel", "device_type"),
]
THREE_WAY_INTERACTION_COLS = list(combinations(CATEGORICAL_COLS, 3))

for c in NUMERIC_COLS + CATEGORICAL_COLS + BOOL_COLS:
    assert c in train.columns, f"missing train column: {c}"
    assert c in test.columns, f"missing test column: {c}"

print(f"numeric={len(NUMERIC_COLS)} categorical={len(CATEGORICAL_COLS)} bool={len(BOOL_COLS)}")
print(f"pair interactions={len(INTERACTION_PAIRS)} | three-way interactions={len(THREE_WAY_INTERACTION_COLS)}")


class FeatureEngineer:
    def __init__(
        self,
        numeric_cols=NUMERIC_COLS,
        categorical_cols=CATEGORICAL_COLS,
        bool_cols=BOOL_COLS,
        interaction_pairs=INTERACTION_PAIRS,
        three_way_cols=THREE_WAY_INTERACTION_COLS,
        groupby_cols=CFG.GROUPBY_COLS,
        group_stat_cols=CFG.GROUP_STAT_COLS,
        group_stats=CFG.GROUP_STATS,
        te_smoothing=CFG.TE_SMOOTHING,
        te_min_samples=CFG.TE_MIN_SAMPLES,
        use_target_encoding=CFG.USE_TARGET_ENCODING,
        te_n_folds=CFG.TE_N_FOLDS,
        te_random_state=CFG.RANDOM_SEED,
        value_cols=VALUE_ENCODE_COLS,
        use_value_encoding=CFG.USE_VALUE_ENCODING,
        value_te_smoothing=CFG.VALUE_TE_SMOOTHING,
        value_te_min_samples=CFG.VALUE_TE_MIN_SAMPLES,
        value_te_max_unique=CFG.VALUE_TE_MAX_UNIQUE,
    ):
        self.numeric_cols = list(numeric_cols)
        self.categorical_cols = list(categorical_cols)
        self.bool_cols = list(bool_cols)
        self.interaction_pairs = [tuple(p) for p in interaction_pairs]
        self.three_way_cols = [tuple(t) for t in three_way_cols]
        self.groupby_cols = list(groupby_cols)
        self.group_stat_cols = list(group_stat_cols)
        self.group_stats = list(group_stats)
        self.te_smoothing = float(te_smoothing)
        self.te_min_samples = float(te_min_samples)
        self.use_target_encoding = bool(use_target_encoding)
        self.te_n_folds = int(te_n_folds)
        self.te_random_state = int(te_random_state)
        self.value_cols_ = list(value_cols) if use_value_encoding else []
        self.use_value_encoding = bool(use_value_encoding)
        self.value_te_smoothing = float(value_te_smoothing)
        self.value_te_min_samples = float(value_te_min_samples)
        self.value_te_max_unique = int(value_te_max_unique)

        self.interaction_cols_ = [f"{a}_x_{b}" for a, b in self.interaction_pairs]
        self.three_way_cols_ = ["_x_".join(map(str, t)) for t in self.three_way_cols]
        self.freq_cols_ = self.categorical_cols + self.interaction_cols_ + self.three_way_cols_
        self.count_cols_ = self.categorical_cols + self.interaction_cols_
        self.te_cols_ = (self.categorical_cols + self.interaction_cols_) if self.use_target_encoding else []

        self.risk_threshold_ = None
        self.num_medians_ = {}
        self.cat_modes_ = {}
        self.group_stat_specs_ = []
        self.cat_freq_maps_ = {}
        self.cat_count_maps_ = {}
        self.te_maps_ = {}
        self.value_te_maps_ = {}
        self.global_mean_ = 0.0
        self.encoded_cols_ = []
        self.feature_names_ = None

    @staticmethod
    def _clean_y(y):
        a = np.asarray(y, dtype=float)
        if a.size == 0:
            return a
        finite = np.isfinite(a)
        if not finite.all():
            fill = float(np.mean(a[finite])) if finite.any() else 0.0
            a = a.copy(); a[~finite] = fill
        return a

    @staticmethod
    def _safe_div(num, den):
        den = pd.to_numeric(den, errors='coerce').astype(float)
        tiny = den.abs() < 1e-6
        den = den.mask(tiny, np.where(den >= 0, 1e-6, -1e-6))
        return num / den

    @staticmethod
    def _bool_int(s):
        v = pd.to_numeric(s, errors='coerce')
        if v.isna().any() and getattr(s, 'dtype', None) == object:
            mapped = s.map({'True': 1, 'False': 0, 'true': 1, 'false': 0, 'TRUE': 1, 'FALSE': 0, True: 1, False: 0})
            v = v.fillna(mapped)
        return v.fillna(0).astype(int)

    def _create_base_features(self, df):
        d = df.copy()
        required = {
            'amount_usd', 'account_balance_usd', 'hours_since_last_txn', 'distance_from_home_km',
            'velocity_score', 'txn_count_last_24h', 'card_age_months', 'customer_age',
            'merchant_risk_score', 'time_of_day_hour', 'day_of_week', 'prior_disputes',
        } | set(self.bool_cols)
        missing = sorted(c for c in required if c not in d.columns)
        if missing:
            raise ValueError(f"FeatureEngineer missing required columns: {missing}")

        for c in self.categorical_cols:
            if c in d.columns:
                d[c] = d[c].astype(object).fillna('Unknown').astype(str)

        medians = getattr(self, 'num_medians_', None) or {}
        for c in self.numeric_cols:
            if c not in d.columns:
                continue
            d[c] = pd.to_numeric(d[c], errors='coerce').replace([np.inf, -np.inf], np.nan)
            if c in medians:
                med = medians[c]
            else:
                med = d[c].median()
            d[c] = d[c].fillna(0.0 if pd.isna(med) else med)

        a = d['amount_usd']; b = d['account_balance_usd']; h = d['hours_since_last_txn']
        d['log_amount'] = np.log1p(a.clip(lower=0))
        d['log_balance'] = np.log1p(b.clip(lower=0))
        d['log_distance'] = np.log1p(d['distance_from_home_km'].clip(lower=0))
        d['log_velocity'] = np.log1p(d['velocity_score'].clip(lower=0))
        d['log_hours'] = np.log1p(h.clip(lower=0))
        d['log_txn_count'] = np.log1p(d['txn_count_last_24h'].clip(lower=0))

        d['amount_to_balance'] = self._safe_div(a, b + 1.0)
        d['balance_after'] = b - a
        d['balance_after_ratio'] = self._safe_div(b - a, b + 1.0)
        d['amount_per_card_month'] = self._safe_div(a, d['card_age_months'] + 1.0)
        d['amount_per_customer_year'] = self._safe_div(a, d['customer_age'] + 1.0)
        d['txn_per_hour'] = self._safe_div(d['txn_count_last_24h'], h + 1.0)
        d['distance_per_hour'] = self._safe_div(d['distance_from_home_km'], h + 1.0)
        d['velocity_per_txn'] = self._safe_div(d['velocity_score'], d['txn_count_last_24h'] + 1.0)
        d['velocity_x_distance'] = d['velocity_score'] * d['distance_from_home_km']
        d['amount_x_velocity'] = a * d['velocity_score']
        d['risk_x_amount'] = d['merchant_risk_score'] * d['log_amount']
        d['risk_x_velocity'] = d['merchant_risk_score'] * d['velocity_score']
        d['distance_x_risk'] = d['distance_from_home_km'] * d['merchant_risk_score']
        d['hours_x_distance'] = h * d['distance_from_home_km']
        d['disputes_x_amount'] = d['prior_disputes'] * a

        d['hour_sin'] = np.sin(2 * np.pi * d['time_of_day_hour'] / 24.0)
        d['hour_cos'] = np.cos(2 * np.pi * d['time_of_day_hour'] / 24.0)
        d['dow_sin'] = np.sin(2 * np.pi * d['day_of_week'] / 7.0)
        d['dow_cos'] = np.cos(2 * np.pi * d['day_of_week'] / 7.0)

        bool_df = pd.DataFrame({c: self._bool_int(d[c]) for c in self.bool_cols})
        d['risk_flags_sum'] = bool_df.sum(axis=1)
        d['is_night'] = d['time_of_day_hour'].isin([0, 1, 2, 3, 4, 5]).astype(int)
        d['foreign_vpn'] = (bool_df['is_foreign_transaction'].eq(1) & bool_df['used_vpn'].eq(1)).astype(int)
        d['mismatch_any'] = (bool_df['ip_country_mismatch'].eq(1) | bool_df['billing_shipping_mismatch'].eq(1)).astype(int)
        d['new_merchant_x_foreign'] = bool_df['is_new_merchant'] * bool_df['is_foreign_transaction']
        d['vpn_x_mismatch'] = bool_df['used_vpn'] * bool_df['ip_country_mismatch']

        risk_threshold = 70.0 if self.risk_threshold_ is None else self.risk_threshold_
        d['high_risk_flag'] = (d['merchant_risk_score'] >= risk_threshold).astype(int)

        for c1, c2 in self.interaction_pairs:
            if c1 in d.columns and c2 in d.columns:
                d[f'{c1}_x_{c2}'] = d[c1].astype(str) + '_' + d[c2].astype(str)
        for t in self.three_way_cols:
            if all(c in d.columns for c in t):
                name = '_x_'.join(map(str, t))
                d[name] = d[t[0]].astype(str) + '_' + d[t[1]].astype(str) + '_' + d[t[2]].astype(str)
        return d.replace([np.inf, -np.inf], np.nan)

    def _fit_te_maps(self, d, y):
        y_series = pd.Series(self._clean_y(y), index=d.index)
        global_mean = float(y_series.mean())
        if not np.isfinite(global_mean):
            global_mean = 0.0
        maps = {}
        for c in self.te_cols_:
            if c not in d.columns:
                continue
            stats = d.assign(_y=y_series).groupby(c, dropna=False)['_y'].agg(['mean', 'count'])
            smoothed = (stats['mean'] * stats['count'] + global_mean * self.te_smoothing) / (stats['count'] + self.te_smoothing)
            maps[c] = smoothed.where(stats['count'] >= self.te_min_samples, global_mean).to_dict()
        return maps, global_mean

    def _fit_value_te_maps(self, d, y):
        y_series = pd.Series(self._clean_y(y), index=d.index)
        global_mean = float(y_series.mean())
        if not np.isfinite(global_mean):
            global_mean = 0.0
        maps = {}
        for c in self.value_cols_:
            if c not in d.columns or int(d[c].nunique(dropna=True)) > self.value_te_max_unique:
                continue
            stats = d.assign(_y=y_series).groupby(c, dropna=False)['_y'].agg(['mean', 'count'])
            smoothed = (stats['mean'] * stats['count'] + global_mean * self.value_te_smoothing) / (stats['count'] + self.value_te_smoothing)
            maps[c] = smoothed.where(stats['count'] >= self.value_te_min_samples, global_mean).to_dict()
        return maps, global_mean

    def fit(self, X, y=None):
        X = X.reset_index(drop=True)
        y_arr = None if y is None else self._clean_y(y)
        if 'merchant_risk_score' in X.columns:
            mr = pd.to_numeric(X['merchant_risk_score'], errors='coerce')
            q = mr.quantile(0.90)
            self.risk_threshold_ = float(q) if np.isfinite(q) else 70.0
        else:
            self.risk_threshold_ = 70.0

        d = self._create_base_features(X)
        self.num_medians_ = {
            c: float(pd.to_numeric(d[c], errors='coerce').median())
            for c in self.numeric_cols if c in d.columns
        }
        self.num_medians_ = {c: (v if np.isfinite(v) else 0.0) for c, v in self.num_medians_.items()}

        self.cat_modes_ = {}
        for c in self.categorical_cols + self.bool_cols:
            if c not in d.columns:
                continue
            m = d[c].mode()
            self.cat_modes_[c] = m.iloc[0] if not m.empty else ('Unknown' if c in self.categorical_cols else 0)

        self.group_stat_specs_ = []
        for g in self.groupby_cols:
            if g not in d.columns:
                continue
            for s in self.group_stat_cols:
                if s not in d.columns:
                    continue
                try:
                    agg = d.groupby(g, dropna=False)[s].agg(list(self.group_stats))
                    for stat in self.group_stats:
                        name = f'{g}__{s}__{stat}'
                        try:
                            fb = float(d[s].agg(stat))
                        except Exception:
                            fb = 0.0
                        if not np.isfinite(fb):
                            fb = 0.0
                        mp = {k: (float(v) if np.isfinite(v) else fb) for k, v in agg[stat].to_dict().items()} if stat in agg.columns else {}
                        self.group_stat_specs_.append((name, g, s, stat, mp, fb))
                except Exception:
                    continue

        self.cat_freq_maps_ = {c: d[c].value_counts(normalize=True).to_dict() for c in self.freq_cols_ if c in d.columns}
        self.cat_count_maps_ = {c: d[c].value_counts().to_dict() for c in self.count_cols_ if c in d.columns}

        self.global_mean_ = float(np.mean(y_arr)) if y_arr is not None and len(y_arr) else 0.0
        if not np.isfinite(self.global_mean_):
            self.global_mean_ = 0.0
        if self.use_target_encoding and y_arr is not None:
            self.te_maps_, self.global_mean_ = self._fit_te_maps(d, y_arr)
            self.value_te_maps_, _ = self._fit_value_te_maps(d, y_arr)
        else:
            self.te_maps_ = {}
            self.value_te_maps_ = {}

        blocked = {ID, TARGET} | set(self.categorical_cols) | set(self.interaction_cols_) | set(self.three_way_cols_)
        base_cols = [c for c in d.columns if c not in blocked]
        freq_present = [c for c in self.freq_cols_ if c in d.columns]
        count_present = [c for c in self.count_cols_ if c in d.columns]
        te_present = [c for c in self.te_cols_ if c in d.columns] if self.use_target_encoding else []
        value_present = [c for c in self.value_cols_ if c in d.columns and c in self.value_te_maps_] if self.use_value_encoding else []
        self.encoded_cols_ = list(dict.fromkeys(
            [spec[0] for spec in self.group_stat_specs_]
            + [f'{c}_freq' for c in freq_present]
            + [f'{c}_cnt' for c in count_present]
            + [f'{c}__ve' for c in value_present]
            + [f'{c}_te' for c in te_present]
        ))
        self.feature_names_ = list(dict.fromkeys(base_cols + self.encoded_cols_))
        return self

    def fit_transform(self, X, y):
        X = X.reset_index(drop=True)
        y_arr = self._clean_y(y)
        self.fit(X, y_arr)

        te_oof = None
        value_oof = None
        if self.use_target_encoding and len(y_arr):
            y_int = np.clip(np.rint(y_arr), 0, 1).astype(int)
            counts = pd.Series(y_int).value_counts()
            if counts.size >= 2:
                n_splits = min(self.te_n_folds, int(counts.min()))
                if n_splits >= 2:
                    d_all = self._create_base_features(X)
                    te_oof = {c: np.full(len(X), np.nan) for c in self.te_cols_ if c in d_all.columns}
                    value_oof = {
                        c: np.full(len(X), np.nan) for c in self.value_cols_
                        if c in d_all.columns and int(d_all[c].nunique(dropna=True)) <= self.value_te_max_unique
                    }
                    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=self.te_random_state)
                    for tr_idx, va_idx in skf.split(d_all, y_int):
                        maps, gmean = self._fit_te_maps(d_all.iloc[tr_idx], y_arr[tr_idx])
                        vmaps, vgmean = self._fit_value_te_maps(d_all.iloc[tr_idx], y_arr[tr_idx])
                        for c in te_oof:
                            vals = pd.to_numeric(d_all.iloc[va_idx][c].map(maps.get(c, {})), errors='coerce')
                            te_oof[c][va_idx] = vals.fillna(gmean).to_numpy(dtype=float)
                        for c in value_oof:
                            vals = pd.to_numeric(d_all.iloc[va_idx][c].map(vmaps.get(c, {})), errors='coerce')
                            value_oof[c][va_idx] = vals.fillna(vgmean).to_numpy(dtype=float)
        return self._transform(X, te_oof=te_oof, value_oof=value_oof)

    def transform(self, X):
        return self._transform(X, te_oof=None, value_oof=None)

    def _transform(self, X, te_oof=None, value_oof=None):
        assert self.feature_names_ is not None, 'FeatureEngineer must be fitted before transform'
        X = X.reset_index(drop=True)
        d = self._create_base_features(X)

        for c in self.numeric_cols:
            if c in d.columns:
                d[c] = pd.to_numeric(d[c], errors='coerce').fillna(self.num_medians_.get(c, 0.0))
        for c in self.categorical_cols + self.bool_cols:
            if c in d.columns:
                default = 'Unknown' if c in self.categorical_cols else 0
                d[c] = d[c].fillna(self.cat_modes_.get(c, default))
        for c in self.bool_cols:
            if c in d.columns:
                d[c] = self._bool_int(d[c])

        for name, g, s, stat, mp, fb in self.group_stat_specs_:
            if g not in d.columns or s not in d.columns:
                continue
            key = d[g].astype(object).fillna('Unknown').astype(str)
            vals = pd.to_numeric(key.map(mp), errors='coerce').replace([np.inf, -np.inf], np.nan)
            d[name] = vals.fillna(fb)

        for c in self.freq_cols_:
            if c in d.columns and c in self.cat_freq_maps_:
                d[f'{c}_freq'] = pd.to_numeric(d[c].map(self.cat_freq_maps_[c]), errors='coerce').fillna(0.0)
        for c in self.count_cols_:
            if c in d.columns and c in self.cat_count_maps_:
                d[f'{c}_cnt'] = pd.to_numeric(d[c].map(self.cat_count_maps_[c]), errors='coerce').fillna(0.0).astype(float)

        if self.use_target_encoding:
            for c in self.te_cols_:
                if c not in d.columns:
                    continue
                col = f'{c}_te'
                if te_oof is not None and c in te_oof:
                    d[col] = pd.Series(te_oof[c], index=d.index).fillna(self.global_mean_).astype(float)
                elif c in self.te_maps_:
                    d[col] = pd.to_numeric(d[c].map(self.te_maps_[c]), errors='coerce').fillna(self.global_mean_)
                else:
                    d[col] = self.global_mean_
        if self.use_value_encoding:
            for c in self.value_cols_:
                if c not in d.columns:
                    continue
                col = f'{c}__ve'
                if value_oof is not None and c in value_oof:
                    d[col] = pd.Series(value_oof[c], index=d.index).fillna(self.global_mean_).astype(float)
                elif c in self.value_te_maps_:
                    d[col] = pd.to_numeric(d[c].map(self.value_te_maps_[c]), errors='coerce').fillna(self.global_mean_)
                else:
                    d[col] = self.global_mean_

        missing = [c for c in self.feature_names_ if c not in d.columns]
        assert not missing, f'Missing engineered feature columns: {missing[:10]}'
        out = d[self.feature_names_].copy()
        num_cols = out.select_dtypes(include=[np.number]).columns
        if len(num_cols):
            out[num_cols] = out[num_cols].replace([np.inf, -np.inf], np.nan)
            med = out[num_cols].median(numeric_only=True)
            out[num_cols] = out[num_cols].fillna(med).fillna(0.0)
            assert np.isfinite(out[num_cols].to_numpy(dtype=float)).all(), 'non-finite engineered features'
        assert not out.isna().any().any(), 'NaN engineered features'
        return out


fe_demo = FeatureEngineer()
X_demo = fe_demo.fit_transform(train.iloc[:20000], y.iloc[:20000])
X_demo_va = fe_demo.transform(test.iloc[:1000])
assert X_demo.shape[0] == 20000 and X_demo_va.shape[0] == 1000
assert X_demo.shape[1] == X_demo_va.shape[1]
assert list(X_demo.columns) == list(X_demo_va.columns)
assert not X_demo.isna().any().any() and not X_demo_va.isna().any().any()
_num_demo = X_demo.select_dtypes(include=[np.number])
assert np.isfinite(_num_demo.to_numpy(dtype=float)).all() and np.isfinite(X_demo_va.select_dtypes(include=[np.number]).to_numpy(dtype=float)).all()
print(f'feature matrix: {X_demo.shape[1]} columns')
print(f'group-stat features: {len(fe_demo.group_stat_specs_)}')
print(f'three-way freq features: {len(fe_demo.three_way_cols_)}')

In [ ]:
def _v22_fe_predict(kind, Xa, ya, Xb, yb, Xt):
    """Fit v22's FeatureEngineer inside the fold and train one GBM on its output."""
    fe = FeatureEngineer()
    Xtr = fe.fit_transform(Xa, ya)
    Xva = fe.transform(Xb)
    Xte = fe.transform(Xt)
    if kind == 'lgbm':
        m = lgb.LGBMClassifier(n_estimators=20000, learning_rate=0.05, num_leaves=16, min_child_samples=100,
                               subsample=0.8, subsample_freq=1, colsample_bytree=0.5, reg_lambda=1.0,
                               n_jobs=THREADS, verbose=-1)
        m.fit(Xtr, ya, eval_set=[(Xva, yb)], eval_metric='auc',
              callbacks=[lgb.early_stopping(300, verbose=False)])
        return m.predict_proba(Xva)[:, 1], m.predict_proba(Xte)[:, 1]
    if kind == 'xgb':
        m = xgb.XGBClassifier(n_estimators=20000, learning_rate=0.05, max_depth=3, min_child_weight=10,
                              subsample=0.8, colsample_bytree=0.5, tree_method='hist',
                              device='cuda' if GPU else 'cpu', enable_categorical=False,
                              eval_metric='auc', early_stopping_rounds=300, n_jobs=THREADS)
        m.fit(Xtr, ya, eval_set=[(Xva, yb)], verbose=False)
        return m.predict_proba(Xva)[:, 1], m.predict_proba(Xte)[:, 1]
    if kind == 'cat':
        m = CatBoostClassifier(iterations=3 * CAT_ITERS, learning_rate=0.08, depth=4, eval_metric='AUC',
                               od_type='Iter', od_wait=300, thread_count=THREADS, verbose=0,
                               random_seed=0, train_dir=str(OUT / 'catboost_info'),
                               task_type='GPU' if GPU else 'CPU')
        m.fit(Xtr, ya, eval_set=(Xva, yb))
        return m.predict_proba(Xva)[:, 1], m.predict_proba(Xte)[:, 1]
    raise ValueError(kind)


def v22_lgbm_fe(Xa, ya, Xb, yb, Xt): return _v22_fe_predict('lgbm', Xa, ya, Xb, yb, Xt)
def v22_xgb_fe(Xa, ya, Xb, yb, Xt):  return _v22_fe_predict('xgb',  Xa, ya, Xb, yb, Xt)
def v22_cat_fe(Xa, ya, Xb, yb, Xt):  return _v22_fe_predict('cat',  Xa, ya, Xb, yb, Xt)

run('v22 LightGBM engineered', v22_lgbm_fe, 'raw')
run('v22 XGBoost engineered',  v22_xgb_fe,  'raw')
run('v22 CatBoost engineered', v22_cat_fe,  'raw')

### Public prediction banks

Three public notebooks publish their out-of-fold and test predictions, so they can join the stack as members without
retraining: **lazyyeou** (Digit-Trie TE, Joint GLM and nested stack), **nilanjansaha123** (96-key digit trie, full refit)
and **yekenot** (RealMLP via pytabkit). Credit to them for the models; the stacker below decides how much each is worth.
Measured offline before adding them: our 35 v14 members stack to CV 0.95092, and the 12 banks together add +0.00020
(the two joint GLMs +0.00012 each, the XGBoost bags +0.00011; mikhailnaumov's XGBoost added nothing and is left out).
v16 scored 0.95176 with those (v15 0.95146). v17 adds four more: **goodpjw2008** Part 6 (GLM boost, +0.00036 on top of v16),
**megayak** (four GLM-boost members, up to +0.00036), **masayakawamata** (a same-split stack with GPT-2 token keys, the largest
single gain: +0.00060) and **ektarr** (RealMLP stack, +0.00012); all four together +0.00097 on v16's 47 members, offline.


In [ ]:
BANKS = {   # member name -> (part of the source notebook's slug, OOF file, test file); test = the author's full-data refit
    'bank lazyyeou joint GLM 96 keys': ('digit-trie-te', 'glm_joint_96key_oof.npy', 'glm_joint_96key_full.npy'),
    'bank lazyyeou XGBoost d2 bag': ('digit-trie-te', 'xgb_d2_bag_oof.npy', 'xgb_d2_bag_full.npy'),
    'bank lazyyeou XGBoost l4 bag': ('digit-trie-te', 'xgb_l4_bag_oof.npy', 'xgb_l4_bag_full.npy'),
    'bank lazyyeou LightGBM l4 bag': ('digit-trie-te', 'lgb_l4_bag_oof.npy', 'lgb_l4_bag_full.npy'),
    'bank lazyyeou LightGBM s100 bag': ('digit-trie-te', 'lgb_s100_bag_oof.npy', 'lgb_s100_bag_full.npy'),
    'bank lazyyeou LightGBM raw view': ('digit-trie-te', 'lgb_rawview_oof.npy', 'lgb_rawview_full.npy'),
    'bank nilanjan joint GLM 96 keys': ('96-key', 'glm_unified_96key_oof.npy', 'glm_unified_96key_full.npy'),
    'bank nilanjan XGBoost 96 keys bag': ('96-key', 'xgb_unified_96key_bag_oof.npy', 'xgb_unified_96key_bag_full.npy'),
    'bank nilanjan LightGBM 96 keys bag': ('96-key', 'lgb_unified_96key_bag_oof.npy', 'lgb_unified_96key_bag_full.npy'),
    'bank nilanjan LightGBM s100 bag': ('96-key', 'lgb_unified_s100_bag_oof.npy', 'lgb_unified_s100_bag_full.npy'),
    'bank nilanjan LightGBM l7': ('96-key', 'lgb_unified_l7_oof.npy', 'lgb_unified_l7_full.npy'),
    'bank goodpjw2008 GLM boost (Part 6)': ('s6e11-part-', 'oof_gb_s0.npy', 'refit_gb_s0.npy'),          # matched on the series prefix: the author renames the kernel with every part (6, 7, 8); same files
    'bank goodpjw2008 joint GLM (Part 6)': ('s6e11-part-', 'glmoof_gb_s0.npy', 'glmrefit_gb_s0.npy'),
    'bank megayak GLM boost': ('per-value-digit-te', 'oof_glm_boost.npy', 'refit_glm_boost.npy'),
    'bank megayak GLM boost XGBoost': ('per-value-digit-te', 'oof_glm_boost_xgb.npy', 'refit_glm_boost_xgb.npy'),
    'bank megayak GLM boost raw': ('per-value-digit-te', 'oof_glm_boost_raw.npy', 'refit_glm_boost_raw.npy'),
    'bank megayak GLM': ('per-value-digit-te', 'oof_glm_gen.npy', 'refit_glm_gen.npy'),
    'bank masayakawamata stack (GPT-2 keys)': ('logreg-stacking-same-split', 'oof.npy', 'test.npy'),
}
CSV_BANKS = {   # member name -> (slug part, OOF csv, its column, test csv); rows sorted by id
    'bank yekenot RealMLP': ('realmlp-pytabkit', 'oof_preds.csv', 'is_fraud', 'submission.csv'),
    'bank ektarr RealMLP stack': ('realmlp-stacking', 'oof_predictions.csv', 'stack', 'submission.csv'),
}
import glob as _glob
def _find(name, owner):
    hits = [h for h in _glob.glob(f'/kaggle/input/**/{name}', recursive=True) if owner in h.lower()]
    return min(hits, key=len) if hits else None
def _ok_pred(o, t, n_oof=len(X), n_test=len(XT)):
    o, t = np.asarray(o, float).ravel(), np.asarray(t, float).ravel()
    return (len(o) == n_oof and len(t) == n_test and np.isfinite(o).all() and np.isfinite(t).all()
            and np.unique(o).size > 1 and np.unique(t).size > 1)
def _aligned_csv(path, ids, col=None):
    df = pd.read_csv(path)
    if 'id' not in df.columns:
        return None
    c = col if col in df.columns else df.columns[-1]
    s = df.set_index('id')[c]
    v = s.reindex(ids).to_numpy(dtype=float)
    return v if np.isfinite(v).all() else None
for name, (owner, fo, ft) in BANKS.items():
    po, pt = _find(fo, owner), _find(ft, owner)
    if po is None or pt is None:
        print(f'{name}: not mounted, skipped'); continue
    try:
        o, t = np.load(po).ravel(), np.load(pt).ravel()
        if not _ok_pred(o, t):
            print(f'{name}: bad shape or non-finite values, skipped'); continue
        oof[name], pred[name], secs[name] = o, t, 0.0          # trained elsewhere: no minutes here
        print(f'{name}: OOF AUC {roc_auc_score(y, o):.5f}')
    except Exception as e:
        print(f'{name}: unreadable bank, skipped ({type(e).__name__}: {str(e)[:80]})')
for name, (owner, fo, col, ft) in CSV_BANKS.items():
    po, pt = _find(fo, owner), _find(ft, owner)
    if po is None or pt is None:
        print(f'{name}: not mounted, skipped'); continue
    try:
        o = _aligned_csv(po, train['id'].to_numpy(), col)
        t = _aligned_csv(pt, test['id'].to_numpy())
        if o is None or t is None or not _ok_pred(o, t):
            print(f'{name}: missing ids or bad values, skipped'); continue
        oof[name], pred[name], secs[name] = o, t, 0.0
        print(f'{name}: OOF AUC {roc_auc_score(y, o):.5f}')
    except Exception as e:
        print(f'{name}: unreadable bank, skipped ({type(e).__name__}: {str(e)[:80]})')

### v18: a GLM booster trained on ten folds

**What it is.** **sameershaikh27**'s [GLM Boost on Ten Folds](https://www.kaggle.com/code/sameershaikh27/s6e11-fraud-glm-boost-on-ten-folds)
(OOF 0.95159, LB 0.95118 alone): a plain GLM on the encoding logits, then 4-leaf LightGBM trees that start from the GLM's logit,
bagged over two encoder seeds. Its test prediction is a refit on all rows. Credit to Sameer Shaikh for the model and the published predictions.

**What is different.** Every other GLM booster in this stack trains on five folds, so each fold's encodings come from 80% of the rows.
This one uses ten folds: each fold's encodings come from 90% of the rows. Rare values (the median `account_balance_usd` value has 3 rows)
get a rate from more rows, so this member makes different errors on the rare rows.

**Why it is in.** We measured it offline before adding it: v17's 58 members, the same logistic stacker, the same 5-fold split.
It adds **+0.00015** (0.95210 -> 0.95224), the only one of 11 new public banks to pass our bar of +0.00005.
Its highest rank correlation with any member already in the stack is 0.9922. The 5-fold GLM boosters (goodpjw2008's 5-seed bag, 0.9989) add only +0.00004.
Same model, same data, a different number of folds: the encoder's training share is a free knob that adds diversity to a stack.

**What the board said (v21).** It did not hold up. With it, the stacker's held-out AUC rose and the public LB fell, twice:

| version | members | stacker held-out | public LB |
|---|---|---|---|
| v17 | 58, without the ten-fold member | 0.95212 | **0.95211** |
| v18 | 57 + the ten-fold member (two banks not mounted) | 0.95225 | 0.95195 |
| v20 | 58 + the ten-fold member | 0.95226 | 0.95202 |

goodpjw2008 measured the reason in [Part 9.2](https://www.kaggle.com/code/goodpjw2008/s6e11-part-12-wide-stack-2nd-lm-lb-0-95298):
a member's out-of-fold predictions come from models that saw 90% of the rows, but its test predictions are a refit on 100% of the rows, the same as every 5-fold member's.
So ten folds improve only the validation side. The stacker reads the 90%-of-rows OOF, believes the member is stronger than it will be on the test rows,
and gives it the second-largest weight (0.416 in v20). Credit to goodpjw2008 for the measurement, and to Sadam Torres, who first noticed the large weight.
The lesson for any stack: **every member's OOF must be made under the same conditions**, or the stacker's weights reward the difference in conditions, not the model.
The public 20% has a noise of about ±0.00035 (goodpjw2008 Part 7.4), so two scores are evidence, not proof; the member stays in v21 so that v21 changes one thing (the CNN).

**A correction (v22).** goodpjw2008 withdrew the strong form of this in Part 11.3: their wide stack contains ten-fold GLM boosts, its CV rose and its public score rose with it,
and "two public scores 0.00004 and 0.00007 apart cannot show an illusion". The test-side argument above still holds; the verdict does not.
In our v22 wide pool the ten-fold member takes the second-largest weight again (0.291 offline).


In [ ]:
V18_BANKS = {   # member name -> (part of the source notebook's slug, OOF file, test file); test = the author's full-data refit
    'bank sameershaikh27 GLM boost, ten folds': ('ten-folds', 'gb10_bag_oof.npy', 'gb10_bag_full.npy'),
}
for name, (owner, fo, ft) in V18_BANKS.items():
    po, pt = _find(fo, owner), _find(ft, owner)
    if po is None or pt is None:
        print(f'{name}: not mounted, skipped'); continue
    try:
        o, t = np.load(po).ravel(), np.load(pt).ravel()
        if not _ok_pred(o, t):
            print(f'{name}: bad shape or non-finite values, skipped'); continue
        oof[name], pred[name], secs[name] = o, t, 0.0
        print(f'{name}: OOF AUC {roc_auc_score(y, o):.5f}')
    except Exception as e:
        print(f'{name}: unreadable bank, skipped ({type(e).__name__}: {str(e)[:80]})')

### v21: a CNN that reads the digits

**What it is.** **queyrusi**'s [shared digit CNN](https://www.kaggle.com/code/queyrusi/fraud-a-shared-digit-cnn)
(OOF 0.94974, public LB 0.95132 in their 90/10 blend with trees). One small convolutional net reads the decimal string of each of 13 numeric columns,
character by character, with the same filters for every column. A DeepSets layer adds the per-column scores, and embeddings carry the categoricals.
Three seeds, five folds on the community split `StratifiedKFold(5, shuffle=True, random_state=42)`; the test side is the mean rank of all 15 models.
The predictions come from the author's dataset `queyrusi/fraud-shared-digit-cnn-weights`. Credit to queyrusi for the model, the weights and the cached predictions.

**What it does differently.** Every other member reads the digits through target encodings: a key such as `int(x/10)` is replaced by the fraud rate of rows with that key.
The CNN never sees a fraud rate. It learns its own digit patterns from the strings, so it can make errors the encoded members do not make.

**Why it is in.** We measured it offline before adding it, on v20's 59 members with the same stacker:
stacker held-out **+0.00010** on one 5-fold split of the OOF matrix and **+0.00005** on a second (0.95223 -> 0.95233 and 0.95226 -> 0.95231), at our bar of +0.00005 on both.
Its highest rank correlation with any member in the stack is 0.9795; the encoded tree banks sit at 0.99+ with each other.
goodpjw2008 ([Part 9.4](https://www.kaggle.com/code/goodpjw2008/s6e11-part-12-wide-stack-2nd-lm-lb-0-95298)) measured the same model on their stack: nested CV +0.00007, public LB 0.95235 -> 0.95224.
Their public 20% has a noise of about ±0.00035, so that one score does not decide it; v21 measures it on our stack.


In [ ]:
V21_NPZ_BANKS = {   # member name -> (part of the dataset path, npz file); the npz holds ids_train, ids_test, oof, test (ranks)
    'bank queyrusi digit CNN bag': ('digit-cnn', 'predictions.npz'),
}
for name, (owner, fz) in V21_NPZ_BANKS.items():
    pz = _find(fz, owner)
    if pz is None:
        print(f'{name}: not mounted, skipped'); continue
    try:
        z = np.load(pz)
        o = pd.Series(np.asarray(z['oof'], float).ravel(), index=z['ids_train']).reindex(train['id'].to_numpy()).to_numpy(dtype=float)
        t = pd.Series(np.asarray(z['test'], float).ravel(), index=z['ids_test']).reindex(test['id'].to_numpy()).to_numpy(dtype=float)
        if not _ok_pred(o, t):
            print(f'{name}: missing ids or bad values, skipped'); continue
        oof[name], pred[name], secs[name] = o, t, 0.0
        print(f'{name}: OOF AUC {roc_auc_score(y, o):.5f}')
    except Exception as e:
        print(f'{name}: unreadable bank, skipped ({type(e).__name__}: {str(e)[:80]})')

### v22: when a source notebook's output disappears

A bank member is read from the newest version of its author's notebook. On 2026-11-06 masayakawamata reran
[LogReg Stacking, Same-Split OOFs](https://www.kaggle.com/code/masayakawamata/s6e11-logreg-stacking-same-split-oofs) as a code-only update, and that version has no output files.
Our stack's largest weight (0.736 in v21) would have dropped out without an error: the loader above prints "not mounted, skipped" and goes on.
The author kept the old files in the dataset [S6E11 Public Stack v10 Output Archive](https://www.kaggle.com/datasets/masayakawamata/s6e11-public-stack-v10-outputs).
We checked them against the copy v17-v21 used: `oof.npy` and `test.npy` are byte-identical (OOF AUC 0.95153). Credit and thanks to Masaya Kawamata for keeping the archive.
The cell below reads the archive only if the notebook output is missing. The lesson for anyone stacking public banks: **print every member you expected and did not get**, and count them before the stacker runs.


In [ ]:
V22_ARCHIVE_BANKS = {   # member name -> (part of the dataset path, OOF file, test file); used only if the notebook output did not mount
    'bank masayakawamata stack (GPT-2 keys)': ('public-stack-v10-outputs', 'oof.npy', 'test.npy'),
}
for name, (owner, fo, ft) in V22_ARCHIVE_BANKS.items():
    if name in oof:
        print(f'{name}: from the notebook output'); continue
    po, pt = _find(fo, owner), _find(ft, owner)
    if po is None or pt is None:
        print(f'{name}: archive not mounted either, skipped'); continue
    try:
        o, t = np.load(po).ravel(), np.load(pt).ravel()
        if not _ok_pred(o, t):
            print(f'{name}: bad shape or non-finite values, skipped'); continue
        oof[name], pred[name], secs[name] = o, t, 0.0
        print(f'{name}: from the archive, OOF AUC {roc_auc_score(y, o):.5f}')
    except Exception as e:
        print(f'{name}: unreadable archive, skipped ({type(e).__name__}: {str(e)[:80]})')
expected = list(BANKS) + list(CSV_BANKS) + list(V18_BANKS) + list(V21_NPZ_BANKS)
print('bank members expected', len(expected), '| loaded', sum(k in oof for k in expected), '| missing', [k for k in expected if k not in oof])

### v22: frozen copies of a bank that keeps changing

**What happened.** megayak reran [Per-Value + Digit TE | Shallow-Tree Stack](https://www.kaggle.com/code/megayak/s6e11-per-value-digit-te-shallow-tree-stack)
on 2026-11-05 and again on 2026-11-07. The 11-05 version no longer writes `glm_gen` (our member "bank megayak GLM"), and its `glm_boost_raw` changed
(OOF AUC 0.95100 -> 0.95115, rank correlation with the old file 0.99645). The loader above reads the newest version, so a push would have changed our stack silently.

**What we do.** The four files v17-v21 stacked are frozen in our dataset [S6E11 frozen public prediction banks](https://www.kaggle.com/datasets/alexchilton/s6e11-frozen-public-banks),
byte for byte as megayak published them on 2026-11-04 (OOF AUC 0.95137, 0.95132, 0.95100, 0.95047, the same numbers v21's log printed).
The cell below reads them in place of the live notebook. The live notebook's new members (`glm_cnt`, `glm_boost_c`) go into the wide pool further down.
Credit for the models stays with megayak; please upvote the source.


In [ ]:
V22_FROZEN_BANKS = {   # member name -> (OOF file, test file) in our frozen dataset; replaces whatever the live source publishes today
    'bank megayak GLM boost': ('oof_glm_boost.npy', 'refit_glm_boost.npy'),
    'bank megayak GLM boost XGBoost': ('oof_glm_boost_xgb.npy', 'refit_glm_boost_xgb.npy'),
    'bank megayak GLM boost raw': ('oof_glm_boost_raw.npy', 'refit_glm_boost_raw.npy'),
    'bank megayak GLM': ('oof_glm_gen.npy', 'refit_glm_gen.npy'),
}
for name, (fo, ft) in V22_FROZEN_BANKS.items():
    po, pt = _find(fo, 'frozen-public-banks'), _find(ft, 'frozen-public-banks')
    if po is None or pt is None:
        print(f'{name}: frozen copy not mounted, kept the live one' if name in oof else f'{name}: not mounted, skipped'); continue
    try:
        o, t = np.load(po).ravel(), np.load(pt).ravel()
        if not _ok_pred(o, t):
            print(f'{name}: frozen copy has bad shape or non-finite values, kept live' if name in oof else f'{name}: frozen copy bad, skipped'); continue
        live = 'live file differs' if name in oof and not np.array_equal(oof[name], o) else 'same as live' if name in oof else 'gone from the live source'
        oof[name], pred[name], secs[name] = o, t, 0.0
        print(f'{name}: frozen copy, OOF AUC {roc_auc_score(y, o):.5f} ({live})')
    except Exception as e:
        print(f'{name}: frozen copy unreadable ({type(e).__name__}: {str(e)[:80]})')
print('bank members expected', len(expected), '| loaded', sum(k in oof for k in expected), '| missing', [k for k in expected if k not in oof])

### Blend

Each model's out-of-fold predictions are turned into ranks (AUC only sees order). Two ways to combine them, compared on rows they did not fit:

- a **hill climb** that adds, with replacement, whichever model raises the blend's OOF AUC most;
- a **logistic-regression stacker** on the members' logit-ranks, the combiner every s6e8 and s6e9 top write-up used.

Both are refitted inside a 5-fold split of the OOF matrix, so neither is scored on the rows it was fitted on; the better one makes the submission.


In [ ]:

from scipy.stats import rankdata
R = {k: rankdata(v) / len(v) for k, v in oof.items()}; RT = {k: rankdata(v) / len(v) for k, v in pred.items()}
names = list(R)                                           # the hill climb gives a weaker LightGBM variant weight 0 itself
display(pd.DataFrame({k: [roc_auc_score(y, oof[k]), secs[k] / 60] for k in R}, index=['OOF AUC', 'minutes']).T.round(5))
display(pd.DataFrame(np.corrcoef([R[k] for k in names]), index=names, columns=names).round(3))
w = {k: 0 for k in names}; best, cur = -1, np.zeros(len(X))
for step in range(30):
    cand = {k: roc_auc_score(y, (cur * step + R[k]) / (step + 1)) for k in names}
    k = max(cand, key=cand.get)
    if cand[k] <= best + 1e-7: break
    best = cand[k]; w[k] += 1; cur = (cur * step + R[k]) / (step + 1)
w = {k: v / sum(w.values()) for k, v in w.items()}
print('weights', {k: round(v, 3) for k, v in w.items()}, f'| blend OOF AUC {best:.5f} | equal-weight rank mean {roc_auc_score(y, np.mean([R[k] for k in names], 0)):.5f}')
hc_test = sum(w[k] * RT[k] for k in names)

# a logistic-regression stacker on the members' logit-ranks, scored by its own 5-fold CV over the OOF matrix (so it is not judged on rows it fitted)
from scipy.special import logit
Z = np.column_stack([logit(np.clip(R[k], 1e-6, 1 - 1e-6)) for k in names]); ZT = np.column_stack([logit(np.clip(RT[k], 1e-6, 1 - 1e-6)) for k in names])
st_runs = []                                              # three outer splits: the spread is the noise of this comparison
st_hold = {}                                              # v25: the held-out predictions per split, for the two-stack mix below
for seed in (7, 8, 9):
    st_oof = np.zeros(len(Z))
    for tr_i, va_i in StratifiedKFold(5, shuffle=True, random_state=seed).split(Z, y):
        st_oof[va_i] = LogisticRegression(C=0.05, max_iter=1000).fit(Z[tr_i], y.iloc[tr_i]).decision_function(Z[va_i])
    st_runs.append(roc_auc_score(y, st_oof)); st_hold[seed] = st_oof.copy()
st_auc = st_runs[0]                                       # split 7, the one the hill climb is refitted on below
print(f'logistic stacker held-out AUC over 3 outer splits: {np.mean(st_runs):.5f} +- {np.std(st_runs):.5f}', np.round(st_runs, 5))
# the hill climb's own number is fitted on the same OOF it is scored on; re-fit it per fold the same way for a fair comparison
hc_oof = np.zeros(len(Z))
for tr_i, va_i in StratifiedKFold(5, shuffle=True, random_state=7).split(Z, y):
    ww = {k: 0 for k in names}; bb, cc = -1, np.zeros(len(tr_i))
    for step in range(30):
        cand = {k: roc_auc_score(y.iloc[tr_i], (cc * step + R[k][tr_i]) / (step + 1)) for k in names}
        k = max(cand, key=cand.get)
        if cand[k] <= bb + 1e-7: break
        bb = cand[k]; ww[k] += 1; cc = (cc * step + R[k][tr_i]) / (step + 1)
    hc_oof[va_i] = sum(ww[k] * R[k][va_i] for k in names) / max(1, sum(ww.values()))   # normalised: folds must share one scale
hc_auc = roc_auc_score(y, hc_oof)
stk = LogisticRegression(C=0.05, max_iter=1000).fit(Z, y)
print(f'held-out AUC: hill climb {hc_auc:.5f}, logistic stacker {st_auc:.5f}; stacker weights', dict(zip(names, np.round(stk.coef_[0], 3))))
blend = stk.decision_function(ZT) if st_auc > hc_auc else hc_test
print('submitting the', 'logistic stacker' if st_auc > hc_auc else 'hill climb')
sub = pd.DataFrame({'id': test['id'], TARGET: rankdata(blend) / len(blend)}); sub.to_csv(OUT / 'submission.csv', index=False)
print(sub.shape, sub.head(3).to_dict('records'))


### What each member is worth to the blend

Left: each model alone. Right: how much the stacker's held-out AUC falls when that one model is removed and the stacker is refitted.
A model can be strong on the left and worth nothing on the right, when another member already carries the same information.


In [ ]:

drop = {}                                                 # 3 folds here (one refit per member), so the base is refitted the same way
base3 = np.zeros(len(Z))
for tr_i, va_i in StratifiedKFold(3, shuffle=True, random_state=7).split(Z, y):
    base3[va_i] = LogisticRegression(C=0.05, max_iter=1000).fit(Z[tr_i], y.iloc[tr_i]).decision_function(Z[va_i])
base3 = roc_auc_score(y, base3)
for j, k in enumerate(names):
    keep = [i for i in range(len(names)) if i != j]; o = np.zeros(len(Z))
    for tr_i, va_i in StratifiedKFold(3, shuffle=True, random_state=7).split(Z, y):
        o[va_i] = LogisticRegression(C=0.05, max_iter=1000).fit(Z[tr_i][:, keep], y.iloc[tr_i]).decision_function(Z[va_i][:, keep])
    drop[k] = base3 - roc_auc_score(y, o)
own = pd.Series({k: roc_auc_score(y, oof[k]) for k in names}).sort_values()
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 0.38 * len(names) + 1.5), sharey=True)
a1.barh(own.index, own.values, color='#9bb7d4'); a1.set_xlim(own.min() - 0.002, own.max() + 0.0005); a1.set_title('OOF AUC alone')
d = pd.Series(drop)[own.index]; a2.barh(d.index, d.values * 1e4, color=['#4a6fa5' if v > 0 else '#c0392b' for v in d.values])
a2.axvline(0, color='#999', lw=0.8); a2.set_title('AUC lost when removed (x 1e-4)'); plt.tight_layout(); plt.show()


### v22: a wide pool of public members, and why the top public notebooks score above our stack

**The question.** On 2026-11-07 the best public notebooks scored 0.95277-0.95285 and our best was 0.95211.
[goodpjw2008 Part 11](https://www.kaggle.com/code/goodpjw2008/s6e11-part-12-wide-stack-2nd-lm-lb-0-95298) (LB 0.95285; the notebook is now called Part 12) is a 70 / 30 rank blend of two stacks;
[sameershaikh27's Wide Pooled Stack](https://www.kaggle.com/code/sameershaikh27/s6e11-fraud-wide-pooled-stack-every-bank-found) (nested CV 0.95241, LB 0.95277) is one stack of 178 public members.
Is the gain in the blend of public files, or in the width of the pool? We measured both on our own OOF.

**What it is.** Every public bank published on the same folds (`StratifiedKFold(5, shuffle=True, random_state=42)`) that this stack did not have yet:
sadamtorres' [TabPFN and GBDT pairs](https://www.kaggle.com/datasets/sadamtorres/ps-s6e11-tabpfn-gbdt-oof-test-pairs) (14),
masayakawamata's [GPT-2 target-first classifier](https://www.kaggle.com/datasets/masayakawamata/s6e11-gpt2-target-first-oof) and [TabPack](https://www.kaggle.com/datasets/masayakawamata/s6e11-tabpack-cpu-starter-predictions),
sameershaikh27's [digit-sequence CNN](https://www.kaggle.com/code/sameershaikh27/s6e11-fraud-digit-sequence-cnn-member),
wguesdon's [tree ensemble predictions](https://www.kaggle.com/datasets/wguesdon/s6e11-tree-ensemble-predictions) (8),
malyshevdanil's [pooled stack v9 bank](https://www.kaggle.com/datasets/malyshevdanil/s6e11-pooled-stack-v9) (32),
goodpjw2008's [OOF bank](https://www.kaggle.com/datasets/goodpjw2008/s6e11-oof-bank) (70) and megayak's two new GLM members.
Where a bank publishes its fold ids, the cell checks that they are our partition.
A member enters if its OOF AUC is within 0.008 of the best and its rank correlation with every member already in is below 0.99995 (sameershaikh27's and goodpjw2008's rules).
The same logistic stacker as above then runs on all of them, at `C = 0.01` instead of 0.05.

**What we measured offline** (v21's 60 members, held-out AUC of the stacker on outer splits 7 and 11; `scratch/banks/wide_1107.py` in github.com/alexchilton/kaggley):

| arm | split 7 | split 11 |
|---|---|---|
| v21's 60 members, C 0.05 (v21 as submitted) | 0.95233 | 0.95232 |
| v21's 60 members, C 0.01 | 0.95231 | 0.95230 |
| 60 + 106 public members, C 0.05 | 0.95246 | 0.95238 |
| **60 + 106 public members, C 0.01** | **0.95250** | **0.95245** |
| 60 + goodpjw2008's wide stack as one member, C 0.05 | 0.95241 | 0.95240 |
| goodpjw2008's wide stack alone (its nested OOF) | 0.95250 | 0.95250 |
| rank blend 0.7 goodpjw2008 wide stack + 0.3 our wide stack | 0.95252 | 0.95251 |

**What it says.**
1. **The width is the gain; the public-file blend is not.** 106 more members add +0.00017 / +0.00013. Blending our wide stack with goodpjw2008's adds +0.00002 / +0.00006 on the OOF.
   goodpjw2008's blend moved the public board +0.00018 (0.95267 -> 0.95285), inside the ±0.0003 that a 20% public sample moves by chance.
   Our wide stack alone reaches what goodpjw2008's reaches alone (0.95250).
2. **One member at a time understates a pool.** We tested the GPT-2 target-first member alone on 2026-11-03: -0.00010. In the wide pool it takes the largest stacker weight (0.461 offline), then
   sameershaikh27's ten-fold GLM boost (0.291) and sadamtorres' GLM-boost pairs (0.235, 0.216). HYPOTHESIS (unverified): these members correct each other,
   and a member whose errors only help once another member is there cannot pass a one-at-a-time test.
3. **Regularisation has to grow with width.** `C = 0.01` costs 0.00002 on 60 members and gains 0.00004-0.00007 on 166.

**Credit.** Sameer Shaikh built the first wide pool and the gate; goodpjw2008 rebuilt it and published the blend; every bank author above trained the members. Please upvote them.
The cell below submits the wide stack if its held-out AUC over splits 7, 8 and 9 is above the 60-member stacker's.


In [ ]:
import os, json, glob as _g, time as _time
from joblib import Parallel, delayed

# ---------------------------------------------------------------------------
# Checkpoint everything already computed BEFORE touching external banks or
# spawning workers: a failure below must not cost the 9 hours spent above.
# ---------------------------------------------------------------------------
CKPT = OUT / 'v22_wide_pool_checkpoints'
CKPT.mkdir(parents=True, exist_ok=True)

def _log(*a):
    print(f'[{_time.strftime("%H:%M:%S")}]', *a, flush=True)

def _atomic(path, write_fn):
    path = Path(path)
    tmp = path.with_name(path.name + '.tmp')
    try:
        with open(tmp, 'wb') as f:
            write_fn(f)
        os.replace(tmp, path)                                # atomic: no half-written checkpoints
    except Exception as e:
        _log('checkpoint WRITE FAILED', path.name, type(e).__name__, str(e)[:160])
        try: tmp.unlink(missing_ok=True)
        except Exception: pass

def _ckpt(tag):
    g = globals()
    arrays, meta = {}, {'tag': tag, 'time': _time.strftime('%Y-%m-%d %H:%M:%S')}
    def _put(prefix, mapping):
        order = list(mapping)
        meta[prefix + '_names'] = order
        for j, k in enumerate(order):
            try: arrays[f'{prefix}_{j:03d}'] = np.asarray(mapping[k], dtype=np.float64)
            except Exception as e: _log('checkpoint skipped', prefix, k, type(e).__name__)
    _put('oof', g.get('oof', {}))
    _put('pred', g.get('pred', {}))
    _put('R', g.get('R', {}))
    _put('RT', g.get('RT', {}))
    pool_ = g.get('pool') or {}
    _put('pool_oof', {k: v[0] for k, v in pool_.items()})
    _put('pool_test', {k: v[1] for k, v in pool_.items()})
    _put('weights', g.get('w', {}))
    for key in ('y', 'Z', 'ZT', 'ZW', 'ZWT', 'hc_oof', 'blend', 'st_runs', 'wide_runs'):
        v = g.get(key)
        if v is None: continue
        try: arrays[key] = np.asarray(v)
        except Exception as e: _log('checkpoint skipped', key, type(e).__name__)
    arrays['train_id'] = np.asarray(train['id'].values)
    arrays['test_id'] = np.asarray(test['id'].values)
    for dkey in ('st_hold', 'wide_hold'):
        for seed, v in (g.get(dkey) or {}).items():
            arrays[f'{dkey}_{seed}'] = np.asarray(v)
    meta.update({k: g.get(k) for k in ('names', 'wnames', 'kept', 'st_runs', 'wide_runs', 'secs', 'pauc')})
    _atomic(CKPT / f'{tag}.npz', lambda f: np.savez(f, **arrays))
    _atomic(CKPT / f'{tag}.json', lambda f: f.write(json.dumps(meta, default=str).encode('utf-8')))
    if g.get('sub') is not None:
        _atomic(CKPT / 'submission_checkpoint.csv', lambda f: f.write(g['sub'].to_csv(index=False).encode('utf-8')))
    _log('checkpoint saved', tag, len(arrays), 'arrays ->', CKPT)

_ckpt('00_before_external_banks')

pool = {}                                                 # member name -> (OOF in train order, test in test order)
def _pool_add(n, o, t, where=''):
    try:
        o = np.asarray(o, dtype=np.float64).ravel()
        t = np.asarray(t, dtype=np.float64).ravel()
    except Exception as e:
        _log('SKIP', n, where, 'not numeric', type(e).__name__, str(e)[:100]); return False
    bad = []
    if len(o) != len(X): bad.append(f'OOF len {len(o)} != len(X) {len(X)}')
    if len(t) != len(XT): bad.append(f'test len {len(t)} != len(XT) {len(XT)}')
    if not np.isfinite(o).all(): bad.append('OOF has NaN/inf')
    if not np.isfinite(t).all(): bad.append('test has NaN/inf')
    if np.unique(o).size <= 1: bad.append('OOF constant')
    if np.unique(t).size <= 1: bad.append('test constant')
    if bad:
        _log('SKIP', n, where, '; '.join(bad)); return False
    pool[n] = (o.copy(), t.copy())
    _log('added', n, f'{len(o)}/{len(t)}', where)
    return True

def _file(p, n):
    if p and os.path.isfile(p): return True
    _log('SKIP', n, 'file missing:', p); return False

def _npz_keys(path, needed, n):
    try:
        with np.load(path, allow_pickle=False) as z:
            files = list(z.files)
        missing = [k for k in needed if k not in files]
    except Exception as e:
        _log('SKIP', n, 'np.load failed', type(e).__name__, str(e)[:120]); return None
    if missing:
        _log('SKIP', n, 'npz missing keys', missing, 'has', files); return None
    return files

def _dir(fname, owner):
    p = _find(fname, owner)
    _log('bank source', owner, '->', p if p else 'NOT MOUNTED')
    return None if p is None else os.path.dirname(p)
fold42 = np.zeros(len(X), int)
for f_, (_, va_i) in enumerate(StratifiedKFold(5, shuffle=True, random_state=42).split(np.zeros(len(X)), y)): fold42[va_i] = f_
def _same_folds(f, n='folds'):
    try:
        f = np.asarray(f).ravel()
        if len(f) != len(X):
            _log('SKIP', n, f'fold len {len(f)} != len(X) {len(X)}'); return False
        if not np.isfinite(f).all():
            _log('SKIP', n, 'fold ids contain NaN/inf'); return False
    except Exception as e:
        _log('SKIP', n, 'fold ids not numeric', type(e).__name__, str(e)[:100]); return False
    ok = all(len(set(f[fold42 == k])) == 1 for k in range(5))
    if not ok: _log('SKIP', n, 'fold ids do not match fold42')
    return ok
def _folds_csv_ok(path, n):
    if not _file(path, n): return False
    try:
        df = pd.read_csv(path)
        miss = [c for c in ('id', 'fold') if c not in df.columns]
        if miss:
            _log('SKIP', n, 'folds.csv missing columns', miss); return False
        return _same_folds(df.set_index('id').reindex(tr_id)['fold'].values, n)
    except Exception as e:
        _log('SKIP', n, 'folds.csv unreadable', type(e).__name__, str(e)[:120]); return False
def _csv_indexed(path, ids, cols, n):
    if not _file(path, n): return None
    try: df = pd.read_csv(path)
    except Exception as e:
        _log('SKIP', n, 'CSV unreadable', type(e).__name__, str(e)[:120]); return None
    miss = [c for c in ('id',) + tuple(cols) if c not in df.columns]
    if miss:
        _log('SKIP', n, 'CSV missing columns', miss); return None
    try:
        s = df.set_index('id')
        if s.index.has_duplicates:
            _log('SKIP', n, 'duplicate ids in CSV'); return None
        missing = len(ids) - int(np.isin(ids, s.index.values).sum())
        if missing:
            _log('SKIP', n, f'{missing} ids missing from CSV'); return None
        return s.reindex(ids)
    except Exception as e:
        _log('SKIP', n, 'id alignment failed', type(e).__name__, str(e)[:120]); return None
tr_id, te_id = train['id'].values, test['id'].values
def _by_id(df, ids, n):
    if 'id' not in df.columns:
        _log('SKIP', n, 'no id column'); return None
    s = pd.Series(np.asarray(df.iloc[:, -1], dtype=np.float64), index=df['id'].values)
    if pd.Index(s.index).has_duplicates:
        _log('SKIP', n, 'duplicate ids in CSV'); return None
    missing = len(ids) - int(np.isin(ids, s.index.values).sum())
    if missing:
        _log('SKIP', n, f'{missing} ids missing from CSV'); return None
    return s.reindex(ids).to_numpy(dtype=np.float64)
d = _dir('pair_oof_v5_glm.csv', 'tabpfn-gbdt-oof-test-pairs')
for p in sorted(_g.glob(f'{d}/pair_oof_*.csv')) if d else []:
    n = os.path.basename(p)[len('pair_oof_'):-4]
    tp = f'{d}/pair_test_{n}.csv'
    if not _file(tp, f'sadamtorres {n} test'): continue
    try:
        o = _by_id(pd.read_csv(p), tr_id, f'sadamtorres {n} OOF')
        t = _by_id(pd.read_csv(tp), te_id, f'sadamtorres {n} test')
        if o is not None and t is not None: _pool_add(f'sadamtorres {n}', o, t, p)
    except Exception as e: _log('SKIP sadamorres', n, type(e).__name__, str(e)[:120])
d = _dir('fold_ids.npy', 'gpt2-target-first-oof')
if d:
    try:
        pf, po, pt = f'{d}/fold_ids.npy', f'{d}/oof.npy', f'{d}/test.npy'
        if all(_file(p, 'gpt2-target-first') for p in (pf, po, pt)):
            fi = np.load(pf, allow_pickle=False)
            if _same_folds(fi, 'gpt2-target-first folds'):
                _pool_add('masayakawamata GPT-2 target-first', np.load(po, allow_pickle=False), np.load(pt, allow_pickle=False), d)
    except Exception as e: _log('SKIP gpt2-target-first', type(e).__name__, str(e)[:120])
d = _dir('oof_tabpack.csv', 'tabpack-cpu-starter')
if d:
    try:
        to = _csv_indexed(f'{d}/oof_tabpack.csv', tr_id, ('fold', 'is_fraud'), 'tabpack OOF')
        tt = _csv_indexed(f'{d}/test_tabpack.csv', te_id, ('is_fraud',), 'tabpack test')
        if to is not None and tt is not None and _same_folds(to['fold'].values, 'tabpack folds'):
            _pool_add('masayakawamata TabPack', to['is_fraud'].values, tt['is_fraud'].values, d)
    except Exception as e: _log('SKIP tabpack', type(e).__name__, str(e)[:120])
d = _dir('dcnn_bag_oof.npy', 'digit-sequence-cnn')
if d:
    po, pt = f'{d}/dcnn_bag_oof.npy', f'{d}/dcnn_bag_full.npy'
    if all(_file(p, 'digit-sequence CNN') for p in (po, pt)):
        try: _pool_add('sameershaikh27 digit-sequence CNN', np.load(po, allow_pickle=False), np.load(pt, allow_pickle=False), d)
        except Exception as e: _log('SKIP digit-sequence CNN', type(e).__name__, str(e)[:120])
d = _dir('folds.csv', 'tree-ensemble-predictions')
if d and _folds_csv_ok(f'{d}/folds.csv', 'tree-ensemble-predictions'):
    for p in sorted(_g.glob(f'{d}/*.npz')):
        n = os.path.basename(p)[:-4]
        try:
            if _npz_keys(p, ('train_id', 'test_id'), f'wguesdon {n}') is None: continue
            with np.load(p, allow_pickle=False) as z:
                files = list(z.files)
                okey = next((k for k in ('stack_oof', 'seed_oof', 'oof') if k in files), None)
                tkey = next((k for k in ('stack_test', 'fold_test') if k in files), None)
                if okey is None or tkey is None:
                    _log('SKIP wguesdon', n, 'missing prediction keys, has', files); continue
                o, t = z[okey], z[tkey]
                if okey == 'seed_oof': o = np.asarray(o).mean(1)
                if tkey == 'fold_test': t = np.asarray(t).mean(0)
                o = pd.Series(o, index=z['train_id']).reindex(tr_id).values
                t = pd.Series(t, index=z['test_id']).reindex(te_id).values
                _pool_add(f'wguesdon {n}', o, t, p)
        except Exception as e: _log('SKIP wguesdon', n, type(e).__name__, str(e)[:120])
d = _dir('pool_bank.npz', 'pooled-stack-v9')
if d:
    pz, pm = f'{d}/pool_bank.npz', f'{d}/pool_manifest.json'
    if _file(pz, 'pooled-stack-v9') and _file(pm, 'pooled-stack-v9 manifest'):
        try:
            if _npz_keys(pz, ('y', 'oof', 'test'), 'pooled-stack-v9') is not None:
                with np.load(pz, allow_pickle=False) as z, open(pm, encoding='utf-8') as fh:
                    mn = json.load(fh)['names']
                    yb, ob, tb = z['y'], z['oof'], z['test']
                if not isinstance(mn, (list, tuple)): _log('SKIP pooled-stack-v9', 'manifest names is not a list')
                elif yb.ndim != 1 or len(yb) != len(X): _log('SKIP pooled-stack-v9', f'y shape {yb.shape} != ({len(X)},)')
                elif not (yb == y.values).all(): _log('SKIP pooled-stack-v9', 'y values differ')
                elif (ob.ndim != 2 or tb.ndim != 2 or ob.shape[1] != len(mn) or tb.shape[1] != len(mn)):
                    _log('SKIP pooled-stack-v9', 'oof/test shape vs manifest', ob.shape, tb.shape, len(mn))
                else:
                    for j, n in enumerate(mn): _pool_add(f'malyshevdanil {n}', ob[:, j], tb[:, j], pz)
        except Exception as e: _log('SKIP pooled-stack-v9', type(e).__name__, str(e)[:120])
d = _dir('folds.csv', 's6e11-oof-bank')
if d and _folds_csv_ok(f'{d}/folds.csv', 's6e11-oof-bank'):
    for p in sorted(_g.glob(f'{d}/oof_*.npy')):
        n = os.path.basename(p)[4:-4]
        tp = f'{d}/test_{n}.npy'
        if not _file(tp, f'goodpjw2008 {n} test'): continue
        try: _pool_add(f'goodpjw2008 {n}', np.load(p, allow_pickle=False), np.load(tp, allow_pickle=False), d)
        except Exception as e: _log('SKIP goodpjw2008', n, type(e).__name__, str(e)[:120])
for n in ['glm_cnt', 'glm_boost_c']:
    po, pt = _find(f'oof_{n}.npy', 'per-value-digit-te'), _find(f'refit_{n}.npy', 'per-value-digit-te')
    _log('bank source megayak', n, '->', po if po else 'NOT MOUNTED', pt if pt else 'NOT MOUNTED')
    if po and pt:
        try: _pool_add(f'megayak {n}', np.load(po, allow_pickle=False), np.load(pt, allow_pickle=False), 'per-value-digit-te')
        except Exception as e: _log('SKIP megayak', n, type(e).__name__, str(e)[:120])
po, pt = _find('lm_oof.npy', 'target-first-lm'), _find('lm_test.npy', 'target-first-lm')   # v23: megayak's target-first language model
_log('bank source target-first-lm ->', po if po else 'NOT MOUNTED', pt if pt else 'NOT MOUNTED')
if po and pt:
    try: _pool_add('megayak target-first LM', np.load(po, allow_pickle=False), np.load(pt, allow_pickle=False), 'target-first-lm')
    except Exception as e: _log('SKIP megayak target-first LM', type(e).__name__, str(e)[:120])
else: _log('megayak target-first LM: not mounted, skipped')
src = pd.Series([k.split()[0] for k in pool]).value_counts()
_log(len(pool), 'public members loaded:', src.to_dict())
_ckpt('01_external_banks_loaded')

def _std(v):
    r = rankdata(v); s = float(r.std())
    return ((r - r.mean()) / (s if s > 0 else 1.0)).astype(np.float32)
pauc = {}
for k, v in pool.items():
    try: pauc[k] = float(roc_auc_score(y, v[0]))
    except Exception as e: _log('SKIP AUC', k, type(e).__name__, str(e)[:120])
own_best = max((roc_auc_score(y, oof[k]) for k in names), default=0.0)
top = max(max(pauc.values(), default=0.0), own_best)
S = np.stack([_std(oof[k]) for k in names]); kept = []
for k in sorted(pauc, key=pauc.get, reverse=True):
    if pauc[k] < top - 0.008: continue
    s = _std(pool[k][0])
    if (S @ s / len(s)).max() < 0.99995: kept.append(k); S = np.vstack([S, s])
del S
_log(f'best single member {top:.5f}; public members within 0.008 and not near-copies: {len(kept)}')
_ckpt('02_kept_members')

lr_ = lambda v: logit(np.clip(rankdata(v) / len(v), 1e-6, 1 - 1e-6))
ZW = np.column_stack([Z] + [lr_(pool[k][0]) for k in kept]); ZWT = np.column_stack([ZT] + [lr_(pool[k][1]) for k in kept])
wnames = names + kept
_log('wide matrices', ZW.shape, ZWT.shape)
if ZW.shape[0] != len(X) or ZWT.shape[0] != len(XT):
    _log('FATAL: wide matrices misaligned; checkpoint 00 is on disk')
    raise RuntimeError(f'wide matrix rows {ZW.shape[0]}/{ZWT.shape[0]} != {len(X)}/{len(XT)}')
def _fold(A, yy, tr_i, va_i): return va_i, LogisticRegression(C=0.01, max_iter=3000, tol=1e-6).fit(A[tr_i], yy[tr_i]).decision_function(A[va_i])
def _fold_results(folds, tag):
    # Heavy GPU work has already happened: fork/loky can fail or double memory here.
    # Default to in-process folds; opt in with WIDE_POOL_JOBS=N if RAM is plentiful.
    approx_gb = float(np.asarray(ZW).nbytes) / 1e9
    try:
        import psutil
        avail_gb = float(psutil.virtual_memory().available) / 1e9
    except Exception:
        avail_gb = float('inf')
    by_mem = max(1, int(avail_gb / max(approx_gb, 0.1)) - 1) if np.isfinite(avail_gb) else 2
    n_jobs = int(os.environ.get('WIDE_POOL_JOBS', '0')) or min(2, THREADS, by_mem)
    if GPU or approx_gb > 2.0: n_jobs = 1
    _log(tag, f'n_jobs={n_jobs}', f'ZW={approx_gb:.2f}GB', f'avail={avail_gb:.1f}GB', f'GPU={GPU}')
    if n_jobs > 1:
        try:
            out = Parallel(n_jobs=n_jobs, prefer='threads', verbose=0)(
                delayed(_fold)(ZW, y.values, tr_i, va_i) for tr_i, va_i in folds)
            return list(out)
        except Exception as e:
            _log(tag, 'Parallel failed -> sequential fallback', type(e).__name__, str(e)[:160])
    out, prior = [], float(np.mean(y.values))
    for j, (tr_i, va_i) in enumerate(folds):
        try: out.append(_fold(ZW, y.values, tr_i, va_i))
        except Exception as e:
            _log(tag, f'fold {j} failed', type(e).__name__, str(e)[:120], '-> neutral prior')
            out.append((va_i, np.full(len(va_i), prior)))
    return out
wide_runs = []
wide_hold = {}                                            # v25: kept for the two-stack mix below
for seed in (7, 8, 9):                                    # the same three outer splits as the 60-member stacker above
    folds = list(StratifiedKFold(5, shuffle=True, random_state=seed).split(ZW, y))
    s = np.zeros(len(ZW), dtype=np.float64)
    for va_i, dv in _fold_results(folds, f'wide seed {seed}'): s[va_i] = np.asarray(dv, dtype=np.float64)
    wide_runs.append(float(roc_auc_score(y, s))); wide_hold[seed] = s.copy()
    _log('wide seed', seed, f'AUC {wide_runs[-1]:.5f}')
    _ckpt(f'03_wide_seed_{seed}')
_log(f'held-out AUC over splits 7, 8, 9: {len(names)}-member stacker {np.mean(st_runs):.5f} {np.round(st_runs, 5)} | wide stacker ({len(wnames)} members, C 0.01) {np.mean(wide_runs):.5f} {np.round(wide_runs, 5)}')
wstk = LogisticRegression(C=0.01, max_iter=3000, tol=1e-6).fit(ZW, y)
wcoef = pd.Series(wstk.coef_[0], index=wnames)
_log('largest wide-stacker weights:'); print(wcoef.reindex(wcoef.abs().sort_values(ascending=False).index).head(15).round(3).to_string())
def _group(k):
    if k == 'megayak target-first LM': return 'megayak LM (new in v23)'
    if k in pool: return k.split()[0] + ' (new in v22)'
    return 'public banks of v16-v21' if k.startswith('bank ') else 'trained in this notebook'
g = wcoef.abs().groupby([_group(k) for k in wnames]).sum().sort_values()
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 4))
a1.barh(g.index, g.values, color=['#c98b2b' if 'new' in i else '#4a6fa5' for i in g.index]); a1.set_title('sum of |stacker weight| by source, wide stack')
a2.bar([f'{len(names)} members\nC 0.05', f'{len(wnames)} members\nC 0.01'], [np.mean(st_runs), np.mean(wide_runs)], color=['#4a6fa5', '#c98b2b'],
       yerr=[np.std(st_runs), np.std(wide_runs)], capsize=6)
a2.set_ylim(min(np.mean(st_runs), np.mean(wide_runs)) - 0.0003, max(np.mean(st_runs), np.mean(wide_runs)) + 0.0002); a2.set_title('stacker held-out AUC, splits 7 / 8 / 9')
plt.tight_layout(); plt.show()
if np.mean(wide_runs) > np.mean(st_runs):
    blend = wstk.decision_function(ZWT)
    sub = pd.DataFrame({'id': test['id'], TARGET: rankdata(blend) / len(blend)})
    _atomic(OUT / 'submission.csv', lambda f: f.write(sub.to_csv(index=False).encode('utf-8')))
    _log('submitting the wide stacker', sub.shape)
else:
    _log('the wide stacker did not beat the 60-member stacker here; submission.csv is the one written above')
_ckpt('04_final')


### v23: a second language model in the wide pool (megayak's target-first GPT-2)

**What it is.** [megayak](https://www.kaggle.com/code/megayak/s6e11-per-value-digit-te-shallow-tree-stack) trained their own GPT-2 small on one laptop GPU
and published its out-of-fold and test predictions: [S6E11 target-first LM predictions](https://www.kaggle.com/datasets/megayak/s6e11-target-first-lm) (CC0), on our five folds (`StratifiedKFold(5, shuffle=True, random_state=42)`).
v23 adds it to the wide pool as one more member. Nothing else changes.

**How it works.** Each row is written as text, **label first**: `is_fraud` comes before the columns, then one learned token per column name followed by the GPT-2 tokens of its value, in a fresh random column order each time.
The model learns to write fraud rows and legitimate rows. To score a row, it reads the row twice, once after "fraud" and once after "legit", and compares the two likelihoods (Bayes' rule turns them into log-odds).
The idea is Chris Deotte's (S6E10 1st place); masayakawamata's public S6E11 member uses the same recipe with different settings, and megayak's row format is more compact.

**Why it should help.** A language model reads the digits of every number in the order the generator wrote them; trees and GLMs see encodings of values and digit keys.
Measured here (`scratch/banks/wide_1108.py` in github.com/alexchilton/kaggley):
- alone it scores OOF **0.95043**, below our best members (0.95159);
- its rank correlation with masayakawamata's GPT-2 member is **0.9802** (OOF), and with every member already in our stack at most **0.9824**. Our tree members correlate with each other at 0.99+.
A member that is weaker alone but ranks rows differently is what a stack can use (lesson 1 below; v22's GPT-2 member took the largest weight).

**Who found it.** [goodpjw2008 Part 12](https://www.kaggle.com/code/goodpjw2008/s6e11-part-12-wide-stack-2nd-lm-lb-0-95298) added it to their 200-member wide stack:
nested CV 0.952504 -> 0.952565, public LB 0.95285 -> **0.95298**, 4 of 5 folds up. They found it because
[areebbb](https://www.kaggle.com/code/areebbb/s6e11-wide-stack-shallow-tree-fusion-0-95291) had reached 0.95291 by mixing megayak's *submission file* into goodpjw2008's files at 20%, with the weight picked on the public board.
megayak's file is far from the others because of this language model. Mixing a file in can only be judged on the public board; out-of-fold predictions can go into the stack and be measured on all 700k rows.
Credit: megayak for training and publishing the model, goodpjw2008 for the out-of-fold test, areebbb for the blend that pointed at it, Chris Deotte and masayakawamata for the recipe. Please upvote them.

**What to read in the output.** The wide-stacker line of the cell above now includes this member. v22 printed **0.95251** for the same three splits (7 / 8 / 9) without it,
and every other source in the pool was checked byte-identical to v22's copy on 2026-11-08. The difference between the two lines is what this member adds; the weights line shows its stacker weight.


### v25: the 0.95300 public recipe, measured on our own two stacks

**The question.** On 2026-11-08 three public notebooks reached public LB **0.95300** with the same two files:
[malyshevdanil's Fraud Signal Lab](https://www.kaggle.com/code/malyshevdanil/fraud-signal-lab-gpt-2-and-glm-fusion) (v5, the recipe),
[areebbb's Wide Stack + Shallow-Tree Fusion](https://www.kaggle.com/code/areebbb/s6e11-wide-stack-shallow-tree-fusion-0-95300) and
[lucifer19's Golden Signal V22](https://www.kaggle.com/code/lucifer19/the-golden-signal-consensus-blend-v22).
Our best is 0.95260. What is in that recipe, and does it hold on rows a model did not fit?

**What it is.** 65% of [goodpjw2008 Part 12](https://www.kaggle.com/code/goodpjw2008/s6e11-part-12-wide-stack-2nd-lm-lb-0-95298)'s wide stack (201 public and own members, one logistic stacker)
and 35% of goodpjw2008's **Part 9**, their earlier, narrower stack of their own members, with the test side averaged over every refit.
Both files are turned into ranks, then logits (`logit((rank - 0.5) / n)`), and mixed. Nothing is trained.
The files are frozen in [s6e11-fraud-series-artifacts](https://www.kaggle.com/datasets/goodpjw2008/s6e11-fraud-series-artifacts); the cell below checks their SHA-256 against the hashes the 0.95300 notebooks check.

**What goodpjw2008 measured themselves** (Part 11, section 11.2, the same mix with the older wide stack):

| weight of the wide stack | nested CV | public LB |
|---|---|---|
| 1 | 0.952504 | 0.95267 |
| 0.8 | 0.952480 | 0.95282 |
| 0.7 | 0.952452 | 0.95285 |
| 0.5 | 0.952368 | 0.95283 |

The cross-validated AUC **falls** as Part 9 comes in, and the public score rises. Part 9 has no published out-of-fold predictions, so the 0.95300 mix cannot be scored on the training rows.

**How we test it.** Our notebook has the same two kinds of stack: the 60-member stacker (our own members and the v16-v21 banks, the role of Part 9) and the wide stacker (those 60 plus 107 public members, the role of Part 12).
The wide stack already contains every member of the narrow one, so a mix can only help if the narrow stacker, with fewer columns and less regularisation, ranks some rows better.
Both stackers are held out on the same three outer splits (7, 8, 9), so their mix is scored on rows neither fitted. The cell prints the AUC for wide-stack weights 1, 0.9, 0.8, 0.65, 0.5 and 0 (the narrow stack alone).

**The test side.** The cell also prints how far apart the test files are (1 - rank correlation, in 1e-4): our wide and narrow stacks, Part 9, Part 12, and both 65/35 mixes.
goodpjw2008 measured their wide stack 95.0e-4 from Part 9 and called it "an unusually large distance between strong files"; the same number for our two stacks says whether our narrow stack can play Part 9's role at all.

**What is submitted.** `submission.csv` stays the wide stacker, unless the 65/35 mix of our two stacks beats it on all three splits (the weight is fixed at the public recipe's 0.65, not picked on these splits).
Two more files are written for the public board, where the recipe's evidence lives: `submission_mix_ours.csv` (65% our wide + 35% our narrow) and `submission_mix_part9.csv` (65% our wide + 35% goodpjw2008's Part 9, i.e. the recipe with our wide stack in Part 12's place).

**Credit.** goodpjw2008 for both stacks, the frozen files and the honest CV-vs-LB table; malyshevdanil for the 65/35 rank-logit recipe; areebbb and lucifer19 for the public-board probes around it. Please upvote them.


In [ ]:
# v25: the two-stack mix (malyshevdanil's 65/35 rank-logit recipe), scored on held-out rows of our own two stackers
import hashlib
def _lr(v): return logit(np.clip((rankdata(v) - 0.5) / len(v), 1e-7, 1 - 1e-7))
_sp = (7, 8, 9)
mix = {w: [roc_auc_score(y, w * _lr(wide_hold[s]) + (1 - w) * _lr(st_hold[s])) for s in _sp] for w in (1.0, 0.9, 0.8, 0.65, 0.5)}
mix[0.0] = [roc_auc_score(y, st_hold[s]) for s in _sp]
mt = pd.DataFrame(mix, index=[f'split {s}' for s in _sp]).T.sort_index(ascending=False); mt['mean'] = mt.mean(1)
mt['vs wide alone (x 1e-4)'] = (mt['mean'] - mt.loc[1.0, 'mean']) * 1e4
mt.index.name = 'weight of the wide stack'; display(mt.round(5))
print('rank correlation of the two held-out stacks per split:', [round(np.corrcoef(rankdata(wide_hold[s]), rankdata(st_hold[s]))[0, 1], 5) for s in _sp])
t_w, t_n = wstk.decision_function(ZWT), stk.decision_function(ZT)
T = {'our wide': t_w, 'our narrow (60)': t_n}
_h = {'part9_test.npy': 'e899205391b8755e82a2ae6b06fb1e026ef29132e4bcdbfad96535d7f5608abd', 'wide_lm_stack_test.npy': 'b37471e6c34a50715284e807d68e15e5f3ad033e72400904f27876b24041f8cc'}
for f_, lab in [('part9_test.npy', 'goodpjw2008 Part 9'), ('wide_lm_stack_test.npy', 'goodpjw2008 Part 12 wide')]:
    p_ = _find(f_, 'fraud-series-artifacts')
    if p_ is None: print(f_, 'not mounted, skipped'); continue
    try:
        hx = hashlib.sha256(open(p_, 'rb').read()).hexdigest()
        print(f_, 'SHA-256 matches the 0.95300 notebooks' if hx == _h[f_] else f'SHA-256 CHANGED: {hx}')
        v_ = np.load(p_).ravel().astype(float)
        if len(v_) == len(XT) and np.isfinite(v_).all() and np.unique(v_).size > 1:
            T[lab] = v_
        else:
            print(f_, 'bad length or values, skipped')
    except Exception as e:
        print(f_, 'unreadable, skipped:', type(e).__name__, str(e)[:80])
T['mix 0.65 our wide + 0.35 our narrow'] = 0.65 * _lr(t_w) + 0.35 * _lr(t_n)
if 'goodpjw2008 Part 9' in T:
    T['mix 0.65 our wide + 0.35 Part 9'] = 0.65 * _lr(t_w) + 0.35 * _lr(T['goodpjw2008 Part 9'])
if 'goodpjw2008 Part 9' in T and 'goodpjw2008 Part 12 wide' in T:
    T['mix 0.65 Part 12 + 0.35 Part 9 (LB 0.95300)'] = 0.65 * _lr(T['goodpjw2008 Part 12 wide']) + 0.35 * _lr(T['goodpjw2008 Part 9'])
TR = np.corrcoef([rankdata(v) for v in T.values()])
print('test files: distance 1 - rank correlation, x 1e-4'); display(pd.DataFrame((1 - TR) * 1e4, index=list(T), columns=list(T)).round(1))
fig, a1 = plt.subplots(figsize=(8, 4))
for j, s in enumerate(_sp): a1.plot(mt.index, mt[f'split {s}'], 'o-', alpha=0.5, label=f'split {s}')
a1.plot(mt.index, mt['mean'], 'k-', lw=2, label='mean'); a1.axvline(0.65, color='#c98b2b', ls='--', lw=1, label='public recipe 0.65')
a1.set_xlabel('weight of the wide stack (rest: the 60-member stack)'); a1.set_ylabel('held-out AUC'); a1.set_ylim(mt['mean'].max() - 0.0004, mt['mean'].max() + 0.0001)
a1.set_title('the two-stack mix on held-out rows'); a1.legend(fontsize=8); plt.tight_layout(); plt.show()
for n_, v_ in [('submission_mix_ours.csv', T['mix 0.65 our wide + 0.35 our narrow']), ('submission_mix_part9.csv', T.get('mix 0.65 our wide + 0.35 Part 9'))]:
    if v_ is not None: pd.DataFrame({'id': test['id'], TARGET: rankdata(v_) / len(v_)}).to_csv(OUT / n_, index=False); print('wrote', n_)
if np.mean(wide_runs) > np.mean(st_runs) and all(a > b for a, b in zip(mix[0.65], mix[1.0])):
    v_ = T['mix 0.65 our wide + 0.35 our narrow']
    pd.DataFrame({'id': test['id'], TARGET: rankdata(v_) / len(v_)}).to_csv(OUT / 'submission.csv', index=False)
    print('the 65/35 mix beats the wide stacker on all three splits: submission.csv is the mix')
else:
    print('the 65/35 mix does not beat the wide stacker on all three splits: submission.csv stays as written above')

### What the board said about v25 (submitted 2026-11-09)

| file | what | public LB |
|---|---|---|
| `submission.csv` | our wide stacker (v23's ranking, unchanged) | 0.95260 (v23) |
| `submission_mix_part9.csv` | 0.65 our wide + 0.35 goodpjw2008 Part 9 | 0.95259 |
| `submission_mix_ours.csv` | 0.65 our wide + 0.35 our 60-member stack | 0.95244 |

Held-out AUC (splits 7 / 8 / 9): wide 0.95257 / 0.95259 / 0.95255, the 65/35 mix of our two stacks 0.95255 / 0.95259 / 0.95255.
Both say the same: mixing a narrow stack into our wide one does not help. Part 9 on top of our wide stack does nothing on the board (-0.00001), and our own narrow stack costs 0.00016.
So the 0.95300 is not "a wide stack plus a narrow stack" in general. On the test rows our wide stack is already 25e-4 from Part 9, while goodpjw2008's Part 12 is 83e-4 from both.
The next question is which half of the 0.95300 mix carries the gain. The section below tests it.

### v26: the 0.95330 file, and how large a leaderboard step is

**The question.** On 2026-11-09 the public best moved from 0.95300 to **0.95330**: [Moran Z's Grandmaster Ladder](https://www.kaggle.com/code/moranzavdi/s6e11-grandmaster-ladder-lb-0-95330) published a frozen file, `submission_095330.csv` ("Strike 9"),
and [areebbb's 0.95330 Ladder](https://www.kaggle.com/code/areebbb/s6e11-the-0-95330-ladder-what-actually-moved-lb) copies it with CV checks around it.
The file has no out-of-fold predictions. Moran Z's ladder lists ten public-board steps from the Part 12 backbone (0.95307) to 0.95330; four of them are "LB-gradient ascent" or "continue the high-slope axis",
i.e. steps chosen by their public score. From 0.95319 upward each step is +0.00001 to +0.00003. How large is that next to the noise of the public board, and how far is the file from ours?

**What the cell measures.**
1. **Distance on the test rows** (1 - rank correlation, in 1e-4) between our wide stack, goodpjw2008's Part 9 and Part 12, the 0.95300 mix, the 0.95330 file, and one new mix.
2. **The new mix**, `submission_p12_ours.csv`: 0.65 goodpjw2008 Part 12 + 0.35 **our** wide stack, i.e. the 0.95300 recipe with our wide stack in Part 9's place.
   v25 put our stack in Part 12's place and scored 0.95259. If this one scores near 0.95300, the gain of the recipe lives in Part 12's test predictions, and the 35% partner can be any good Part 9-like file.
3. **The noise of a public-board step.** The public board scores about 20% of the test rows. We draw 200 random subsets of that size from our held-out rows (split 7)
   and, for pairs of files at a known distance, measure the spread of their AUC *difference*. The same for 80%-sized subsets, the size of the private board.
   If the spread of a difference is larger than a ladder step, a step picked on the public board is mostly a draw of that noise, and it need not carry to the private board.

**Credit.** Moran Z for the 0.95330 file and the ladder; areebbb for the copy, the CV checks and the note that late steps are the size of the noise; goodpjw2008 for Part 9 and Part 12. Please upvote them.


In [ ]:
# v26: the 0.95330 file (Moran Z's Strike 9) on the test side, our wide stack in Part 9's place, and the noise of a public-board step
p_ = _find('submission_095330.csv', 'strike9')
if p_ is None: print('submission_095330.csv not mounted, skipped')
else:
    try:
        hx = hashlib.sha256(open(p_, 'rb').read()).hexdigest()
        print('submission_095330.csv', 'SHA-256 matches Moran Z' if hx == 'f4a7a4f8636c99fabd82945c7d81956c5559c609dcb55cde6d4e03f80b0ea105' else f'SHA-256 CHANGED: {hx}')
        s9_ = pd.read_csv(p_)
        v9 = s9_.set_index('id')[TARGET].reindex(test['id'].to_numpy()).to_numpy(dtype=float) if 'id' in s9_.columns else s9_[TARGET].to_numpy(dtype=float)
        if len(v9) == len(XT) and np.isfinite(v9).all() and np.unique(v9).size > 1:
            T['Moran Z Strike 9 (LB 0.95330)'] = v9
        else:
            print('submission_095330.csv bad length or values, skipped')
    except Exception as e:
        print('submission_095330.csv unreadable, skipped:', type(e).__name__, str(e)[:80])
if 'goodpjw2008 Part 12 wide' in T:
    T['mix 0.65 Part 12 + 0.35 our wide'] = 0.65 * _lr(T['goodpjw2008 Part 12 wide']) + 0.35 * _lr(t_w)
    v_ = T['mix 0.65 Part 12 + 0.35 our wide']
    pd.DataFrame({'id': test['id'], TARGET: rankdata(v_) / len(v_)}).to_csv(OUT / 'submission_p12_ours.csv', index=False); print('wrote submission_p12_ours.csv')
k26 = [k for k in ['our wide', 'goodpjw2008 Part 9', 'goodpjw2008 Part 12 wide', 'mix 0.65 Part 12 + 0.35 Part 9 (LB 0.95300)', 'mix 0.65 Part 12 + 0.35 our wide', 'Moran Z Strike 9 (LB 0.95330)'] if k in T]
if len(k26) >= 2:
    D26 = pd.DataFrame((1 - np.corrcoef([rankdata(T[k]) for k in k26])) * 1e4, index=k26, columns=[str(i) for i in range(len(k26))])
    print('test files: distance 1 - rank correlation, x 1e-4 (columns in row order)'); print(D26.round(1).to_string())
else:
    D26 = pd.DataFrame([[0.0]], index=k26, columns=[str(i) for i in range(len(k26))])
# the noise of a step: AUC differences of two held-out files on random public-sized (20%) and private-sized (80%) subsets of split 7's rows
yv, rng26 = y.to_numpy(), np.random.default_rng(26)
a_ = wide_hold[7]
pairs26 = {'wide vs 0.9 wide + 0.1 narrow': 0.9 * _lr(a_) + 0.1 * _lr(st_hold[7]), 'wide vs 0.65 wide + 0.35 narrow': 0.65 * _lr(a_) + 0.35 * _lr(st_hold[7]), 'wide vs narrow (60)': st_hold[7]}
rows26, draws26 = [], {}
for lab, b_ in pairs26.items():
    full = (roc_auc_score(yv, a_) - roc_auc_score(yv, b_)) * 1e4
    r_ = [lab, (1 - np.corrcoef(rankdata(a_), rankdata(b_))[0, 1]) * 1e4, full]
    for frac in (0.2, 0.8):
        n_ = int(frac * len(XT)); d_ = []
        for _ in range(200):
            i_ = rng26.choice(len(yv), n_, replace=False); d_.append((roc_auc_score(yv[i_], a_[i_]) - roc_auc_score(yv[i_], b_[i_])) * 1e4)
        draws26[(lab, frac)] = np.array(d_); r_ += [np.std(d_), np.mean(np.sign(d_) != np.sign(full))]
    rows26.append(r_)
N26 = pd.DataFrame(rows26, columns=['pair', 'distance x 1e-4', 'AUC difference, all rows x 1e-4', 'spread on 20% (public) x 1e-4', 'share of 20% draws with the sign flipped',
                                    'spread on 80% (private) x 1e-4', 'share of 80% draws with the sign flipped']).set_index('pair')
print(f'noise of an AUC difference, 200 random subsets of split 7 held-out rows ({int(0.2 * len(XT))} and {int(0.8 * len(XT))} rows)'); print(N26.round(3).to_string())
_ss = [np.std([roc_auc_score(yv[i_], a_[i_]) for i_ in [rng26.choice(len(yv), int(0.2 * len(XT)), replace=False) for _ in range(100)]]) * 1e4]
print(f'spread of one file\'s AUC on public-sized subsets: {_ss[0]:.1f}e-4')
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.2), gridspec_kw={'width_ratios': [1, 1.2]})
im = a1.imshow(D26.values, cmap='Blues'); a1.set_xticks(range(len(k26))); a1.set_yticks(range(len(k26)))
_short = [k.replace('goodpjw2008 ', '').replace(' (LB 0.95300)', '\n(0.95300)').replace(' (LB 0.95330)', '\n(0.95330)').replace('mix 0.65 ', '').replace(' + 0.35 ', ' / ') for k in k26]
a1.set_xticklabels(_short, rotation=60, ha='right', fontsize=7); a1.set_yticklabels(_short, fontsize=7)
for i in range(len(k26)):
    for j in range(len(k26)): a1.text(j, i, f'{D26.values[i, j]:.0f}', ha='center', va='center', fontsize=7, color='k' if D26.values[i, j] < 60 else 'w')
a1.set_title('test files: 1 - rank correlation (x 1e-4)', fontsize=10)
_lab = 'wide vs 0.65 wide + 0.35 narrow'
for frac, col in [(0.2, '#c98b2b'), (0.8, '#4a6fa5')]: a2.hist(draws26[(_lab, frac)], bins=30, alpha=0.6, color=col, label=f'{int(frac * 100)}% of the test size')
a2.axvline(N26.loc[_lab, 'AUC difference, all rows x 1e-4'], color='k', lw=1.5, label='all held-out rows')
for s_ in (0.1, 0.3): a2.axvspan(-s_, s_, color='#999', alpha=0.15)
a2.set_xlabel('AUC(wide) - AUC(65/35 mix), x 1e-4  (grey: a ladder step of 0.00001 / 0.00003)'); a2.set_title(f'the same two files, scored on random subsets: {_lab}', fontsize=10); a2.legend(fontsize=8)
plt.tight_layout(); plt.show()

### What the board said about v26 (submitted 2026-11-10)

| file | what | public LB |
|---|---|---|
| `submission.csv` | our wide stacker (v23's ranking, unchanged) | 0.95260 (v23) |
| `submission_p12_ours.csv` | 0.65 goodpjw2008 Part 12 + 0.35 our wide stack | **0.95305** |
| (public) | 0.65 Part 12 + 0.35 Part 9, the 0.95300 recipe | 0.95300 |
| `submission_mix_part9.csv` (v25) | 0.65 our wide + 0.35 Part 9 | 0.95259 |

With our stack in Part 9's place the recipe scores 0.95305; with our stack in Part 12's place it scored 0.95259. So the recipe's gain comes with goodpjw2008's Part 12 test file, and the 35% partner can be Part 9 or our stack.
Our wide stack and Part 12 have the same held-out AUC (0.95257 both), so the gap is on the test side. HYPOTHESIS (unverified): Part 12's test columns were made differently (full-data refits against fold averages).
0.95305 against 0.95300 is +0.00005 between files 3.1e-4 apart; the noise table of v26 gives a spread of 0.29e-4 to 0.87e-4 for such pairs on the public rows. Treat the two as equal.

**The noise of a public-board difference, as v26 measured it** (200 random subsets of held-out rows; values x 1e-4):

| pair | distance | AUC difference, all rows | spread on public-sized subsets | spread on private-sized subsets |
|---|---|---|---|---|
| wide vs 0.9 wide + 0.1 narrow | 0.33 | -0.09 | 0.29 | 0.11 |
| wide vs 0.65 wide + 0.35 narrow | 4.0 | 0.01 | 0.87 | 0.39 |
| wide vs narrow (60 members) | 32.9 | 2.40 | 2.80 | 1.08 |

The noise of a *difference* grows with the distance between the two files. A ladder step of 0.00001 between files 0.3e-4 apart is a third of one spread.

### v27: the public ladder, replayed on held-out rows

**The question.** On 2026-11-10 and 11 the public best moved from 0.95330 to **0.95347** in six steps:
[amanatar's Podium Apex](https://www.kaggle.com/code/amanatar/s6e11-podium-apex-public-lb-0-95340) ("Strike 10", 0.95340, a frozen file),
[goodpjw2008's LM Ladder Part 1](https://www.kaggle.com/code/goodpjw2008/s6e11-lm-ladder-part-1-own-lm-step-lb-0-95346) (0.95346) and
[areebbb's Three Ladders, Two Steps](https://www.kaggle.com/code/areebbb/s6e11-three-ladders-two-steps-lb-0-95347) (0.95347).
All three say plainly that each step was chosen by its public score and has no out-of-fold check. areebbb's final recipe, in logit ranks, with `F` = Strike 9 and `A` = Strike 10:

```
B     = A + 0.6 (A - F) + 0.12 (LM - F) + 0.06 (L7 - F)        LM: megayak's language model, L7: goodpjw2008's own seed-7 language model
score = B - 0.05 (MLP - F) - 0.025 (GB - F)                    MLP, GB: two members of goodpjw2008's OOF bank
```

A step is `file + w (member - base)`: move the ranking a little towards one member (or away from it, with a negative `w`), try it on the board, keep it if the score went up.
The files `F` and `A` have no out-of-fold predictions, so nobody can score these steps on labelled rows. But the *method* can be replayed where the labels are known.

**How the replay works.**
1. The base is our wide stacker's held-out score on all 700k training rows (split 7): each row is scored by a stacker that did not see it. This plays the role of `F`.
2. The step directions are the four members the public ladders use, as out-of-fold predictions: megayak's language model, masayakawamata's GPT-2, and goodpjw2008's MLP bag (`gbn_bag5_w512`) and GB bag (`gb_bag5`).
3. One draw: pick 299,984 rows at random (the size of the test set); the first 20% are the "public board", the other 80% the "private board".
4. Climb: try every direction at `w` = ±0.03, ±0.06, ±0.12 (24 candidate files), keep the one with the best *public* AUC if it beats the current file, and repeat up to six times, as a ladder does.
5. Score the final file on the private rows. 100 draws.

If the steps find something real that the stacker missed, the private rows gain too. If they fit the public rows' noise, the public gain is there by selection and the private rows gain nothing or lose.

**Measured before this version ran** (same code on the published predictions, `scratch/banks/ladder_1111/sim.py` in github.com/alexchilton/kaggley; the cell below prints its own numbers):

| | AUC change x 1e-4 |
|---|---|
| public rows, mean over 100 draws (the steps were chosen here) | **+2.2** |
| private rows, mean over 100 draws | **-1.6** |
| share of draws where the private rows gained | 8% |
| correlation of public gain and private gain over the draws | -0.66 |

The climb gains +0.00022 on its public rows and loses 0.00016 on the private rows; the larger the public gain of a draw, the larger its private loss.
For scale: the public ladder climbed +0.00017 (0.95330 to 0.95347). A gain of that size appears from selection alone.

**One step without any selection** (all 700k held-out rows; AUC change x 1e-4):

| step | megayak LM | masayakawamata GPT-2 | MLP bag | GB bag |
|---|---|---|---|---|
| w = -0.12 | -0.03 | +0.16 | -0.63 | -0.55 |
| w = -0.06 | +0.05 | +0.15 | -0.26 | -0.23 |
| w = +0.06 | -0.19 | -0.30 | +0.14 | +0.13 |
| w = +0.12 | -0.51 | -0.75 | +0.15 | +0.17 |

On held-out rows, a step **towards** megayak's LM loses, and a step **away** from the MLP and GB bags loses. The public ladders took exactly these steps (+0.12 LM, -0.05 MLP, -0.025 GB) and gained on the board.
areebbb measured the same disagreement on goodpjw2008's nested OOF (full tree move -7.4e-5, +5% LM -1.5e-5).

**What this does and does not show.** It shows that the ladder *method*, started from a fitted stack, produces public gains of the size seen on the board with a private loss behind them.
It does not prove the real files will fall: the replay's base is a stacker that is already at its held-out optimum, so every direction is downhill, and the test-side predictions of a member are not the same thing as its out-of-fold predictions
(megayak's test file averages five fold models; an out-of-fold row is scored by one). HYPOTHESIS (unverified): part of the board's taste for the LM is that its test file is stronger than its OOF column says.
The private board will say. For your two final picks, the advice in areebbb's and goodpjw2008's notebooks holds: one best-public file and one file with an out-of-fold score.

**The test side, and two files for the board.** The cell rebuilds the 0.95347 file from the six public inputs (each checked by SHA-256) and prints the distance table with Strike 10, the 0.95347 file and our files.
It also takes the ladder's most-used step from our own files and writes them, because for these we have the held-out number above to put next to the board's:
- `submission_wide_lm12.csv`: our wide stack + 0.12 (megayak LM - our wide stack). Held-out: **-0.51e-4**. If the board gains, the board and the held-out rows disagree about the same step on the same stack.
- `submission_p12_ours_lm12.csv`: the same step from `submission_p12_ours.csv` (0.95305).
It also splits each ladder file's move from Strike 9 into its member directions by least squares, as areebbb did for Strike 10.

**Credit.** amanatar (Strike 10), goodpjw2008 (the LM ladder, the seed-7 language model, the OOF bank), areebbb (the 0.95347 recipe, every probe published including the losing ones, and the CV checks), Moran Z (Strike 9), megayak (the language model), samanyu1808 (the LM-step probes). Please upvote them.


In [ ]:
# v27: the public ladder replayed on held-out rows (public-sized subset picks the steps, private-sized subset scores them), and the ladder files on the test side
def _fauc(yy, s):                                         # AUC from ranks: the same number as roc_auc_score, faster
    r = rankdata(s); n1 = int(yy.sum()); return (r[yy == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * (len(yy) - n1))
yv27 = y.to_numpy().astype(int); F27 = _lr(wide_hold[7])
_d27 = {'megayak LM': 'megayak target-first LM', 'masayakawamata GPT-2': 'masayakawamata GPT-2 target-first', 'goodpjw2008 MLP bag': 'goodpjw2008 gbn_bag5_w512', 'goodpjw2008 GB bag': 'goodpjw2008 gb_bag5'}
DIR27 = {k: _lr(pool[v][0]) for k, v in _d27.items() if v in pool}
print('step directions found in the pool:', list(DIR27))
a27 = _fauc(yv27, F27); WS27 = (-0.12, -0.06, -0.03, 0.03, 0.06, 0.12)
if DIR27:
    one27 = pd.DataFrame({k: [(_fauc(yv27, F27 + w * (d_ - F27)) - a27) * 1e4 for w in WS27] for k, d_ in DIR27.items()}, index=WS27); one27.index.name = 'w'
    print(f'held-out AUC of the base (wide stacker, split 7): {a27:.5f}'); print('one step on all held-out rows, AUC change x 1e-4 (no selection):'); print(one27.round(3).to_string())
    rng27, n_t, n_pub, rows27 = np.random.default_rng(27), len(XT), int(0.2 * len(XT)), []
    for it in range(100):
        i_ = rng27.choice(len(yv27), n_t, replace=False); pu, pr = i_[:n_pub], i_[n_pub:]
        cur = np.zeros(len(yv27)); p0 = _fauc(yv27[pu], F27[pu]); best = p0; path = []
        for rnd in range(6):
            a_, k_, w_ = max((_fauc(yv27[pu], (F27 + cur + w * (d_ - F27))[pu]), k, w) for k, d_ in DIR27.items() for w in WS27)
            if a_ <= best: break
            best = a_; cur = cur + w_ * (DIR27[k_] - F27); path.append((k_, w_))
        rows27.append([len(path), (best - p0) * 1e4, (_fauc(yv27[pr], (F27 + cur)[pr]) - _fauc(yv27[pr], F27[pr])) * 1e4, path[0][0] if path else '-'])
    S27 = pd.DataFrame(rows27, columns=['steps', 'public gain x 1e-4', 'private gain x 1e-4', 'first step'])
    print(f'ladder replay, 100 draws of {n_t} held-out rows ({n_pub} public, {n_t - n_pub} private):'); print(S27.describe().round(3).to_string())
    print(f"mean public gain {S27['public gain x 1e-4'].mean():+.2f}e-4 | mean private gain {S27['private gain x 1e-4'].mean():+.2f}e-4 | share of draws with a private gain {(S27['private gain x 1e-4'] > 0).mean():.2f} | "
          f"correlation {np.corrcoef(S27['public gain x 1e-4'], S27['private gain x 1e-4'])[0, 1]:.3f} | first step: {S27['first step'].value_counts().to_dict()}")
else:
    one27 = pd.DataFrame(columns=list(DIR27)); S27 = pd.DataFrame(columns=['steps', 'public gain x 1e-4', 'private gain x 1e-4', 'first step'])
    print('no ladder step directions in the pool; replay skipped')
# the test side: Strike 10, the 0.95347 recipe rebuilt from its public inputs, and the LM step on our own files
_h27 = {'submission_095340.csv': ('strike10', '12b525b84f29e32d6511e9d6105f6dc5eac79ae41b10140eed3353ea57afc159'), 'lm7_test_f0.npy': ('compact-lm', '78af7498ee398b300556a8afc9d661fdfd2b42d8d632e6af9cd412aa3430a369'),
        'lm_test.npy': ('target-first-lm', '3168779be644a530c6bdc4c141cae168bfc2feb01ac21090c917cc978a7687d3')}
P27 = {}
for f_, (own_, h_) in _h27.items():
    p_ = _find(f_, own_)
    if p_ is None: print(f_, 'not mounted, skipped'); continue
    try:
        hx = hashlib.sha256(open(p_, 'rb').read()).hexdigest(); print(f_, 'SHA-256 matches the 0.95347 notebook' if hx == h_ else f'SHA-256 CHANGED: {hx}'); P27[f_] = p_
    except Exception as e:
        print(f_, 'unreadable, skipped:', type(e).__name__)
LMt = _lr(pool['megayak target-first LM'][1]) if 'megayak target-first LM' in pool else None
if LMt is not None:
    T['our wide + 0.12 (LM - our wide)'] = _lr(t_w) + 0.12 * (LMt - _lr(t_w))
    if 'mix 0.65 Part 12 + 0.35 our wide' in T:
        m_ = _lr(T['mix 0.65 Part 12 + 0.35 our wide']); T['P12/our wide mix + 0.12 (LM - mix)'] = m_ + 0.12 * (LMt - m_)
    for n_, k_ in [('submission_wide_lm12.csv', 'our wide + 0.12 (LM - our wide)'), ('submission_p12_ours_lm12.csv', 'P12/our wide mix + 0.12 (LM - mix)')]:
        if k_ in T: pd.DataFrame({'id': test['id'], TARGET: rankdata(T[k_]) / len(T[k_])}).to_csv(OUT / n_, index=False); print('wrote', n_)
S9_ = 'Moran Z Strike 9 (LB 0.95330)'
if 'submission_095340.csv' in P27 and S9_ in T:
    try:
        s10_ = pd.read_csv(P27['submission_095340.csv']); assert (s10_['id'].to_numpy() == test['id'].to_numpy()).all()
        A27, F9 = _lr(s10_[TARGET].to_numpy()), _lr(T[S9_]); T['amanatar Strike 10 (LB 0.95340)'] = A27
        _need = ['goodpjw2008 gbn_bag5_w512', 'goodpjw2008 gb_bag5']
        if 'lm7_test_f0.npy' in P27 and LMt is not None and all(k in pool for k in _need):
            try:
                a7 = np.load(P27['lm7_test_f0.npy']); L7 = _lr((a7[..., 1] - a7[..., 0]).mean(1)); MLPt, GBt = _lr(pool[_need[0]][1]), _lr(pool[_need[1]][1])
                G27 = A27 + 0.6 * (A27 - F9) + 0.12 * (LMt - F9) + 0.06 * (L7 - F9) - 0.05 * (MLPt - F9) - 0.025 * (GBt - F9); T['areebbb ladder, rebuilt (LB 0.95347)'] = G27
                Xd = np.column_stack([LMt - F9, L7 - F9, MLPt - F9, GBt - F9, _lr(t_w) - F9])
                for n_, v_ in [('Strike 10 - Strike 9', A27 - F9), ('0.95347 file - Strike 9', G27 - F9)]:
                    c_ = np.linalg.lstsq(Xd, v_, rcond=None)[0]
                    print(f'{n_}, least squares on member directions:', dict(zip(['megayak LM', 'LM seed 7', 'MLP bag', 'GB bag', 'our wide'], np.round(c_, 3))), '| share of the move explained', round(1 - ((v_ - Xd @ c_) ** 2).sum() / (v_ ** 2).sum(), 3))
            except Exception as e:
                print('0.95347 rebuild skipped:', type(e).__name__, str(e)[:100])
    except Exception as e:
        print('Strike 10 file skipped:', type(e).__name__, str(e)[:100])
k27 = [k for k in ['our wide', 'our wide + 0.12 (LM - our wide)', 'mix 0.65 Part 12 + 0.35 our wide', 'P12/our wide mix + 0.12 (LM - mix)', S9_, 'amanatar Strike 10 (LB 0.95340)', 'areebbb ladder, rebuilt (LB 0.95347)'] if k in T]
if len(k27) >= 2:
    D27 = pd.DataFrame((1 - np.corrcoef([rankdata(T[k]) for k in k27])) * 1e4, index=k27, columns=[str(i) for i in range(len(k27))])
    print('test files: distance 1 - rank correlation, x 1e-4 (columns in row order)'); print(D27.round(1).to_string())
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.4))
if 'rows27' in dir() and len(rows27):
    a1.scatter(S27['public gain x 1e-4'], S27['private gain x 1e-4'], s=18, color='#4a6fa5', alpha=0.7); a1.axhline(0, color='k', lw=0.8); a1.axvline(0, color='k', lw=0.8)
    a1.axvline(1.7, color='#c98b2b', ls='--', lw=1, label='the public ladder: 0.95330 -> 0.95347')
    a1.set_xlabel('gain on the public-sized rows that picked the steps (x 1e-4)'); a1.set_ylabel('gain on the private-sized rows (x 1e-4)')
    a1.set_title('a ladder climbed on held-out rows: 100 draws', fontsize=10); a1.legend(fontsize=8)
for k in one27.columns: a2.plot(one27.index, one27[k], 'o-', label=k)
a2.axhline(0, color='k', lw=0.8); a2.set_xlabel('step w:  file + w (member - file)'); a2.set_ylabel('held-out AUC change, all rows (x 1e-4)')
a2.set_title('one step, no selection: which way is uphill on held-out rows', fontsize=10); a2.legend(fontsize=8); plt.tight_layout(); plt.show()

### What the board said about v27 (submitted 2026-11-12)

| file | what | public LB | the same file without the step |
|---|---|---|---|
| `submission_p12_ours_lm12.csv` | (0.65 Part 12 + 0.35 our wide stack) + 0.12 (megayak LM - file) | **0.95305** | 0.95305 |
| `submission_wide_lm12.csv` | our wide stack + 0.12 (megayak LM - stack) | 0.95260 | 0.95260 |

The ladder's most-used step changes neither of our files at five decimals. On held-out rows the same step measured -0.51e-4.
The public ladders report +0.00001 to +0.00003 for it on their files. Our files already hold megayak's LM as a stacked member (since v23), at the weight the labelled rows chose.
One board step of 0.00001 is below the noise v26 measured, so "no change" and "+0.00001" cannot be told apart here. What the board does not show is a gain.

### v28: the 0.95370 file, taken apart and replayed on labelled rows

**The question.** On 2026-11-13 the public best is **0.95370**: a file by [Aman Atar](https://www.kaggle.com/datasets/amanatar/s6e11-rank1-q17-meta-bundle), which
[goodpjw2008's LM Ladder | Public vs CV Check](https://www.kaggle.com/code/goodpjw2008/s6e11-lm-ladder-public-vs-cv-check-lb-0-95370) starts from.
Our best is 0.95305, and our own stack 0.95260. The file has no out-of-fold predictions. What is in it, and what would its mix score on rows with labels?

**The method (goodpjw2008's, run here against a second referee).** Aman Atar's bundle ships the file together with the test *and* out-of-fold predictions of 16 public models
(megayak's and masayakawamata's language models, goodpjw2008's two new seeds, two TabPFN runs, yekenot's RealMLP, sadamtorres' GBDTs, goodpjw2008's MLP and GB bags, sameershaikh27's ten-fold GLM boost, daoviet's GP + DVAE, megayak's shallow trees, and `alex_glm_gen`, a GLM member of this notebook).
1. Turn everything into logit ranks. Fit the file by least squares on the 16 models' **test** predictions. The weights say what the file is made of.
2. Apply the same weights to the 16 models' **out-of-fold** predictions. This is the file's mix on rows with labels: the replay.
3. Compare the replay with a stack that was fitted on labels. goodpjw2008 used their wide LM stack (nested CV 0.95257). We use ours: the wide stacker's held-out score on split 7, where every row is scored by a stacker that did not see it.
   The two stacks were built by different people from overlapping public members, so this is a second measurement of the same gap, not a copy of the first.

**What we measured** (same code on the published files before this version ran, `scratch/banks/v28_1113/sim28.py` in github.com/alexchilton/kaggley; the cell below prints its own numbers; 1 point = 0.00001 AUC):

| | AUC on labelled rows | public LB |
|---|---|---|
| the 0.95370 file (its mix replayed on out-of-fold rows) | 0.95186 | **0.95370** |
| 90% file + 10% our wide stack | 0.95200 (+13.4 points, 5 of 5 folds) | this version writes it: `submission_a370_wide10.csv` |
| 75% file + 25% our wide stack | 0.95217 (+30.9, 5 of 5) | `submission_a370_wide25.csv` |
| 50% / 50% | 0.95239 (+52.8, 5 of 5) | |
| our wide stack alone (held-out) | **0.95257** (+70.1, 5 of 5) | 0.95260 |

- The 16 models explain **98.3%** of the file on the test rows. The replay scores **0.95186**, the number goodpjw2008's notebook prints for the same step.
- Our held-out stack is **70 points above the replay**. On random subsets of the labelled rows the gap is -72 ± 37 points at the public board's size (59,996 rows; the file is ahead in 4 of 100 draws) and -70 ± 15 at the private board's size (239,987 rows; 0 of 100).
  On the public board the file is 110 points *above* our stack (0.95370 against 0.95260). goodpjw2008 measured -67 ± 34 against their stack. Two referees, one answer: the order on labelled rows is the reverse of the order on the public board, and sampling noise of the public rows does not cover the difference.
- **What the file is made of, and what the labels would choose for the same 16 models.** The cell fits a second set of weights on the same 16 out-of-fold columns, this time on the labels (logistic stacker, C 0.01, held out on the members' five folds).
  That stack scores **0.95211**: +25 points over the file's weights with the same members. The rest of the gap to our wide stack (+45) comes with the wider pool (167 members).
  Where the two sets of weights differ (file / labels, both scaled to the same sum): sadamtorres' XGBoost 0.67 / 0.16 and goodpjw2008's GB bag 0.61 / 0.18 (the file holds three to four times more);
  masayakawamata's GPT-2 0.06 / 0.18, its second seed -0.02 / 0.15, sameershaikh27's ten-fold GLM boost -0.04 / 0.21, megayak's shallow trees -0.19 / 0.14, sadamtorres' LightGBM -0.30 / 0.05 (the file holds much less, or subtracts).
  Where they agree: megayak's LM is large in both (0.48 / 0.18), and our GLM member is **subtracted in both** (-0.31 / -0.24). A negative weight is not by itself a mark of board fitting: the labels subtract that member too.
  The ten-fold GLM boost is the member v18 to v21 found raising our held-out AUC while lowering our public score (section 4). The file, tuned on the public board, drops it; the labels keep it. Same disagreement, seen from the other side.
- **More of a CV-backed stack helps at every share** on labelled rows. goodpjw2008 scored two such mixes on the board with their stack: 10% cost 2 public points (0.95368), 25% cost 8 (0.95362).

**What this does and does not show.** The replay is a stand-in: 1.7% of the file is not explained by the 16 models and is not replayed. The bundle also ships an `oof_anchor_095370` column; we do not use it, for goodpjw2008's reason:
it scores 0.95254 and has rank correlation 0.9987 with our held-out stack, while the replay of the file's own weights has 0.9943. It is the out-of-fold column of a wide stack, not of this file's weights.
This is not a verdict on the file or its author; every ladder on this board was built with public probes, and the authors publish them.
The practical reading is the one goodpjw2008 and areebbb give: for the two final picks, one best-public file and one file with a score on labelled rows. The two mix files below sit between them.

### v28: two more language-model seeds in the wide pool

**What it is.** goodpjw2008 trained a second seed of each public target-first language model and published the predictions on our folds (`StratifiedKFold(5, shuffle=True, random_state=42)`, CC0):
[seed 7 of megayak's compact LM](https://www.kaggle.com/datasets/goodpjw2008/s6e11-compact-lm-seed7) (five folds, OOF 0.94945) and
[seed 2026 of masayakawamata's GPT-2](https://www.kaggle.com/datasets/goodpjw2008/s6e11-target-first-gpt2-seed-2026) (OOF on five folds 0.95014; the test predictions come from the fold-0 model only, the other four are still scoring).
v28 adds both to the wide pool as members 168 and 169 and refits the same logistic stacker (C 0.01).

**Why it should help.** The two first seeds hold the largest weights in our wide stack. A language model trained with another seed makes the same kind of prediction with different noise, so the pair is a less noisy version of one member.
The public ladders use them as a *rung*: `+ 0.06 (second seed - first seed)`, which moves 0.06 of the weight from one seed to the other. goodpjw2008 and areebbb call the GPT-2 rung "the first rung that gains on both boards" (+2 points public, +1.9e-5 on nested CV, 5 of 5 folds).
A stacker can do more than a rung: it picks each seed's weight on all 700k labelled rows.

**What we measured** (offline, same script):

| | split 7 | split 8 | split 9 |
|---|---|---|---|
| wide stacker, 167 members (held-out AUC) | 0.952566 | 0.952593 | 0.952552 |
| + both seeds, 169 members | 0.952587 | 0.952605 | 0.952572 |
| change, points | **+2.1** | **+1.2** | **+1.9** |
| + seed 7 of the compact LM only | +0.6 | | |
| + seed 2026 of the GPT-2 only | +1.5 | | |

| the public rung, no selection, all labelled rows | on our held-out wide stack | on the replayed 0.95370 file |
|---|---|---|
| + 0.06 (GPT-2 seed 2026 - masayakawamata's GPT-2) | +2.0 points, 5 of 5 folds | +0.2, 3 of 5 |
| + 0.06 (LM seed 7 - megayak's LM) | +1.3, 4 of 5 | +3.5, 5 of 5 |

- **A second seed is real information, and it is small.** Both seeds together add +1.2 to +2.1 points on three splits; every split is up. goodpjw2008's numbers for the rungs (+1.9 and +3.5) reproduce on our rows to the decimal for the file they were made on.
- **The stacker splits the weight by itself.** Before: masayakawamata's GPT-2 0.338, megayak's LM 0.299 (v27's log). With the seeds in: GPT-2 0.230 and its second seed 0.212, megayak's LM 0.232. The pair of GPT-2 seeds holds 0.44 where one seed held 0.34: two noisy copies are worth more weight than one.
- **The public board cannot see a gain of this size.** On random held-out subsets of the public board's size the change is +1.3 ± 7.6 points and positive in 60% of 200 draws; at the private board's size +2.6 ± 3.2, positive in 84%.
  A rung that "gained 2 points on the board" and a member that adds 2 points on 700k labelled rows are different kinds of evidence, even when the number is the same.
- **`submission.csv` does not change.** Our bar for a member to enter the submitted stack is +5 points (0.00005) on the mean of the three splits with every split up. +1.7 is below it.
  The 169-member stack is written as `submission_wide_seeds.csv` for the board; on the test rows it is 2.2e-4 from the wide stack (1 - rank correlation).
- A caveat on seed 2026: its test predictions come from one fold model. Its out-of-fold column is as good as the first seed's (0.95014 against 0.95003), but its test column is the noisier of the two until the other four fold models are published.

### v28: two checks on every member of the pool

A wide pool takes other people's files on trust. [dariushafshar's 145 OOF Pairs, One Stack](https://www.kaggle.com/code/dariushafshar/s6e11-145-oof-pairs-one-stack-lb-0-95246) runs checks on every pair before stacking it. Two of them, on our 169 columns:

**1. The fold signature.** Out-of-fold predictions come from five models, one per fold, and each model has its own offset. So the mean prediction differs between the five folds by more than chance allows, *if* the column was built on these folds.
`F` = the variance of the five fold means, divided by what random groups of the same size would give. About 1 means no signature. The control: the same statistic on a split no member was built on (seed 7).

- On our folds: median F **74**; on the control split: median 0.52, largest 2.3. The statistic separates the two cases cleanly.
- Six members have F below 5 on our folds: queyrusi's digit CNN bag (0.0), three of sadamtorres' v6 pairs (1.1 to 1.2), ektarr's RealMLP stack (1.2) and sadamtorres' raw LightGBM (3.6).
  sameershaikh27's ten-fold GLM boost has 5.0, as it should: ten folds, not our five. goodpjw2008's seed-7 LM has 6.0 and is on our folds by its README.
- **A low F is a question, not a verdict.** We looked at the lowest one. queyrusi's bank ships its fold ids, and they are our partition; its out-of-fold column has mean 0.5 and the same 10% and 90% quantiles in every fold: it was turned into ranks *inside each fold*, which removes the signature by construction.
  So F says "this column does not look like five raw fold models on these folds": other folds, a bag over several splits, or a per-fold rank transform. For the five others we have no fold ids to check, and their out-of-fold claim rests on the author's description.
  The stacker that weights them is itself held out, so a leak would show as a held-out AUC that the board does not confirm. v18 to v21 saw that pattern after the ten-fold member came in (section 4). HYPOTHESIS (unverified) that the two are linked.

**2. Is a member closer to the stack on the test rows than out of fold?** For each member: its rank correlation with the wide stack on out-of-fold rows, and the same on the test rows.
An out-of-fold row is scored by **one** fold model. A test row is usually scored by the **average of five** (or by a refit on all rows). An average is less noisy, so the test column sits closer to the stack than the OOF column.

- Nearly every member is closer to the stack on test. Medians by source: members trained here +0.0058, public banks +0.0035, the public pool +0.0038 (v28's text said +0.006, +0.005 and +0.004, written from a smoke run; corrected in v29 to the values the cell prints). The neural members move most: RealMLP on digits +0.038, RealMLP +0.032, TabM +0.021.
- The language models give a clean natural experiment. megayak's LM and seed 7 average five fold models on test: **+0.0024** and **+0.0035**. Seed 2026 scores the test rows with the fold-0 model only: **0.0000**. masayakawamata's GPT-2: +0.0005.
- Why it matters: the stacker learns each member's weight from its out-of-fold column, the noisy version. A member whose test column is five times averaged is better on test than the stacker was told, so its weight is too low for the test rows.
  This is a measured piece of v27's hypothesis for why the public board rewards "a step towards the LM" that held-out rows reject. It is not proof: the board gains of the ladders are also the size of selection noise (v27).

**Credit.** Aman Atar (the 0.95370 file and the bundle of 16 models), goodpjw2008 (the replay method, the board scores of the mixes, both new seeds), megayak, masayakawamata and Matapanino (the language-model code and first seeds), Chris Deotte (the target-first recipe),
dariushafshar (the pool checks), asteriospanoras ([How Much of the Public Top Is Real?](https://www.kaggle.com/code/asteriospanoras/s6e11-how-much-of-the-public-top-is-real): the noise of a public gap on out-of-fold rows), areebbb (the two-seed rung rebuilt and probed), and the authors of the 16 models in the bundle. Please upvote them.


In [ ]:
# v28: the public 0.95370 file taken apart and replayed on labelled rows, two more language-model seeds in the wide pool, and two checks on every pool member
from sklearn.linear_model import LinearRegression
_h28 = {'rank1_q17_streams.npz': ('rank1-q17', '0fe4ab1263a673c8065d7145590b92da440263161bf860ce45f2976b5c4fa56f'),
        'lm7_oof_5fold.npy': ('compact-lm', 'c56d6b7282e7ea8c4c5927ac27e8b286d539ffbd66d5795564323c8b92b65242'), 'lm7_test_5fold.npy': ('compact-lm', '68639fec79f081e97d40185443fa3903c9db3e62698abb01dfe31dcee217462a'),
        'gpt2_s2026_oof_5fold.npy': ('seed-2026', '4a902ecd3d725d21123f5b6239ac2190a9fd94f3de37e3157be391fcf3fe996a'), 'gpt2_s2026_test_f0.npy': ('seed-2026', '40e08058ed7632ee328c2fe7d97040b856b36d2636f32cc6d2781ec9db080695')}
P28 = {}
for f_, (own_, h_) in _h28.items():
    p_ = _find(f_, own_)
    if p_ is None: print(f_, 'not mounted, skipped'); continue
    hx = hashlib.sha256(open(p_, 'rb').read()).hexdigest(); print(f_, 'SHA-256 matches the 2026-11-13 download' if hx == h_ else f'SHA-256 CHANGED: {hx}'); P28[f_] = p_
yv28 = y.to_numpy().astype(int); F28 = _lr(wide_hold[7]); a28 = _fauc(yv28, F28); n_pub28, n_prv28 = int(0.2 * len(XT)), int(0.8 * len(XT))
def _ups28(v, b): return sum(_fauc(yv28[fold42 == k], v[fold42 == k]) > _fauc(yv28[fold42 == k], b[fold42 == k]) for k in range(5))
def _dist28(a, b): return (1 - np.corrcoef(rankdata(a), rankdata(b))[0, 1]) * 1e4

# 1. the 0.95370 file: least squares on the bundle's 16 public models (test rows), the same weights replayed on their out-of-fold predictions
W28 = M28 = AO28 = None
if 'rank1_q17_streams.npz' in P28:
    Zb = np.load(P28['rank1_q17_streams.npz'], allow_pickle=False); assert (Zb['y_train'] == yv28).all() and _same_folds(Zb['folds'])
    N28 = [str(s) for s in Zb['stream_names']]
    XO28 = np.column_stack([_lr(Zb['oof_' + n]) for n in N28]); XT28 = np.column_stack([_lr(Zb['test_' + n]) for n in N28]); AT28 = _lr(Zb['test_anchor_095370'])
    fit28 = LinearRegression().fit(XT28, AT28); AO28 = _lr(XO28 @ fit28.coef_)
    sl_ = np.zeros(len(yv28))                              # the same 16 models, weighted by the labels instead: our stacker's settings, held out on the members' five folds
    for k in range(5): sl_[fold42 == k] = LogisticRegression(C=0.01, max_iter=3000, tol=1e-6).fit(XO28[fold42 != k], yv28[fold42 != k]).decision_function(XO28[fold42 == k])
    cl_ = LogisticRegression(C=0.01, max_iter=3000, tol=1e-6).fit(XO28, yv28).coef_[0]; LAB28 = _fauc(yv28, sl_)
    W28 = pd.DataFrame({'weight in the file': fit28.coef_, 'weight the labels choose (same sum)': cl_ / cl_.sum() * fit28.coef_.sum(), 'OOF AUC alone': [_fauc(yv28, XO28[:, j]) for j in range(len(N28))]}, index=N28).sort_values('weight in the file', ascending=False)
    print(f'the 16 public models explain R2 = {fit28.score(XT28, AT28):.4f} of the 0.95370 file on the test rows'); print(W28.round({'weight in the file': 3, 'weight the labels choose (same sum)': 3, 'OOF AUC alone': 5}).to_string())
    print(f'the same 16 models with weights fitted on the labels (held out): AUC {LAB28:.5f}, {(LAB28 - _fauc(yv28, AO28)) * 1e5:+.1f} points against the weights of the file; correlation of the two weight vectors {np.corrcoef(fit28.coef_, cl_)[0, 1]:.2f}')
    print(f'the mix replayed on out-of-fold rows: AUC {_fauc(yv28, AO28):.5f} | our wide stacker, held-out (split 7): {a28:.5f} | gap {(_fauc(yv28, AO28) - a28) * 1e5:+.1f} points (1 point = 0.00001)')
    rng28 = np.random.default_rng(28); G28 = {}
    for n_ in (n_pub28, n_prv28):
        d_ = []
        for _ in range(100):
            i_ = rng28.choice(len(yv28), n_, replace=False); d_.append((_fauc(yv28[i_], AO28[i_]) - _fauc(yv28[i_], F28[i_])) * 1e5)
        G28[n_] = np.array(d_); print(f'gap (replayed file - our wide) on {n_} random rows: {np.mean(d_):+.1f} +- {np.std(d_):.1f} points; the file ahead in {np.mean(np.array(d_) > 0):.0%} of 100 draws')
    M28 = pd.DataFrame([[w, _fauc(yv28, (1 - w) * AO28 + w * F28), (_fauc(yv28, (1 - w) * AO28 + w * F28) - _fauc(yv28, AO28)) * 1e5, _ups28((1 - w) * AO28 + w * F28, AO28)] for w in (0.0, 0.1, 0.25, 0.5, 0.75, 1.0)],
                       columns=['share of our wide stack', 'AUC on labelled rows', 'points vs the file', 'folds up of 5']).set_index('share of our wide stack')
    print('the replayed file mixed with our held-out wide stack:'); print(M28.round({'AUC on labelled rows': 5, 'points vs the file': 1}).to_string())
    T['amanatar 0.95370 file'] = AT28
    for w, n_ in [(0.10, 'submission_a370_wide10.csv'), (0.25, 'submission_a370_wide25.csv')]:
        T[f'{1 - w:.2f} file + {w:.2f} our wide'] = (1 - w) * AT28 + w * _lr(t_w); v_ = T[f'{1 - w:.2f} file + {w:.2f} our wide']
        pd.DataFrame({'id': test['id'], TARGET: rankdata(v_) / len(v_)}).to_csv(OUT / n_, index=False); print('wrote', n_)
    k28 = [k for k in ['our wide', 'mix 0.65 Part 12 + 0.35 our wide', 'areebbb ladder, rebuilt (LB 0.95347)', 'amanatar 0.95370 file', '0.90 file + 0.10 our wide', '0.75 file + 0.25 our wide'] if k in T]
    D28 = pd.DataFrame((1 - np.corrcoef([rankdata(T[k]) for k in k28])) * 1e4, index=k28, columns=[str(i) for i in range(len(k28))])
    print('test files: distance 1 - rank correlation, x 1e-4 (columns in row order)'); print(D28.round(1).to_string())
    print(f'labelled rows: distance of the replayed file from our held-out wide stack {_dist28(AO28, F28):.1f}e-4')

# 2. two more language-model seeds as members of the wide pool (goodpjw2008's seed-7 compact LM and seed-2026 GPT-2)
new28 = {}
if 'lm7_oof_5fold.npy' in P28 and 'lm7_test_5fold.npy' in P28: new28['goodpjw2008 LM seed 7'] = (np.load(P28['lm7_oof_5fold.npy']).astype(float).ravel(), np.load(P28['lm7_test_5fold.npy']).astype(float).ravel())
if 'gpt2_s2026_oof_5fold.npy' in P28 and 'gpt2_s2026_test_f0.npy' in P28: new28['goodpjw2008 GPT-2 seed 2026'] = (np.load(P28['gpt2_s2026_oof_5fold.npy']).astype(float).ravel(), np.load(P28['gpt2_s2026_test_f0.npy']).astype(float).ravel())
new28 = {k: v for k, v in new28.items() if len(v[0]) == len(X) and len(v[1]) == len(XT) and np.isfinite(v[0]).all() and np.isfinite(v[1]).all()}
_tw28 = {'goodpjw2008 LM seed 7': 'megayak target-first LM', 'goodpjw2008 GPT-2 seed 2026': 'masayakawamata GPT-2 target-first'}       # each new member and its first seed, already in the pool
for k, (o_, t_) in new28.items():
    print(f'{k}: OOF AUC {_fauc(yv28, o_):.5f}' + (f'; rank correlation with {_tw28[k]} {np.corrcoef(rankdata(o_), rankdata(pool[_tw28[k]][0]))[0, 1]:.4f} (OOF)' if _tw28[k] in pool else ''))
R28 = []
for k in new28:
    if _tw28[k] not in pool: continue
    d_ = 0.06 * (_lr(new28[k][0]) - _lr(pool[_tw28[k]][0]))
    for lab, b_ in [('our held-out wide stack', F28)] + ([('the replayed 0.95370 file', AO28)] if AO28 is not None else []):
        R28.append([f'+ 0.06 ({k} - {_tw28[k]})', lab, (_fauc(yv28, b_ + d_) - _fauc(yv28, b_)) * 1e5, _ups28(b_ + d_, b_)])
if R28: print('a rung of 0.06 from the first seed to the second, no selection, all labelled rows:'); print(pd.DataFrame(R28, columns=['rung', 'on', 'points', 'folds up of 5']).round(2).to_string(index=False))
w28_runs, w28_hold, wn28 = [], {}, wnames + list(new28)
if new28:
    ZW28 = np.column_stack([ZW] + [lr_(v[0]) for v in new28.values()]); ZWT28 = np.column_stack([ZWT] + [lr_(v[1]) for v in new28.values()])
    for seed in (7, 8, 9):
        s = np.zeros(len(ZW28))
        for va_i, dv in Parallel(n_jobs=-1)(delayed(_fold)(ZW28, y.values, tr_i, va_i) for tr_i, va_i in StratifiedKFold(5, shuffle=True, random_state=seed).split(ZW28, y)): s[va_i] = dv
        w28_runs.append(roc_auc_score(y, s)); w28_hold[seed] = s
    print(f'held-out AUC over splits 7, 8, 9: wide stacker ({len(wnames)} members) {np.mean(wide_runs):.5f} {np.round(wide_runs, 5)} | with the two seeds ({len(wn28)} members) {np.mean(w28_runs):.5f} {np.round(w28_runs, 5)} | '
          f'change in points {np.round((np.array(w28_runs) - np.array(wide_runs)) * 1e5, 2)}')
    d_ = [(_fauc(yv28[fold42 == k], w28_hold[7][fold42 == k]) - _fauc(yv28[fold42 == k], wide_hold[7][fold42 == k])) * 1e5 for k in range(5)]; print('split 7, change by fold of the members (points):', np.round(d_, 2))
    rng28b = np.random.default_rng(29)
    for n_ in (n_pub28, n_prv28):
        d_ = []
        for _ in range(200):
            i_ = rng28b.choice(len(yv28), n_, replace=False); d_.append((_fauc(yv28[i_], w28_hold[7][i_]) - _fauc(yv28[i_], wide_hold[7][i_])) * 1e5)
        print(f'the same change on {n_} random held-out rows: {np.mean(d_):+.2f} +- {np.std(d_):.2f} points; up in {np.mean(np.array(d_) > 0):.0%} of 200 draws')
    wstk28 = LogisticRegression(C=0.01, max_iter=3000, tol=1e-6).fit(ZW28, y); wc28 = pd.Series(wstk28.coef_[0], index=wn28); t28 = wstk28.decision_function(ZWT28)
    print('largest weights with the two seeds in:'); print(wc28.reindex(wc28.abs().sort_values(ascending=False).index).head(12).round(3).to_string())
    print('the four language models:', {k: round(float(wc28[k]), 3) for k in list(new28) + list(_tw28.values()) if k in wc28.index}, '| before:', {k: round(float(wcoef[k]), 3) for k in _tw28.values() if k in wcoef.index})
    pd.DataFrame({'id': test['id'], TARGET: rankdata(t28) / len(t28)}).to_csv(OUT / 'submission_wide_seeds.csv', index=False)
    print(f'wrote submission_wide_seeds.csv; test side, distance from our wide stack {_dist28(t28, t_w):.2f}e-4')
    del ZWT28

# 3. two checks on every member of the pool (dariushafshar's fold signature; out-of-fold against test correlation with the stack)
def _foldF28(z, f): m = np.array([z[f == k].mean() for k in range(5)]); return float(m.var(ddof=1) / (z.var() / (len(z) / 5)))
f7_28 = np.zeros(len(X), int)
for f_, (_, va_i) in enumerate(StratifiedKFold(5, shuffle=True, random_state=7).split(np.zeros(len(X)), y)): f7_28[va_i] = f_
_cols28 = [(k, ZW[:, j], ZWT[:, j]) for j, k in enumerate(wnames)] + [(k + ' (new)', lr_(v[0]), lr_(v[1])) for k, v in new28.items()]
_ro, _rt = rankdata(wide_hold[7]), rankdata(t_w)
C28 = pd.DataFrame([[k, _fauc(yv28, o_), _foldF28(o_, fold42), _foldF28(o_, f7_28), np.corrcoef(rankdata(o_), _ro)[0, 1], np.corrcoef(rankdata(t_), _rt)[0, 1]] for k, o_, t_ in _cols28],
                   columns=['member', 'OOF AUC', 'fold F, our folds', 'fold F, other folds', 'rho OOF', 'rho test']).set_index('member')
C28['rho test - rho OOF'] = C28['rho test'] - C28['rho OOF']
_lm28 = set(_tw28) | set(_tw28.values())
def _src28(k): return 'language model' if k.replace(' (new)', '') in _lm28 else 'public pool' if k in pool else 'public bank' if k.startswith('bank ') else 'trained here'
C28['source'] = [_src28(k) for k in C28.index]
print(f"fold signature F on our folds: median {C28['fold F, our folds'].median():.0f}, smallest {C28['fold F, our folds'].min():.1f}; on folds no member was built on (split 7): median {C28['fold F, other folds'].median():.2f}, largest {C28['fold F, other folds'].max():.1f}")
print('members with the weakest fold signature:'); print(C28.sort_values('fold F, our folds').head(8).round(4).to_string())
print('test correlation minus out-of-fold correlation with the wide stack, by source:'); print(C28.groupby('source')['rho test - rho OOF'].describe()[['count', 'min', '50%', 'max']].round(4).to_string())
print('the language models:'); print(C28[C28['source'] == 'language model'].round(4).to_string())
fig, ax28 = plt.subplots(2, 2, figsize=(14, 9)); (a1, a2), (a3, a4) = ax28
if W28 is not None:
    w_ = W28.sort_values('weight in the file'); yy_ = np.arange(len(w_))
    a1.barh(yy_ + 0.2, w_['weight in the file'], height=0.4, color='#c98b2b', label='in the 0.95370 file (least squares, test rows)'); a1.barh(yy_ - 0.2, w_['weight the labels choose (same sum)'], height=0.4, color='#4a6fa5', label='chosen by the labels (logistic stacker)')
    a1.axvline(0, color='k', lw=0.8); a1.set_yticks(yy_); a1.set_yticklabels(w_.index, fontsize=7); a1.legend(fontsize=7, loc='lower right')
    a1.set_title(f'the same 16 public models, two sets of weights (the file: R2 {fit28.score(XT28, AT28):.3f})', fontsize=10)
    a2.plot(M28.index, M28['AUC on labelled rows'], 'o-', color='#4a6fa5'); a2.axhline(LAB28, color='#c98b2b', ls='--', lw=1, label='the 16 models, weights fitted on labels'); a2.legend(fontsize=8, loc='lower right')
    for w, lb in [(0.0, 'public 0.95370'), (0.1, 'goodpjw2008, their stack: public 0.95368'), (0.25, 'their stack: public 0.95362')]: a2.annotate(lb, (w, M28.loc[w, 'AUC on labelled rows']), textcoords='offset points', xytext=(8, -4), fontsize=7)
    a2.set_xlabel('share of our wide stack (rest: the 0.95370 file, replayed on out-of-fold rows)'); a2.set_ylabel('AUC on labelled rows'); a2.set_title('more of a CV-backed stack: labelled rows gain at every share', fontsize=10)
a3.scatter(C28['fold F, other folds'].clip(lower=0.01), C28['fold F, our folds'].clip(lower=0.01), s=14, c=C28['source'].map({'trained here': '#4a6fa5', 'public bank': '#7fa37f', 'public pool': '#c98b2b', 'language model': '#b0413e'}))
a3.set_xscale('log'); a3.set_yscale('log'); a3.axhline(5, color='#999', ls='--', lw=1); a3.set_xlabel('fold signature F on folds no member was built on (split 7)'); a3.set_ylabel('F on our folds (seed 42)')
a3.set_title('fold signature: was this member built on our folds?', fontsize=10)
Q28 = C28[C28['rho OOF'] > 0.9]                             # the kNN member (0.57 / 0.66) is left out of the panel so the rest can be read
for s_, c_ in [('trained here', '#4a6fa5'), ('public bank', '#7fa37f'), ('public pool', '#c98b2b'), ('language model', '#b0413e')]:
    q_ = Q28[Q28['source'] == s_]; a4.scatter(q_['rho OOF'], q_['rho test - rho OOF'], s=28 if s_ == 'language model' else 14, color=c_, label=s_, zorder=3 if s_ == 'language model' else 2)
for k in Q28.index[Q28['source'] == 'language model']: a4.annotate(k.replace('goodpjw2008 ', '').replace('masayakawamata ', '').replace('megayak ', '').replace(' target-first', '').replace(' (new)', ''), (Q28.loc[k, 'rho OOF'], Q28.loc[k, 'rho test - rho OOF']), textcoords='offset points', xytext=(6, 4), fontsize=8, color='#b0413e')
a4.axhline(0, color='#999', lw=1); a4.set_xlabel('rank correlation with the wide stack, out-of-fold rows'); a4.set_ylabel('the same on the test rows, minus the out-of-fold value')
a4.set_title('is a member closer to the stack on test than out of fold?', fontsize=10); a4.legend(fontsize=8); plt.tight_layout(); plt.show()
if new28 and all(a_ > b_ for a_, b_ in zip(w28_runs, wide_runs)) and np.mean(w28_runs) - np.mean(wide_runs) >= 0.00005:
    pd.DataFrame({'id': test['id'], TARGET: rankdata(t28) / len(t28)}).to_csv(OUT / 'submission.csv', index=False); print('the two seeds pass the bar (+0.00005 on the mean of three splits, every split up): submission.csv is the stack with the seeds')
else:
    print('the two seeds are below the bar (+0.00005 on the mean of three splits, every split up): submission.csv stays as written above')


### What the board said about v28 (submitted 2026-11-13)

| file | what | public LB | AUC on labelled rows |
|---|---|---|---|
| `submission_a370_wide10.csv` | 90% Aman Atar's 0.95370 file + 10% our wide stack | **0.95367** | 0.95200 (replay) |
| `submission_a370_wide25.csv` | 75% file + 25% our wide stack | 0.95359 | 0.95217 (replay) |
| `submission_wide_seeds.csv` | our wide stack + two language-model seeds (169 members) | 0.95260 | 0.95259 (held-out, mean of 3 splits) |
| `submission.csv` | our wide stack (167 members), as v23 | 0.95260 | 0.95257 |

- Ten percent of our stack costs the 0.95370 file 3 public points (1 point = 0.00001) and adds 13 on labelled rows; 25% costs 11 and adds 31. goodpjw2008 measured 2 and 8 with their own stack: two independent stacks, the same trade.
- The board and the labelled rows rank the three files (file, 90/10, 75/25) in opposite order, in steps of 3 to 8 points. v26 measured the noise of a public difference between files this close at 3 to 9 points, so the board's order of these three is not established.
- The two seeds: +0.00000 on the board, +1.7 points held-out. The public-sized noise of that change is ± 7.6 points (v28), so the board could not have shown it.

### v29: the 0.95388 file, rebuilt byte for byte and scored on labelled rows

**The question.** On 2026-11-14 the public best is **0.95388**, [areebbb's Bold, Then Bolder](https://www.kaggle.com/code/areebbb/s6e11-bold-then-bolder-lb-0-95388), one step past
[goodpjw2008's LM Ladder | Public and CV Up](https://www.kaggle.com/code/goodpjw2008/s6e11-lm-ladder-public-and-cv-up-lb-0-95386) (0.95386).
Both report a gain on out-of-fold rows in all five folds, which no earlier ladder did. What does that gain mean next to a stack fitted on the labels?

**How the files are made (goodpjw2008's method).**
1. Start from the 0.95370 file with a small language-model step: `base = rank(file + 0.06 (LM seed 7 - megayak's LM))`.
2. Move towards 87 public members (the 16 models of Aman Atar's bundle, goodpjw2008's wide LM stack and 70 models of goodpjw2008's OOF bank): `score = (1 - sum w) logit(base) + sum w_k logit_rank(member_k)`.
3. Pick the weights `w` to raise two things together: the out-of-fold AUC of the same mix, and a *forecast of the public score*.
   The forecast is a ridge model fitted on goodpjw2008's 44 scored files: each scored file is one linear equation about the public rows' labels, so a new file's public score can be predicted from how it re-ranks rows (1.4 points error, leave one out).
   Three settings of that trade gave three files: careful (0.95376), balanced (0.95380), bold (0.95386).
4. areebbb extrapolated: `bolder = bold + 0.5 (bold - balanced)`, public **0.95388**.

Both notebooks publish every weight, and each input file is pinned by SHA-256. This cell rebuilds the bolder file from those weights and checks its SHA-256 against areebbb's published one.

**The out-of-fold side.** The file itself has no out-of-fold predictions. goodpjw2008 replays it: least squares of the 0.95370 file on the 16 bundle models *and* their wide stack (test rows), the same weights on the out-of-fold columns, plus the same LM step. Call that the reference.
The mix weights are then applied to the 87 members' out-of-fold columns. The weights were chosen with these out-of-fold rows in view, so every mix number below is **in-sample** (optimistic); goodpjw2008's nested check keeps about 60% of it for the bold mix (+19.5 of +32.8).
Our referee is the same as in v28: the wide stacker's held-out score on split 7, where every row is scored by a stacker that did not see it.

**What we measured** (offline on the published files before this version ran, `scratch/banks/v29_1114/sim29.py`; the cell below prints its own numbers):

| | AUC on labelled rows | points against our held-out stack | public LB |
|---|---|---|---|
| the 0.95370 file, replayed (16 models) | 0.95186 (v28) | -70 | 0.95370 |
| the reference: 16 models + their wide stack + LM step | 0.95208 | -48 | 0.95370 |
| careful (in-sample) | 0.95237 | -20 | 0.95376 |
| balanced (in-sample) | 0.95227 | -29 | 0.95380 |
| bold (in-sample) | 0.95241 | -15 | 0.95386 |
| **bolder, the 0.95388 file** (in-sample) | **0.95246** | **-11** | **0.95388** |
| our wide stack (held-out) | **0.95257** | 0 | 0.95260 |

- **The rebuild is exact.** Our file has the SHA-256 areebbb publishes (`b0979694...`). Every test input is therefore the one they used.
- **The ladder closed most of the gap on labelled rows.** The 0.95370 file sat 70 points below our held-out stack; the 0.95388 file sits 11 below, and that number is in-sample. On random subsets of public-board size the gap is -9 ± 27 points (the file ahead in 37 of 100 draws), at private-board size -11 ± 13 (19 of 100).
  For the first time a public ladder file is within sampling noise of a label-fitted stack on labelled rows, measured by a second referee.
- **Where the gain comes from.** The largest moves of the bold mix: megayak's shallow trees +0.28, daoviet's GP + DVAE +0.14, megayak's LM -0.12, sadamtorres' XGBoost -0.10, masayakawamata's GPT-2 -0.07.
  v28 found the 0.95370 file holding megayak's shallow trees at -0.19 where the labels choose +0.14, and sadamtorres' XGBoost at 0.67 where the labels choose 0.16. The bold move undoes part of both: it moves the file towards the weights the labels choose.

**The same move on our stack.** If the 87-member move carries information that a label-fitted stack lacks, it should help ours too. Applied to our held-out wide stack instead of the reference, `(1 - sum w) stack + sum w_k member_k`:

| move | points on the reference (in-sample) | points on our held-out stack | folds up |
|---|---|---|---|
| careful | +28.4 | **+4.6** | 3 of 5 |
| balanced | +19.0 | +2.3 | 3 of 5 |
| bold | +32.8 | -1.3 | 1 of 5 |
| bolder (0.95388) | +37.1 | -6.3 | 1 of 5 |
| bold + 1.0 (bold - balanced) | +39.8 | -13.6 | 1 of 5 |
| bold + 2.0 (bold - balanced) | +40.1 | -34.6 | 1 of 5 |

- **The direction that raises the reference lowers our stack.** Along `bold - balanced`, the reference gains up to +40.8 points (areebbb's table reproduces to the decimal) and our stack loses up to 35, with one fold of five up.
  The move is a correction of the 0.95370 file's weights towards what labels choose. A stack that was fitted on the labels has no such error to correct, so the move only adds noise to it.
- **The careful mix gains +4.6 on our stack, below our +5 bar and with 3 of 5 folds.** It is also in-sample. `submission.csv` does not change.
- **The board peaks earlier than out-of-fold AUC** (areebbb: half a step 0.95388, 1.5 steps 0.95387, 2 steps 0.95385), and our stack peaks earlier still (at careful). Three referees, three different best step sizes; only the public one was used to choose.

**Board files from this version:** `submission_b388_wide10.csv` (90% the 0.95388 file + 10% our wide stack; on labelled rows +4.7 points over the file, 5 of 5 folds), `submission_b388_wide50.csv` (50 / 50; +15.3, 5 of 5, and +4.3 over our stack alone),
and `submission_wide_bold.csv` (our wide stack + the bold move; -1.3 points on labelled rows): if the board rewards the bold move on our stack too, the board and the labelled rows disagree about it.
The results go in the next version.

**Credit.** areebbb (the bolder step, the rebuild and its probes), goodpjw2008 (the method: moves chosen on CV and a public forecast together, the three mixes, the replay, the OOF bank and the wide LM stack), Aman Atar (the 0.95370 file and its bundle),
asteriospanoras (the noise of a public gap), and the authors of the 87 members (listed in both source notebooks). Please upvote them.


In [ ]:
# v29: the public 0.95388 file rebuilt from its published weights, replayed on labelled rows, and its move applied to our own stack
R29 = json.loads('{"bolder": {"s:alex_glm_gen": 0.004659655001642133, "b:lgb_diglead_l3": 0.07037499081158276, "s:daoviet_gpdvae": 0.16759272015867438, "b:realmlp_v1": 0.028265988392542088, "b:gb_bag10": -0.018152655535374172, "b:lgb_dig_l7": 0.026162003105571548, "b:lgb_dig_s100_bag5": 0.023205824985217123, "s:lm7_5fold": -0.04643526723210693, "b:lgb_te_l4_s300": 0.012845944273216886, "b:lgb_diglead_bag": -0.008520524531314792, "s:masaya_gpt2": -0.06938586034217065, "b:lgb_dig_s100": -0.06905761136150596, "b:gbx_bag3": -0.025385082895315353, "b:lgb_te_l4": 0.035196443631714015, "s:sadam_v6_gb_xgb": -0.13553456267863312, "b:lgb_dig_l4": 0.04984890049165017, "b:glm_sb": -0.015572335325911311, "b:glm_dl_s50_bag": -0.043295376575860016, "b:lgb_te_l4_s100": 0.010731651329495398, "b:glm_ohk_c01": -0.0005446893557200782, "b:gbn_bag5_w512": 0.0551862603361086, "b:glm_ohk": -0.022950181793756356, "b:glm_dl_bag": 0.015527551247048318, "b:realmlp_dig_s100": 0.019529287294532443, "b:gbr_bag3": -0.007872454837865235, "wide": -0.0567073526172608, "s:gbn_bag5_w512": 0.024585711783519787, "b:xgb_dig_d2": -0.04001405009889738, "b:ebm_i25_bag4": 0.052776831945719284, "s:megayak_shallow_tree": 0.3676405944329277, "s:gb_bag5": 0.0022942519878240766, "b:lgb_dig_s100_bag": -0.0031388296177910285, "s:gpt2_seed2026": -0.02172163397492512, "b:gbe_bag3": 0.016771631110015763, "s:yekenot_realmlp": -0.018220789336614255, "s:sadam_tabpfn_100k_tp3": -0.08146952344397645, "s:sadam_v6_gb_lgb": 0.09202978068795398, "b:glm_ohk2": -0.010746902043578546, "b:gb_ohk": -0.0403299060795074, "b:ebm_dig": -0.025668026406425494, "b:ebm_diglead_r11": 0.021711464683545754, "b:xgb_te_d2": -0.017507176405797825, "b:realmlp_rawdig": -0.03995919213462431, "b:gbn_bag5_w128": -0.03944938723524659, "s:sadam_v4_cat": 0.044170457123327754, "b:gb_l3": -0.08781795649166141, "b:lgb_diglead_bag5": -0.0691076844266097, "b:ebm_diglead_r13": 0.03797203360211246, "b:glm_dl": -0.0066911606091787906, "b:ebm_all_bag": -0.05809235897307825, "b:ebm_diglead_bag": -0.011299892151034639, "b:lgb_dig_s100_l3_bag": -0.017863982273807155, "b:realmlp_te_s2": -0.0007511365624976998, "b:realmlp_cnt": 0.010152092525701466, "b:ebm_i25_bag": 0.022755487867529373, "b:lgb_dig_s100_l3": 0.054036258199026535, "b:glm_dig": 0.004367002340381022, "b:lgb_diglead_l2": -0.060799370916732313, "b:lgb_diglead_l3_bag": -0.016998031665317204, "b:lgb_rawdig_l4": 0.018367968146419205, "b:realmlp_te_cnt": -0.0029453567331327295, "b:ebm_dig_r7": 0.04840096288719489, "b:gb_bag5": 0.033276434668693684, "b:gbx_bag5": 0.007256671781074796, "b:glm_ohkl": 0.010941852623233855, "s:tabpfn_slimypunk_tp1": -0.05059762171891728, "b:glm_dl_s50": -0.06411999038593352, "s:sameer_glm10f": 0.09096559940413664, "b:ebm_diglead_i25": -0.020509275358757188, "b:lgb_te_l4_s5": 0.011694566527880524, "b:ebm_dig_bag": -0.02860322960251588, "b:lgb_raw_l31": -0.00015178805153019635, "b:glm_ohk2_bag": -0.03766304548255879, "b:lgb_raw_l4": 0.0091529109828078, "b:xgb_raw_d3": 0.018532570780381338, "b:ebm_lead4_bag": -0.01900597337730603, "s:megayak_5f_lm": -0.1500914472131036, "b:glm_te": -0.005973898251156665, "b:glm_ohkl_bag": 0.021686855681833317, "b:realmlp_allcat": -0.014402601850564621, "b:lgb_raw_l4_cnt": -0.08232013696341364, "b:realmlp_dig": -0.034559470951024046, "b:lgb_dk_l4": 0.013571307927505006, "b:realmlp_dig_s2": 0.018038024291734817, "b:lgb_diglead_l4": 0.006437648823683151, "b:realmlp_te": 0.016229843827016463, "b:gbc_bag3": -0.016244727973699344}, "bold": {"s:megayak_5f_lm": -0.12205919443860914, "s:masaya_gpt2": -0.06997547490341371, "s:lm7_5fold": -0.040956430057266076, "s:gpt2_seed2026": -0.012508511564926699, "s:tabpfn_slimypunk_tp1": -0.039316699160350764, "s:sadam_tabpfn_100k_tp3": -0.061677762652753675, "s:yekenot_realmlp": -0.014637185195237004, "s:alex_glm_gen": 0.004537993603062768, "s:daoviet_gpdvae": 0.1359104270990657, "s:gbn_bag5_w512": 0.015432253915041055, "s:sameer_glm10f": 0.06272836350651828, "s:sadam_v6_gb_xgb": -0.10228295915095288, "s:sadam_v6_gb_lgb": 0.06627713517089104, "s:sadam_v4_cat": 0.03126101632182668, "s:gb_bag5": 0.008855983394809901, "s:megayak_shallow_tree": 0.28129985621486264, "wide": -0.034852407295305754, "b:ebm_all_bag": -0.050611080400723436, "b:ebm_dig": -0.016392951045769617, "b:ebm_dig_bag": -0.021971929033966105, "b:ebm_dig_r7": 0.03715898221284049, "b:ebm_diglead_bag": -0.011280640758210015, "b:ebm_diglead_i25": -0.01760600397929515, "b:ebm_diglead_r11": 0.020606132176681696, "b:ebm_diglead_r13": 0.031952532024431754, "b:ebm_i25_bag": 0.019937230510505454, "b:ebm_i25_bag4": 0.04011014273129195, "b:ebm_lead4_bag": -0.016285931935351022, "b:gb_bag10": -0.017390237028207023, "b:gb_bag5": 0.026782938331671603, "b:gb_l3": -0.06609634322241796, "b:gb_ohk": -0.02882833922678926, "b:gbc_bag3": -0.012871505523924395, "b:gbe_bag3": 0.012674094986094425, "b:gbn_bag5_w128": -0.029120675300013294, "b:gbn_bag5_w512": 0.04504705483635584, "b:gbr_bag3": -0.005251606394839109, "b:gbx_bag3": -0.01901561844474828, "b:gbx_bag5": 0.006799891770028022, "b:glm_dig": 0.003088377583964293, "b:glm_dl": -0.006428517981988452, "b:glm_dl_bag": 0.01502056793927036, "b:glm_dl_s50": -0.0501984153162254, "b:glm_dl_s50_bag": -0.028630562092033966, "b:glm_ohk": -0.017070904616440657, "b:glm_ohk2": -0.004903067440865727, "b:glm_ohk2_bag": -0.029877217995289047, "b:glm_ohk_c01": 0.0003448885987200072, "b:glm_ohkl": 0.006673473442067076, "b:glm_ohkl_bag": 0.01766365615186144, "b:glm_sb": -0.01145379830324528, "b:glm_te": -0.004792258197303907, "b:lgb_dig_l4": 0.0366960138715965, "b:lgb_dig_l7": 0.01949687340453925, "b:lgb_dig_s100": -0.051481356730372616, "b:lgb_dig_s100_bag": -7.44946546564366e-05, "b:lgb_dig_s100_bag5": 0.016566984017469158, "b:lgb_dig_s100_l3": 0.039879653631481535, "b:lgb_dig_s100_l3_bag": -0.014332571073509623, "b:lgb_diglead_bag": -0.00442949281469811, "b:lgb_diglead_bag5": -0.05069793900237127, "b:lgb_diglead_l2": -0.0434403013635199, "b:lgb_diglead_l3": 0.051384825410253424, "b:lgb_diglead_l3_bag": -0.014548087315257047, "b:lgb_diglead_l4": 0.005059024340666011, "b:lgb_dk_l4": 0.010395272568768744, "b:lgb_raw_l31": 6.272373140902098e-05, "b:lgb_raw_l4": 0.006062225361634528, "b:lgb_raw_l4_cnt": -0.060074422527671846, "b:lgb_rawdig_l4": 0.012821826099948688, "b:lgb_te_l4": 0.026453964192257394, "b:lgb_te_l4_s100": 0.007166689614341422, "b:lgb_te_l4_s300": 0.009325395175060891, "b:lgb_te_l4_s5": 0.00877347694863135, "b:realmlp_allcat": -0.010498730399732318, "b:realmlp_cnt": 0.00748969325179405, "b:realmlp_dig": -0.026045115561739243, "b:realmlp_dig_s100": 0.01482032375010226, "b:realmlp_dig_s2": 0.012780340101831603, "b:realmlp_rawdig": -0.029224792823957325, "b:realmlp_te": 0.011902655678168103, "b:realmlp_te_cnt": -0.0016811066269879565, "b:realmlp_te_s2": -0.00033527900132535184, "b:realmlp_v1": 0.02099235698670545, "b:xgb_dig_d2": -0.027214651902440812, "b:xgb_raw_d3": 0.013799277379315805, "b:xgb_te_d2": -0.012792356993134408}, "balanced": {"s:megayak_5f_lm": -0.0659946888896202, "s:masaya_gpt2": -0.07115470402589986, "s:lm7_5fold": -0.029998755707584368, "s:gpt2_seed2026": 0.005917733255070143, "s:tabpfn_slimypunk_tp1": -0.016754854043217735, "s:sadam_tabpfn_100k_tp3": -0.02209424107030813, "s:yekenot_realmlp": -0.007469976912482503, "s:alex_glm_gen": 0.004294670805904037, "s:daoviet_gpdvae": 0.07254584097984837, "s:gbn_bag5_w512": -0.002874661821916413, "s:sameer_glm10f": 0.006253891711281581, "s:sadam_v6_gb_xgb": -0.035779752095592386, "s:sadam_v6_gb_lgb": 0.014771844136765163, "s:sadam_v4_cat": 0.005442134718824526, "s:gb_bag5": 0.02197944620878155, "s:megayak_shallow_tree": 0.10861837977873254, "wide": 0.008857483348604336, "b:ebm_all_bag": -0.035648523256013816, "b:ebm_dig": 0.0021571996755421373, "b:ebm_dig_bag": -0.00870932789686656, "b:ebm_dig_r7": 0.014675020864131676, "b:ebm_diglead_bag": -0.011242137972560769, "b:ebm_diglead_i25": -0.011799461220371074, "b:ebm_diglead_r11": 0.018395467162953583, "b:ebm_diglead_r13": 0.019913528869070343, "b:ebm_i25_bag": 0.014300715796457614, "b:ebm_i25_bag4": 0.014776764302437287, "b:ebm_lead4_bag": -0.010845849051441005, "b:gb_bag10": -0.01586540001387272, "b:gb_bag5": 0.013795945657627443, "b:gb_l3": -0.022653116683931083, "b:gb_ohk": -0.005825205521352985, "b:gbc_bag3": -0.006125060624374499, "b:gbe_bag3": 0.004479022738251753, "b:gbn_bag5_w128": -0.008463251429546712, "b:gbn_bag5_w512": 0.02476864383685032, "b:gbr_bag3": -9.909508786856773e-06, "b:gbx_bag3": -0.006276689543614139, "b:gbx_bag5": 0.005886331747934474, "b:glm_dig": 0.0005311280711308347, "b:glm_dl": -0.0059032327276077766, "b:glm_dl_bag": 0.014006601323714443, "b:glm_dl_s50": -0.022355265176809174, "b:glm_dl_s50_bag": 0.0006990668756181338, "b:glm_ohk": -0.005312350261809259, "b:glm_ohk2": 0.00678460176455991, "b:glm_ohk2_bag": -0.014305563020749566, "b:glm_ohk_c01": 0.002124044507600178, "b:glm_ohkl": -0.0018632849202664824, "b:glm_ohkl_bag": 0.009617257091917682, "b:glm_sb": -0.0032167242579132175, "b:glm_te": -0.0024289780895983912, "b:lgb_dig_l4": 0.010390240631489167, "b:lgb_dig_l7": 0.006166614002474661, "b:lgb_dig_s100": -0.016328847468105947, "b:lgb_dig_s100_bag": 0.006054175271612747, "b:lgb_dig_s100_bag5": 0.0032893020819732264, "b:lgb_dig_s100_l3": 0.01156644449639154, "b:lgb_dig_s100_l3_bag": -0.0072697486729145595, "b:lgb_diglead_bag": 0.003752570618535255, "b:lgb_diglead_bag5": -0.01387844815389441, "b:lgb_diglead_l2": -0.008722162257095078, "b:lgb_diglead_l3": 0.01340449460759475, "b:lgb_diglead_l3_bag": -0.009648198615136732, "b:lgb_diglead_l4": 0.0023017753746317295, "b:lgb_dk_l4": 0.00404320185129622, "b:lgb_raw_l31": 0.0004917472972874557, "b:lgb_raw_l4": -0.00011914588071201624, "b:lgb_raw_l4_cnt": -0.015582993656188236, "b:lgb_rawdig_l4": 0.001729542007007652, "b:lgb_te_l4": 0.008969005313344148, "b:lgb_te_l4_s100": 3.676618403346857e-05, "b:lgb_te_l4_s300": 0.0022842969787489017, "b:lgb_te_l4_s5": 0.002931297790132998, "b:realmlp_allcat": -0.0026909874980677114, "b:realmlp_cnt": 0.0021648947039792175, "b:realmlp_dig": -0.009016404783169642, "b:realmlp_dig_s100": 0.005402396661241892, "b:realmlp_dig_s2": 0.0022649717220251738, "b:realmlp_rawdig": -0.007755994202623359, "b:realmlp_te": 0.0032482793804713803, "b:realmlp_te_cnt": 0.000847393585301589, "b:realmlp_te_s2": 0.0004964361210193442, "b:realmlp_v1": 0.0064450941750321756, "b:xgb_dig_d2": -0.001615855509527677, "b:xgb_raw_d3": 0.00433269057718474, "b:xgb_te_d2": -0.0033627181678075714}, "careful": {"s:megayak_5f_lm": -0.08450474886996073, "s:masaya_gpt2": -0.04902561986411483, "s:lm7_5fold": -0.016891411336523863, "s:gpt2_seed2026": 0.014547738041531408, "s:tabpfn_slimypunk_tp1": -0.0123457470481866, "s:sadam_tabpfn_100k_tp3": -0.015910384586476225, "s:yekenot_realmlp": -0.003253325303861341, "s:alex_glm_gen": 0.002612752988313417, "s:daoviet_gpdvae": 0.04685747302679586, "s:gbn_bag5_w512": 0.0043515522989154235, "s:sameer_glm10f": 0.015478953160303576, "s:sadam_v6_gb_xgb": -0.028037375140335462, "s:sadam_v6_gb_lgb": 0.019533386719025193, "s:sadam_v4_cat": 0.00589878484038633, "s:gb_bag5": 0.02217844040888381, "s:megayak_shallow_tree": 0.10122717023890326, "wide": 0.08081797187087163, "b:ebm_all_bag": -0.03128848896508937, "b:ebm_dig": 0.002420259254897357, "b:ebm_dig_bag": -0.006646622819934664, "b:ebm_dig_r7": 0.010794306141448952, "b:ebm_diglead_bag": -0.0117042245417344, "b:ebm_diglead_i25": -0.008918615668229131, "b:ebm_diglead_r11": 0.017495433781515475, "b:ebm_diglead_r13": 0.019436209350096583, "b:ebm_i25_bag": 0.008522596080523918, "b:ebm_i25_bag4": 0.016476990933893144, "b:ebm_lead4_bag": -0.01130787311582097, "b:gb_bag10": -0.013583645440115318, "b:gb_bag5": 0.016684891246275074, "b:gb_l3": -0.019083828087590973, "b:gb_ohk": -0.006177878317621568, "b:gbc_bag3": -0.007269120619879105, "b:gbe_bag3": 0.004681311487245302, "b:gbn_bag5_w128": -0.005040768827170892, "b:gbn_bag5_w512": 0.023637688632069328, "b:gbr_bag3": -0.00024830818253431273, "b:gbx_bag3": -0.006400911249302103, "b:gbx_bag5": 0.0036937689629500865, "b:glm_dig": -3.7306124542630744e-05, "b:glm_dl": -0.007156160004066149, "b:glm_dl_bag": 0.008441014574504154, "b:glm_dl_s50": -0.01334934207158471, "b:glm_dl_s50_bag": 0.001844211766019183, "b:glm_ohk": -0.00562390612264448, "b:glm_ohk2": 0.008907953791620765, "b:glm_ohk2_bag": -0.015300133811741116, "b:glm_ohk_c01": -0.005892186175925242, "b:glm_ohkl": -0.00010117004854648993, "b:glm_ohkl_bag": 0.008342574170709143, "b:glm_sb": -0.003455433101783884, "b:glm_te": -0.0018554925449663707, "b:lgb_dig_l4": 0.012788668778136636, "b:lgb_dig_l7": 0.006133544143597403, "b:lgb_dig_s100": -0.01581131868317444, "b:lgb_dig_s100_bag": 0.005976683240784171, "b:lgb_dig_s100_bag5": 0.003272572328457066, "b:lgb_dig_s100_l3": 0.011358913543846424, "b:lgb_dig_s100_l3_bag": -0.007671455946372908, "b:lgb_diglead_bag": 0.0026446221720072436, "b:lgb_diglead_bag5": -0.014415491763205732, "b:lgb_diglead_l2": -0.009879458752685616, "b:lgb_diglead_l3": 0.015081200121801388, "b:lgb_diglead_l3_bag": -0.01231464394864841, "b:lgb_diglead_l4": 0.0027104104670708385, "b:lgb_dk_l4": 0.002097897271309318, "b:lgb_raw_l31": 0.0010331280960220208, "b:lgb_raw_l4": -0.0013378069288362123, "b:lgb_raw_l4_cnt": -0.013507197348019013, "b:lgb_rawdig_l4": 0.0028805514532224763, "b:lgb_te_l4": 0.008125965905514013, "b:lgb_te_l4_s100": 0.0030857860977705903, "b:lgb_te_l4_s300": 0.0022465238467168505, "b:lgb_te_l4_s5": -0.0007517378836866291, "b:realmlp_allcat": -0.00013519997284003595, "b:realmlp_cnt": 0.0031564480735426013, "b:realmlp_dig": -0.009290279722601536, "b:realmlp_dig_s100": 0.005582584293801132, "b:realmlp_dig_s2": 0.0018383277487227473, "b:realmlp_rawdig": -0.009112515110809292, "b:realmlp_te": 0.003079185734258433, "b:realmlp_te_cnt": -0.00037244665110067394, "b:realmlp_te_s2": -0.0021716481344977163, "b:realmlp_v1": 0.005001493601478614, "b:xgb_dig_d2": -0.004210955412618814, "b:xgb_raw_d3": 0.004254033686957146, "b:xgb_te_d2": -0.0023974804930285173}}')                                  # weights of four public mixes; s: bundle model, b: OOF-bank model, wide: goodpjw2008's wide LM stack
_bank29 = _find('manifest.csv', 'oof-bank'); _bank29 = Path(_bank29).parent if _bank29 else None
_p29 = {k: _find(f, o) for k, (f, o) in {'wide_oof': ('wide_lm_stack_nested.npy', 'fraud-series'), 'wide_te': ('wide_lm_stack_test.npy', 'fraud-series'),
                                        'lm7': ('lm7_test_5fold.npy', 'compact-lm'), 'lm': ('megalm_test_sha3168.npy', 'compact-lm')}.items()}
_need29 = [k[2:] for k in R29['bolder'] if k.startswith('b:')]
_zf29 = set(getattr(Zb, 'files', [])) if 'Zb' in dir() else set()
_s29 = [k[2:] for k in R29['bolder'] if k.startswith('s:')]
_ok29 = (AO28 is not None and _bank29 is not None and all(_p29.values())
         and all((_bank29 / f'oof_{n}.npy').exists() and (_bank29 / f'test_{n}.npy').exists() for n in _need29)
         and all(f'oof_{n}' in _zf29 and f'test_{n}' in _zf29 for n in _s29))
if not _ok29: print('v29 skipped: inputs not mounted', {k: v is not None for k, v in _p29.items()}, 'bank', _bank29 is not None)
if _ok29:
    def _u29(v): return (rankdata(v) - 0.5) / len(v)
    def _L29(v): return logit(_u29(v))
    WO29, WT29 = _L29(np.load(_p29['wide_oof'])), _L29(np.load(_p29['wide_te']))
    _one = lambda A: np.column_stack([A, np.ones(len(A))])
    b29 = np.linalg.lstsq(_one(np.column_stack([XT28, WT29])), AT28, rcond=None)[0]       # goodpjw2008's replay: the 0.95370 file on 16 models + their wide stack, test rows
    REF29 = _one(np.column_stack([XO28, WO29])) @ b29 + 0.06 * (XO28[:, N28.index('lm7_5fold')] - XO28[:, N28.index('megayak_5f_lm')])
    U029 = _u29(AT28 + 0.06 * (_L29(np.load(_p29['lm7'])) - _L29(np.load(_p29['lm']))))
    SO29 = {r: np.zeros(len(X)) for r in R29}; ST29 = {r: 0 for r in R29}
    for k in R29['bolder']:                                # areebbb's order: the test-side sum must add up in the same order to give the same bytes
        if k.startswith('s:'): o_, t_ = _L29(Zb['oof_' + k[2:]]), _L29(Zb['test_' + k[2:]])
        elif k.startswith('b:'): o_, t_ = _L29(np.load(_bank29 / f'oof_{k[2:]}.npy')), _L29(np.load(_bank29 / f'test_{k[2:]}.npy'))
        else: o_, t_ = WO29, WT29
        for r in R29: SO29[r] = SO29[r] + R29[r][k] * o_; ST29[r] = ST29[r] + R29[r][k] * t_
    sw29 = {r: sum(R29[r].values()) for r in R29}
    b388 = (1 - sw29['bolder']) * logit(U029) + ST29['bolder']
    _f29 = OUT / 'submission_b388_rebuilt.csv'; pd.DataFrame({'id': test['id'], TARGET: _u29(b388)}).to_csv(_f29, index=False)
    h29 = hashlib.sha256(open(_f29, 'rb').read()).hexdigest()
    print('the 0.95388 file rebuilt:', 'SHA-256 matches areebbb\'s published b0979694...' if h29 == 'b097969454128762dcd4929116d08fc9f00b3a6ebfe65fcbf7979ffaea3d729b' else f'SHA-256 differs ({h29[:16]}); the ranks may still agree')
    print(f'rank correlation of the rebuild with the 0.95370 file: {np.corrcoef(rankdata(b388), rankdata(AT28))[0, 1]:.6f}')
    _f29.unlink()                                          # not ours to submit: only the check is kept
    F29 = F28; a29 = a28
    def _mix29(w, s_, base): return (1 - w) * base + s_
    print(f'our wide stacker, held-out split 7: {a29:.6f} | the reference (16 models + their wide stack + LM step), replayed: {_fauc(yv28, REF29):.6f}')
    rows29 = []
    for r in ['careful', 'balanced', 'bold', 'bolder']:
        o_ = _mix29(sw29[r], SO29[r], REF29); o2_ = _mix29(sw29[r], SO29[r], F29)
        rows29.append([r, _fauc(yv28, o_), (_fauc(yv28, o_) - _fauc(yv28, REF29)) * 1e5, _ups28(o_, REF29), (_fauc(yv28, o_) - a29) * 1e5, (_fauc(yv28, o2_) - a29) * 1e5, _ups28(o2_, F29)])
    M29 = pd.DataFrame(rows29, columns=['mix', 'AUC (in-sample)', 'pts vs reference', 'folds up', 'pts vs our held-out stack', 'the move on our stack, pts', 'folds up on our stack']).set_index('mix')
    print('the four public mixes on labelled rows (1 point = 0.00001):'); print(M29.round({'AUC (in-sample)': 6, 'pts vs reference': 2, 'pts vs our held-out stack': 2, 'the move on our stack, pts': 2}).to_string())
    ts29 = [0.0, 0.5, 1.0, 1.5, 2.0]; S29 = []
    for t in ts29:                                         # steps along bold - balanced: linear in the weights, so linear in the sums
        w_ = sw29['bold'] + t * (sw29['bold'] - sw29['balanced']); s_ = SO29['bold'] + t * (SO29['bold'] - SO29['balanced'])
        o_, o2_ = _mix29(w_, s_, REF29), _mix29(w_, s_, F29)
        S29.append([t, (_fauc(yv28, o_) - _fauc(yv28, REF29)) * 1e5, _ups28(o_, REF29), (_fauc(yv28, o2_) - a29) * 1e5, _ups28(o2_, F29)])
    S29 = pd.DataFrame(S29, columns=['t', 'on the reference, pts', 'folds up', 'on our stack, pts', 'folds up ']).set_index('t')
    print('steps bold + t (bold - balanced):'); print(S29.round(2).to_string())
    bo29 = _mix29(sw29['bolder'], SO29['bolder'], REF29); rng29 = np.random.default_rng(29)
    for n_ in (n_pub28, n_prv28):
        d_ = []
        for _ in range(100):
            i_ = rng29.choice(len(yv28), n_, replace=False); d_.append((_fauc(yv28[i_], bo29[i_]) - _fauc(yv28[i_], F29[i_])) * 1e5)
        print(f'gap (the 0.95388 mix, in-sample - our held-out stack) on {n_} random rows: {np.mean(d_):+.1f} +- {np.std(d_):.1f} points; the file ahead in {np.mean(np.array(d_) > 0):.0%} of 100 draws')
    MX29 = pd.DataFrame([[w, _fauc(yv28, (1 - w) * bo29 + w * F29), (_fauc(yv28, (1 - w) * bo29 + w * F29) - _fauc(yv28, bo29)) * 1e5, _ups28((1 - w) * bo29 + w * F29, bo29)] for w in (0.0, 0.1, 0.25, 0.5, 0.75, 1.0)],
                        columns=['share of our wide stack', 'AUC on labelled rows', 'pts vs the file', 'folds up']).set_index('share of our wide stack')
    print('the 0.95388 mix with our held-out wide stack:'); print(MX29.round({'AUC on labelled rows': 6, 'pts vs the file': 1}).to_string())
    _bw = pd.Series(R29['bold']); print('largest moves of the bold mix:', _bw.reindex(_bw.abs().sort_values(ascending=False).index).head(8).round(3).to_dict())
    T['the 0.95388 file (rebuilt)'] = b388
    for n_, v_ in [('submission_b388_wide10.csv', 0.9 * b388 + 0.1 * _lr(t_w)), ('submission_b388_wide50.csv', 0.5 * b388 + 0.5 * _lr(t_w)),
                   ('submission_wide_bold.csv', _mix29(sw29['bold'], ST29['bold'], _lr(t_w)))]:
        T[n_] = v_; pd.DataFrame({'id': test['id'], TARGET: rankdata(v_) / len(v_)}).to_csv(OUT / n_, index=False); print('wrote', n_, f'| distance from our wide stack {_dist28(v_, t_w):.1f}e-4, from the 0.95388 file {_dist28(v_, b388):.1f}e-4')
    fig, (b1, b2) = plt.subplots(1, 2, figsize=(14, 4.6))
    lab_ = ['0.95370 file\n(16 models)', 'reference\n(+ wide, LM step)'] + list(M29.index); val_ = [_fauc(yv28, AO28), _fauc(yv28, REF29)] + list(M29['AUC (in-sample)'])
    pub_ = [0.95370, 0.95370, 0.95376, 0.95380, 0.95386, 0.95388]
    b1.bar(range(len(val_)), val_, color=['#c98b2b'] * 2 + ['#d9a864'] * 4); b1.axhline(a29, color='#4a6fa5', lw=2, label=f'our wide stack, held-out {a29:.5f} (public 0.95260)')
    for i_, (v_, p_) in enumerate(zip(val_, pub_)): b1.annotate(f'public {p_:.5f}', (i_, v_), textcoords='offset points', xytext=(0, 4), ha='center', fontsize=7)
    b1.set_xticks(range(len(val_))); b1.set_xticklabels(lab_, fontsize=8); b1.set_ylim(min(val_) - 0.0002, a29 + 0.0002); b1.set_ylabel('AUC on labelled rows (mixes: in-sample)')
    b1.set_title('the public ladder on labelled rows: each rung closer to a label-fitted stack', fontsize=10); b1.legend(fontsize=8, loc='upper left')
    b2.plot(S29.index, S29['on the reference, pts'], 'o-', color='#c98b2b', label='on the replayed 0.95370 file (in-sample)'); b2.plot(S29.index, S29['on our stack, pts'], 'o-', color='#4a6fa5', label='the same move on our held-out stack')
    b2.axhline(0, color='#999', lw=1); bx = b2.twinx(); bx.plot([0.0, 0.5, 1.0, 1.5, 2.0], [0.95386, 0.95388, 0.95387, 0.95387, 0.95385], 's--', color='#b0413e', label='public LB (areebbb\'s probes)'); bx.set_ylabel('public LB', color='#b0413e'); bx.ticklabel_format(axis='y', useOffset=False, style='plain')
    b2.set_xlabel('t in bold + t (bold - balanced)'); b2.set_ylabel('change in AUC, points (0.00001)'); b2.set_title('one direction, three referees, three best step sizes', fontsize=10)
    h1, l1 = b2.get_legend_handles_labels(); h2, l2 = bx.get_legend_handles_labels(); b2.legend(h1 + h2, l1 + l2, fontsize=8, loc='lower left'); plt.tight_layout(); plt.show()


### What the board said about v29 (submitted 2026-11-15)

| file | what | public LB | change on labelled rows |
|---|---|---|---|
| `submission_b388_wide10.csv` | 90% areebbb's 0.95388 file + 10% our wide stack | **0.95386** (the title) | +5.1 points over the file (in-sample replay, 5 of 5 folds) |
| `submission_b388_wide50.csv` | 50% file + 50% our wide stack | 0.95359 | +17.1 points over the file (5 of 5) |
| `submission_wide_bold.csv` | our wide stack + goodpjw2008's bold 87-member move | 0.95278 | -1.5 points against our stack (1 of 5) |
| `submission.csv` | our wide stack (167 members), as v23 | 0.95260 | 0 |

- **The board and the labelled rows disagree on all three files.** Our stack in the 0.95388 file: -2 and -29 on the board, +5 and +17 on labelled rows. The bold move on our stack: +18 on the board, -1.5 on labelled rows.
- v28 showed the same trade with the 0.95370 file (-3 and -11 on the board, +13 and +31 on labelled rows). Five files in two versions, five times the opposite sign.
- The +18 for the bold move is the size v29 measured for public-board noise at this distance (± 27 points on public-sized subsets for a file 74e-4 away). It may be noise; it may be the forecast model doing what it was fitted to do. The next section measures that question directly.

### v30: the 0.95417 file, step by step, on the board and on labelled rows

**The question.** On 2026-11-15 the public best is **0.95417**, [goodpjw2008's LM Ladder | Forecast the Board](https://www.kaggle.com/code/goodpjw2008/s6e11-lm-ladder-forecast-the-board-lb-0-95417), 29 points above the 0.95388 file it starts from.
The author states plainly that it is tuned on the public rows: "What CV supports stops at about 0.95388. Everything above that in this notebook is bought from the public rows."
That is rare and useful: a ten-step public climb with every weight published. Here each step is scored on labelled rows, so a reader can see what a public-board climb looks like from the other 80%.

**How the steps are made (goodpjw2008's method).**
1. Start from the 0.95388 file (v29).
2. A *public model* forecasts the public score of a file before it is submitted. It is a ridge model fitted on scored files: each scored file is one linear equation about how the public rows' labels line up with the file's ranks. 66 files by the last step (theirs and public files with stated scores).
3. Each step moves towards the same 87 members, `score = (1 - sum w) logit_rank(previous file) + sum w_k logit_rank(member_k)`, with `w` chosen to maximise `CV change + mu x forecast`. Most steps put the forecast in the lead.
4. Submit, refit the public model with the new score, take the next step from the best file so far.

The weights of all ten steps are in the notebook (`STEP_JSON`). This cell rebuilds them. Checked offline against goodpjw2008's own output files: step 7 (their `submission.csv`) and the CV-led step (`submission_cv.csv`) agree to 1e-4 in rank fraction (not byte for byte: the sums are added in another order).

**What we measured** (offline before this version ran, `scratch/banks/v30_1115/sim30.py`; the cell below prints its own numbers; 1 point = 0.00001; "in-sample": the weights saw all five folds):

| step | starts from | public change | change on labelled rows (in-sample) | folds up | goodpjw2008's nested CV change | vs our held-out stack |
|---|---|---|---|---|---|---|
| the 0.95388 file | | | | | | -11.1 |
| joint | 0.95388 | 0 | +7.7 | 4 | +3.3 | -3.4 |
| CV-led | 0.95388 | -1 | **+12.7** | 5 | +7.3 | **+1.6** |
| step 1 | 0.95388 | +2 | +2.3 | 3 | -1.1 | -8.8 |
| step 2, half | step 1 | +3 | -6.6 | 1 | -8.6 | -15.4 |
| step 2 | step 1 | +6 | -14.4 | 1 | -18.4 | -23.2 |
| step 3 | step 2 | +8 | -26.6 | 0 | -29.2 | -49.8 |
| step 4 | step 3 | +4 | -15.9 | 0 | -18.6 | -65.7 |
| step 5 (not kept) | step 4 | -1 | -32.6 | 0 | -33.3 | -98.3 |
| step 6 | step 4 | +4 | +26.9 | 5 | +16.0 | -38.8 |
| **step 7, the 0.95417 file** | step 6 | **+5** | **-13.9** | 0 | -19.0 | **-52.7** |

- **The climb went up 29 points on the board and down 42 on labelled rows** (0.95388 -> 0.95417 public; the replay 0.95246 -> 0.95204). Our in-sample replay and goodpjw2008's nested numbers agree in sign on all ten steps.
- **Can a public-sized subset of labelled rows show +29 by chance?** On 100 random subsets of 59,996 labelled rows, step 7 minus the 0.95388 file is -44 ± 22 points; none of the 100 reached +29. The +29 is not luck in the sense of a random 20%. It is the public model fitting *these* 60,000 rows, which is what it was built to do.
- **Across the ten steps, public change and labelled-row change correlate at -0.28** (with goodpjw2008's nested numbers -0.33). Where the forecast leads, the board pays and the labelled rows lose.
- **The CV-led step is the one file of the ten that beats our held-out stack** (+1.6 points, 3 of 5 folds, in-sample). On the board it scored 0.95387, one point *below* its start. goodpjw2008 names it as their pick for the private rows.
- **What this means for picking.** Kaggle keeps the better of two final picks. kagankoral's [Final-Pick Check](https://www.kaggle.com/code/kagankoral/s6e11-final-pick-check-cv-vs-the-public-top) gives the same advice from a different measurement: one file for each referee.

**Board files from this version:** `submission_s417_wide10.csv` (90% the rebuilt 0.95417 file + 10% our wide stack; on labelled rows +12.2 points over the file, 5 of 5 folds) and `submission_cvled_wide10.csv` (90% the CV-led file + 10% our wide stack; +3.0, 5 of 5).
v28 and v29 found 10% of our stack costs 2 to 3 public points. If it costs the same here, the first file is a 0.9541 file with a better labelled-row score than the 0.95417 file. The results go in the next version.

**Credit.** goodpjw2008 (the public model, the ten steps, the honest limits section, the OOF bank and the wide LM stack), areebbb (the 0.95388 file), Aman Atar (the 0.95370 file and its bundle), kagankoral (the final-pick check), asteriospanoras (the noise of a public gap), and the authors of the 87 members (listed in the source notebooks). Please upvote them.


In [ ]:
# v30: goodpjw2008's ten steps from the 0.95388 file rebuilt (step 7 = the public 0.95417 file), each scored on labelled rows; the board change against the labelled-row change
R30 = json.loads('{"joint": {"s:megayak_5f_lm": -0.01717077159073525, "s:masaya_gpt2": 0.0019183243447994148, "s:lm7_5fold": 0.045423022260665136, "s:gpt2_seed2026": -0.00042944375296627196, "s:tabpfn_slimypunk_tp1": -0.00899495055721427, "s:sadam_tabpfn_100k_tp3": -0.029270717860439565, "s:yekenot_realmlp": -0.00234960739485184, "s:alex_glm_gen": -0.02024058123014512, "s:daoviet_gpdvae": 0.03484720930561664, "s:gbn_bag5_w512": 0.043838711156557714, "s:sameer_glm10f": 0.06984336964612282, "s:sadam_v6_gb_xgb": -0.009431902430786612, "s:sadam_v6_gb_lgb": 0.029233124147598495, "s:sadam_v4_cat": 0.01721852315397201, "s:gb_bag5": -0.017019375679840406, "s:megayak_shallow_tree": 0.025418142677893215, "wide": 0.058343674987245644, "b:ebm_all_bag": 0.04569362232028508, "b:ebm_dig": -0.050631120263571604, "b:ebm_dig_bag": 0.03236181449748603, "b:ebm_dig_r7": -0.016790661098617175, "b:ebm_diglead_bag": -0.02032888065903343, "b:ebm_diglead_i25": 0.016077523937003083, "b:ebm_diglead_r11": -0.015234267117895712, "b:ebm_diglead_r13": -0.002050297510730736, "b:ebm_i25_bag": -0.02278394288002672, "b:ebm_i25_bag4": 0.011423726826693454, "b:ebm_lead4_bag": 0.030716217831992645, "b:gb_bag10": 0.0006193248766763178, "b:gb_bag5": 0.01586673751093695, "b:gb_l3": -0.010729425968314792, "b:gb_ohk": -0.024322587520270455, "b:gbc_bag3": 0.010519392802311005, "b:gbe_bag3": 0.008359864921988952, "b:gbn_bag5_w128": 0.002248917510635357, "b:gbn_bag5_w512": 0.018136959102802233, "b:gbr_bag3": -0.008365004797769256, "b:gbx_bag3": 0.006589007276661395, "b:gbx_bag5": -0.03355885475043985, "b:glm_dig": 0.0012813536285928683, "b:glm_dl": -0.001273981615760972, "b:glm_dl_bag": -0.02872643207621259, "b:glm_dl_s50": -0.008710208055072402, "b:glm_dl_s50_bag": -0.021278755306043704, "b:glm_ohk": -0.02123527953417163, "b:glm_ohk2": -0.026270260484603916, "b:glm_ohk2_bag": -0.017602826594117285, "b:glm_ohk_c01": -0.013770304192067154, "b:glm_ohkl": 0.013870805201599829, "b:glm_ohkl_bag": 0.022583324982946185, "b:glm_sb": -0.008448371582005464, "b:glm_te": 0.0016929278315118475, "b:lgb_dig_l4": 0.020239869250518273, "b:lgb_dig_l7": 0.009344260941331638, "b:lgb_dig_s100": -0.014026745793121813, "b:lgb_dig_s100_bag": -0.03897583634328856, "b:lgb_dig_s100_bag5": 0.011268527889825249, "b:lgb_dig_s100_l3": 0.004982508768429224, "b:lgb_dig_s100_l3_bag": 0.019769181140729782, "b:lgb_diglead_bag": -0.04352076765430828, "b:lgb_diglead_bag5": -0.0005833043594665035, "b:lgb_diglead_l2": -0.04413222287065789, "b:lgb_diglead_l3": 0.033124504055349224, "b:lgb_diglead_l3_bag": 0.009405122110509721, "b:lgb_diglead_l4": -0.0018646983927889356, "b:lgb_dk_l4": 0.0005093211910773871, "b:lgb_raw_l31": 0.001729502053405105, "b:lgb_raw_l4": 0.006285249236442414, "b:lgb_raw_l4_cnt": -0.038610384561969635, "b:lgb_rawdig_l4": 0.01664169846460153, "b:lgb_te_l4": 0.024628528641973735, "b:lgb_te_l4_s100": 0.023996989997212066, "b:lgb_te_l4_s300": -0.00853403591839359, "b:lgb_te_l4_s5": 0.0030603366645029534, "b:realmlp_allcat": -0.009471160685104547, "b:realmlp_cnt": 0.0030608219751133265, "b:realmlp_dig": -0.011108471402391146, "b:realmlp_dig_s100": 0.0003058860814721104, "b:realmlp_dig_s2": 0.015435773625366142, "b:realmlp_rawdig": -0.024481211697770403, "b:realmlp_te": 0.011039507507020605, "b:realmlp_te_cnt": -0.005583261278170029, "b:realmlp_te_s2": -0.006940507772879366, "b:realmlp_v1": 0.012630090172043857, "b:xgb_dig_d2": -0.02322669118450206, "b:xgb_raw_d3": 0.0014099646053971717, "b:xgb_te_d2": -0.0011439174998451335}, "cv_led": {"s:megayak_5f_lm": -0.0504101115868334, "s:masaya_gpt2": 0.01892536826920305, "s:lm7_5fold": 0.03268460975553137, "s:gpt2_seed2026": 0.00019156335507359648, "s:tabpfn_slimypunk_tp1": -0.012642649769990903, "s:sadam_tabpfn_100k_tp3": -0.03039229764870732, "s:yekenot_realmlp": -0.001156919333157887, "s:alex_glm_gen": -0.010049686607455182, "s:daoviet_gpdvae": 0.026524288367069906, "s:gbn_bag5_w512": 0.03197270577142789, "s:sameer_glm10f": 0.07539886111092914, "s:sadam_v6_gb_xgb": -0.03876423955389656, "s:sadam_v6_gb_lgb": 0.050619303134153054, "s:sadam_v4_cat": 0.024899735169078147, "s:gb_bag5": -0.012525566608465507, "s:megayak_shallow_tree": 0.08998587818302896, "wide": 0.05568936953093647, "b:ebm_all_bag": 0.01241314649800849, "b:ebm_dig": -0.033403049606166156, "b:ebm_dig_bag": 0.009782407957898523, "b:ebm_dig_r7": 0.0031988540661077067, "b:ebm_diglead_bag": -0.022134518874358366, "b:ebm_diglead_i25": 0.007513447059301521, "b:ebm_diglead_r11": 0.0004938077778375392, "b:ebm_diglead_r13": 0.014156212789682508, "b:ebm_i25_bag": -0.0147292301078183, "b:ebm_i25_bag4": 0.02385867724779454, "b:ebm_lead4_bag": 0.006688192218254173, "b:gb_bag10": -0.0015321268707586982, "b:gb_bag5": 0.0144427492994866, "b:gb_l3": -0.026744709811242964, "b:gb_ohk": -0.0250889728309019, "b:gbc_bag3": -0.0001499417896917808, "b:gbe_bag3": 0.007162876625302505, "b:gbn_bag5_w128": -0.008308739560439522, "b:gbn_bag5_w512": 0.01723221260596312, "b:gbr_bag3": -0.006241102802695989, "b:gbx_bag3": -0.006240817526956558, "b:gbx_bag5": -0.015449787795720182, "b:glm_dig": 0.0016094490785222377, "b:glm_dl": 0.001269407609764088, "b:glm_dl_bag": -0.019307803416807087, "b:glm_dl_s50": -0.016638174597513808, "b:glm_dl_s50_bag": -0.02308316908617957, "b:glm_ohk": -0.014920227472817757, "b:glm_ohk2": -0.010264756653736738, "b:glm_ohk2_bag": -0.022667189412379152, "b:glm_ohk_c01": -0.014560819249265932, "b:glm_ohkl": 0.012022636416038466, "b:glm_ohkl_bag": 0.010820057263254173, "b:glm_sb": -0.008585709230074623, "b:glm_te": 0.00014461870145916856, "b:lgb_dig_l4": 0.02775568242888243, "b:lgb_dig_l7": 0.01231196265590098, "b:lgb_dig_s100": -0.026963457242898014, "b:lgb_dig_s100_bag": -0.022771393714400665, "b:lgb_dig_s100_bag5": 0.01501939437317305, "b:lgb_dig_s100_l3": 0.020784028527905406, "b:lgb_dig_s100_l3_bag": 0.003401885062408753, "b:lgb_diglead_bag": -0.022936997298612535, "b:lgb_diglead_bag5": -0.026908092680123324, "b:lgb_diglead_l2": -0.0421340354076887, "b:lgb_diglead_l3": 0.04054503152925578, "b:lgb_diglead_l3_bag": -0.00040144349699201566, "b:lgb_diglead_l4": -0.00020162492642476296, "b:lgb_dk_l4": 0.0019528312307037553, "b:lgb_raw_l31": 0.0006356799380731614, "b:lgb_raw_l4": 0.006217819755726177, "b:lgb_raw_l4_cnt": -0.04126303032264386, "b:lgb_rawdig_l4": 0.01575670670651084, "b:lgb_te_l4": 0.018941213542591304, "b:lgb_te_l4_s100": 0.017060420694787578, "b:lgb_te_l4_s300": 0.0015753759554598585, "b:lgb_te_l4_s5": 0.0013086638671635535, "b:realmlp_allcat": -0.006125101859943052, "b:realmlp_cnt": 0.00544837093817107, "b:realmlp_dig": -0.014207768492579254, "b:realmlp_dig_s100": 0.005558545056911285, "b:realmlp_dig_s2": 0.012895054479795923, "b:realmlp_rawdig": -0.024402566794336375, "b:realmlp_te": 0.009895745291643016, "b:realmlp_te_cnt": -0.005084928725762969, "b:realmlp_te_s2": -0.006487850666131134, "b:realmlp_v1": 0.012200835041492056, "b:xgb_dig_d2": -0.03021531859263534, "b:xgb_raw_d3": 0.006410944962609165, "b:xgb_te_d2": -0.005833150956332071}, "step1": {"s:megayak_5f_lm": -0.003351112978983948, "s:masaya_gpt2": -0.004331262757969586, "s:lm7_5fold": 0.033366619003782044, "s:gpt2_seed2026": -0.007827509209426856, "s:tabpfn_slimypunk_tp1": -0.009713901949474838, "s:sadam_tabpfn_100k_tp3": -0.02258942534334505, "s:yekenot_realmlp": -0.004526829340289488, "s:alex_glm_gen": -0.00888266753715394, "s:daoviet_gpdvae": 0.029965005683580372, "s:gbn_bag5_w512": 0.034301067520051236, "s:sameer_glm10f": 0.032646833126727844, "s:sadam_v6_gb_xgb": -0.018135893519251264, "s:sadam_v6_gb_lgb": 0.016938816758909212, "s:sadam_v4_cat": 0.010122157702368371, "s:gb_bag5": -0.03439525156711189, "s:megayak_shallow_tree": 0.016529211843412543, "wide": 0.039253033212382056, "b:ebm_all_bag": 0.07523770624398651, "b:ebm_dig": -0.054123847865869386, "b:ebm_dig_bag": 0.042590023002053626, "b:ebm_dig_r7": -0.030431224464872567, "b:ebm_diglead_bag": -0.007475256474622878, "b:ebm_diglead_i25": 0.012130204546613841, "b:ebm_diglead_r11": -0.029545115889253553, "b:ebm_diglead_r13": -0.025314977657280278, "b:ebm_i25_bag": -0.034251702364764935, "b:ebm_i25_bag4": 0.0021819101691288286, "b:ebm_lead4_bag": 0.05402388421646351, "b:gb_bag10": 0.02127855874172903, "b:gb_bag5": 0.014346413654670603, "b:gb_l3": -0.013769934129875412, "b:gb_ohk": -0.01320976918577355, "b:gbc_bag3": 0.005252926825715231, "b:gbe_bag3": 0.0056608738989543635, "b:gbn_bag5_w128": -0.008066646084482498, "b:gbn_bag5_w512": 0.0027761714797112673, "b:gbr_bag3": -0.0015671193553017644, "b:gbx_bag3": 0.013676703301752498, "b:gbx_bag5": -0.024849339260214022, "b:glm_dig": 0.003075598457342358, "b:glm_dl": 0.010916567005561281, "b:glm_dl_bag": -0.02140273076626105, "b:glm_dl_s50": -0.0048270702443609145, "b:glm_dl_s50_bag": -0.028965516463380712, "b:glm_ohk": -0.009823478201748637, "b:glm_ohk2": -0.0318522897319079, "b:glm_ohk2_bag": 0.008008910874263413, "b:glm_ohk_c01": -0.001537026826110699, "b:glm_ohkl": -0.0007177995052908708, "b:glm_ohkl_bag": 0.02035140308004446, "b:glm_sb": -0.003981381639641186, "b:glm_te": 0.0003080122272580523, "b:lgb_dig_l4": 0.010603573387195964, "b:lgb_dig_l7": 0.004572758181607049, "b:lgb_dig_s100": -0.008401535545529524, "b:lgb_dig_s100_bag": -0.024937178041431557, "b:lgb_dig_s100_bag5": 0.009843322515543112, "b:lgb_dig_s100_l3": 0.005104687606392921, "b:lgb_dig_s100_l3_bag": 0.013836519222034662, "b:lgb_diglead_bag": -0.02825928718723698, "b:lgb_diglead_bag5": -0.00015918700995309258, "b:lgb_diglead_l2": -0.023477703221799207, "b:lgb_diglead_l3": 0.013687939454287347, "b:lgb_diglead_l3_bag": 0.01089390576912047, "b:lgb_diglead_l4": 0.0027356122650579223, "b:lgb_dk_l4": 0.0031170273617469727, "b:lgb_raw_l31": -0.0002778639224024611, "b:lgb_raw_l4": 0.005076659619331169, "b:lgb_raw_l4_cnt": -0.026000953337936953, "b:lgb_rawdig_l4": 0.008909983765388369, "b:lgb_te_l4": 0.014595251015497515, "b:lgb_te_l4_s100": 0.009202947491696535, "b:lgb_te_l4_s300": -0.0036164770792529123, "b:lgb_te_l4_s5": 0.004755908523758278, "b:realmlp_allcat": -0.009180037960367643, "b:realmlp_cnt": 0.002008152697517426, "b:realmlp_dig": -0.006650850028567239, "b:realmlp_dig_s100": 0.0011095904957665577, "b:realmlp_dig_s2": 0.011252248899267365, "b:realmlp_rawdig": -0.01135452534507476, "b:realmlp_te": 0.006400827919491363, "b:realmlp_te_cnt": -0.0023289922376294714, "b:realmlp_te_s2": 0.0004650667245475183, "b:realmlp_v1": 0.007421311984852427, "b:xgb_dig_d2": -0.014921295260094976, "b:xgb_raw_d3": 0.0029908808169654922, "b:xgb_te_d2": -0.004556637612790932}, "step2_half": {"s:megayak_5f_lm": 0.013560479235470527, "s:masaya_gpt2": -0.013415532419070966, "s:lm7_5fold": 0.016924808967985386, "s:gpt2_seed2026": -0.005846561584592116, "s:tabpfn_slimypunk_tp1": -0.00716915745504147, "s:sadam_tabpfn_100k_tp3": -0.012477860876310934, "s:yekenot_realmlp": -0.005847025911995225, "s:alex_glm_gen": -0.0035667420089754978, "s:daoviet_gpdvae": 0.010432227162500262, "s:gbn_bag5_w512": 0.03421288006649682, "s:sameer_glm10f": -0.01753859206217354, "s:sadam_v6_gb_xgb": -0.006008166073410063, "s:sadam_v6_gb_lgb": -0.011189574364595722, "s:sadam_v4_cat": -0.008215256938098263, "s:gb_bag5": -0.013234620290331862, "s:megayak_shallow_tree": -0.05329387696475422, "wide": 0.020055198517721345, "b:ebm_all_bag": 0.08032077305331266, "b:ebm_dig": -0.04821548653806729, "b:ebm_dig_bag": 0.0663608437089962, "b:ebm_dig_r7": -0.04763985976078816, "b:ebm_diglead_bag": 0.0012470873772310392, "b:ebm_diglead_i25": 0.020868319380952464, "b:ebm_diglead_r11": -0.018787199868178617, "b:ebm_diglead_r13": -0.031933987625123665, "b:ebm_i25_bag": -0.02852044811718722, "b:ebm_i25_bag4": -0.021637072121400334, "b:ebm_lead4_bag": 0.02950525709861906, "b:gb_bag10": 0.009815367106015185, "b:gb_bag5": 0.02334053299856081, "b:gb_l3": -0.006951219178063977, "b:gb_ohk": 0.0018332949866820256, "b:gbc_bag3": 0.0038870477010928325, "b:gbe_bag3": 0.0034865909557313356, "b:gbn_bag5_w128": -0.021110583050017305, "b:gbn_bag5_w512": 0.010671461592700661, "b:gbr_bag3": 0.011079141956471712, "b:gbx_bag3": 0.03516678094498248, "b:gbx_bag5": -0.022428290700120656, "b:glm_dig": 0.008039178685718558, "b:glm_dl": 0.010396749892529342, "b:glm_dl_bag": -0.014647055405793691, "b:glm_dl_s50": -0.0007624548737430984, "b:glm_dl_s50_bag": -0.03224603236752054, "b:glm_ohk": -0.0014554228489168505, "b:glm_ohk2": -0.021050754486035085, "b:glm_ohk2_bag": 0.014620275207319703, "b:glm_ohk_c01": 0.009039588532333274, "b:glm_ohkl": -0.015713624347275166, "b:glm_ohkl_bag": 0.03369452774708165, "b:glm_sb": 0.003223769038409277, "b:glm_te": -0.0003188723638362897, "b:lgb_dig_l4": -0.009902399919449884, "b:lgb_dig_l7": -0.006368773687195667, "b:lgb_dig_s100": 0.013405184345650493, "b:lgb_dig_s100_bag": -0.023787932186687104, "b:lgb_dig_s100_bag5": 0.006918520093627308, "b:lgb_dig_s100_l3": -0.008895789461048517, "b:lgb_dig_s100_l3_bag": 0.025983474774013818, "b:lgb_diglead_bag": -0.030799407825723078, "b:lgb_diglead_bag5": 0.026017077980131132, "b:lgb_diglead_l2": 0.0023658909444383616, "b:lgb_diglead_l3": -0.026516477795675772, "b:lgb_diglead_l3_bag": 0.024331011022829137, "b:lgb_diglead_l4": 0.012962785663650984, "b:lgb_dk_l4": 0.00705709120536385, "b:lgb_raw_l31": -0.0013077377856586425, "b:lgb_raw_l4": 0.003361630906731786, "b:lgb_raw_l4_cnt": -0.007073959269025274, "b:lgb_rawdig_l4": -0.0005792167740520289, "b:lgb_te_l4": 0.00777574722953822, "b:lgb_te_l4_s100": -0.005815187528929042, "b:lgb_te_l4_s300": -0.006128145913719751, "b:lgb_te_l4_s5": 0.0049920787148927635, "b:realmlp_allcat": -0.012708496532605128, "b:realmlp_cnt": -0.0007725462723897217, "b:realmlp_dig": 0.0024815610298947965, "b:realmlp_dig_s100": -0.0023806799843974443, "b:realmlp_dig_s2": 0.010535960646496067, "b:realmlp_rawdig": 0.008454413536396574, "b:realmlp_te": -0.0008520837876694293, "b:realmlp_te_cnt": 0.0019657911349665403, "b:realmlp_te_s2": 0.01204414552678721, "b:realmlp_v1": 0.0010588427054775687, "b:xgb_dig_d2": 4.381772846249486e-06, "b:xgb_raw_d3": 0.0017253470690219348, "b:xgb_te_d2": -0.006971556784199019}, "step2": {"s:megayak_5f_lm": 0.027120958470941054, "s:masaya_gpt2": -0.026831064838141933, "s:lm7_5fold": 0.03384961793597077, "s:gpt2_seed2026": -0.011693123169184232, "s:tabpfn_slimypunk_tp1": -0.01433831491008294, "s:sadam_tabpfn_100k_tp3": -0.024955721752621868, "s:yekenot_realmlp": -0.01169405182399045, "s:alex_glm_gen": -0.0071334840179509956, "s:daoviet_gpdvae": 0.020864454325000524, "s:gbn_bag5_w512": 0.06842576013299365, "s:sameer_glm10f": -0.03507718412434708, "s:sadam_v6_gb_xgb": -0.012016332146820127, "s:sadam_v6_gb_lgb": -0.022379148729191445, "s:sadam_v4_cat": -0.016430513876196526, "s:gb_bag5": -0.026469240580663723, "s:megayak_shallow_tree": -0.10658775392950844, "wide": 0.04011039703544269, "b:ebm_all_bag": 0.16064154610662532, "b:ebm_dig": -0.09643097307613457, "b:ebm_dig_bag": 0.1327216874179924, "b:ebm_dig_r7": -0.09527971952157632, "b:ebm_diglead_bag": 0.0024941747544620784, "b:ebm_diglead_i25": 0.04173663876190493, "b:ebm_diglead_r11": -0.037574399736357234, "b:ebm_diglead_r13": -0.06386797525024733, "b:ebm_i25_bag": -0.05704089623437444, "b:ebm_i25_bag4": -0.04327414424280067, "b:ebm_lead4_bag": 0.05901051419723812, "b:gb_bag10": 0.01963073421203037, "b:gb_bag5": 0.04668106599712162, "b:gb_l3": -0.013902438356127955, "b:gb_ohk": 0.0036665899733640513, "b:gbc_bag3": 0.007774095402185665, "b:gbe_bag3": 0.006973181911462671, "b:gbn_bag5_w128": -0.04222116610003461, "b:gbn_bag5_w512": 0.021342923185401323, "b:gbr_bag3": 0.022158283912943423, "b:gbx_bag3": 0.07033356188996497, "b:gbx_bag5": -0.04485658140024131, "b:glm_dig": 0.016078357371437115, "b:glm_dl": 0.020793499785058685, "b:glm_dl_bag": -0.029294110811587382, "b:glm_dl_s50": -0.0015249097474861967, "b:glm_dl_s50_bag": -0.06449206473504109, "b:glm_ohk": -0.002910845697833701, "b:glm_ohk2": -0.04210150897207017, "b:glm_ohk2_bag": 0.029240550414639407, "b:glm_ohk_c01": 0.018079177064666548, "b:glm_ohkl": -0.03142724869455033, "b:glm_ohkl_bag": 0.0673890554941633, "b:glm_sb": 0.006447538076818554, "b:glm_te": -0.0006377447276725794, "b:lgb_dig_l4": -0.019804799838899767, "b:lgb_dig_l7": -0.012737547374391334, "b:lgb_dig_s100": 0.026810368691300986, "b:lgb_dig_s100_bag": -0.04757586437337421, "b:lgb_dig_s100_bag5": 0.013837040187254616, "b:lgb_dig_s100_l3": -0.017791578922097033, "b:lgb_dig_s100_l3_bag": 0.051966949548027636, "b:lgb_diglead_bag": -0.061598815651446155, "b:lgb_diglead_bag5": 0.052034155960262264, "b:lgb_diglead_l2": 0.004731781888876723, "b:lgb_diglead_l3": -0.053032955591351544, "b:lgb_diglead_l3_bag": 0.048662022045658275, "b:lgb_diglead_l4": 0.025925571327301968, "b:lgb_dk_l4": 0.0141141824107277, "b:lgb_raw_l31": -0.002615475571317285, "b:lgb_raw_l4": 0.006723261813463572, "b:lgb_raw_l4_cnt": -0.014147918538050547, "b:lgb_rawdig_l4": -0.0011584335481040577, "b:lgb_te_l4": 0.01555149445907644, "b:lgb_te_l4_s100": -0.011630375057858085, "b:lgb_te_l4_s300": -0.012256291827439501, "b:lgb_te_l4_s5": 0.009984157429785527, "b:realmlp_allcat": -0.025416993065210256, "b:realmlp_cnt": -0.0015450925447794434, "b:realmlp_dig": 0.004963122059789593, "b:realmlp_dig_s100": -0.004761359968794889, "b:realmlp_dig_s2": 0.021071921292992134, "b:realmlp_rawdig": 0.016908827072793148, "b:realmlp_te": -0.0017041675753388585, "b:realmlp_te_cnt": 0.0039315822699330805, "b:realmlp_te_s2": 0.02408829105357442, "b:realmlp_v1": 0.0021176854109551373, "b:xgb_dig_d2": 8.763545692498972e-06, "b:xgb_raw_d3": 0.0034506941380438695, "b:xgb_te_d2": -0.013943113568398039}, "step3": {"s:megayak_5f_lm": 0.04291273505247678, "s:masaya_gpt2": -0.020199757317605413, "s:lm7_5fold": -0.01468694710136923, "s:gpt2_seed2026": 0.025130617752315624, "s:tabpfn_slimypunk_tp1": -0.004207018474268403, "s:sadam_tabpfn_100k_tp3": -0.007856503311289544, "s:yekenot_realmlp": -0.0035719921934940883, "s:alex_glm_gen": -0.00968299752240646, "s:daoviet_gpdvae": -0.0043104617980370685, "s:gbn_bag5_w512": 0.059432477779308095, "s:sameer_glm10f": -0.06276693941875201, "s:sadam_v6_gb_xgb": 0.022456148322158014, "s:sadam_v6_gb_lgb": -0.042356699610454136, "s:sadam_v4_cat": -0.039100958827360595, "s:gb_bag5": -0.012918528414324717, "s:megayak_shallow_tree": -0.14824366284669427, "wide": -0.03978368932406893, "b:ebm_all_bag": 0.1200398804268523, "b:ebm_dig": -0.041702477580910585, "b:ebm_dig_bag": 0.09583943650849604, "b:ebm_dig_r7": -0.08191239292938352, "b:ebm_diglead_bag": 0.01376425646113259, "b:ebm_diglead_i25": 0.04909794514164337, "b:ebm_diglead_r11": -0.011310427337500801, "b:ebm_diglead_r13": -0.057125015279515697, "b:ebm_i25_bag": -0.055536171086130987, "b:ebm_i25_bag4": -0.05374364216150403, "b:ebm_lead4_bag": 0.02357861276219015, "b:gb_bag10": -0.00154617797821281, "b:gb_bag5": 0.05069096767529185, "b:gb_l3": 0.004029341040004309, "b:gb_ohk": 0.01068889956540223, "b:gbc_bag3": -0.0005705614977412649, "b:gbe_bag3": 0.006611024205409296, "b:gbn_bag5_w128": -0.05704424635393862, "b:gbn_bag5_w512": 0.018012229412251064, "b:gbr_bag3": 0.03408846711277557, "b:gbx_bag3": 0.08559259050915491, "b:gbx_bag5": -0.039653789628059245, "b:glm_dig": 0.02116955140650671, "b:glm_dl": 0.004080087233724273, "b:glm_dl_bag": -0.027410071095152855, "b:glm_dl_s50": -0.004560532985992997, "b:glm_dl_s50_bag": -0.04112787498159549, "b:glm_ohk": -0.005032986969082324, "b:glm_ohk2": -0.029147458944055227, "b:glm_ohk2_bag": 0.02415972281018389, "b:glm_ohk_c01": 0.012367133647285219, "b:glm_ohkl": 7.992068950317359e-05, "b:glm_ohkl_bag": 0.051688750332465824, "b:glm_sb": 0.014598944763857322, "b:glm_te": -0.0012856961951914344, "b:lgb_dig_l4": -0.044067513945396465, "b:lgb_dig_l7": -0.023937328022709262, "b:lgb_dig_s100": 0.05015416783062814, "b:lgb_dig_s100_bag": -0.0490099953747574, "b:lgb_dig_s100_bag5": 0.013292507325267445, "b:lgb_dig_s100_l3": -0.035535165439837854, "b:lgb_dig_s100_l3_bag": 0.06438780421173648, "b:lgb_diglead_bag": -0.06663793554784707, "b:lgb_diglead_bag5": 0.08463917295571141, "b:lgb_diglead_l2": 0.03429174384535712, "b:lgb_diglead_l3": -0.0935496417164133, "b:lgb_diglead_l3_bag": 0.05362307552217068, "b:lgb_diglead_l4": 0.0362732151219765, "b:lgb_dk_l4": 0.01552350185504524, "b:lgb_raw_l31": -0.00014661012254933467, "b:lgb_raw_l4": 0.0055072120668919235, "b:lgb_raw_l4_cnt": 0.007272332997137415, "b:lgb_rawdig_l4": -0.010187873970101267, "b:lgb_te_l4": 0.012398979732044752, "b:lgb_te_l4_s100": -0.019261819401553137, "b:lgb_te_l4_s300": -0.01860262372588006, "b:lgb_te_l4_s5": -0.0009787320202008865, "b:realmlp_allcat": -0.023775521024247927, "b:realmlp_cnt": -0.004851163257622534, "b:realmlp_dig": 0.014532461400033083, "b:realmlp_dig_s100": -0.012877357633809388, "b:realmlp_dig_s2": 0.019550644054602478, "b:realmlp_rawdig": 0.03289924143580761, "b:realmlp_te": -0.00795501732687936, "b:realmlp_te_cnt": 0.007044519774050301, "b:realmlp_te_s2": 0.030264994780156384, "b:realmlp_v1": -0.0023391743983786485, "b:xgb_dig_d2": 0.01606411465109771, "b:xgb_raw_d3": -0.0012450782349890336, "b:xgb_te_d2": -0.010690978580007995}, "step4": {"s:megayak_5f_lm": -0.0034888369366524582, "s:masaya_gpt2": -0.015284738285236195, "s:lm7_5fold": -0.03949878826858231, "s:gpt2_seed2026": 0.06946230892735698, "s:tabpfn_slimypunk_tp1": 0.004363023306568827, "s:sadam_tabpfn_100k_tp3": -0.013135678637731263, "s:yekenot_realmlp": 0.011539353358612734, "s:alex_glm_gen": -0.02889340863736857, "s:daoviet_gpdvae": -0.003158411394969963, "s:gbn_bag5_w512": 0.07429864312399857, "s:sameer_glm10f": 0.01083020494444985, "s:sadam_v6_gb_xgb": 0.06192083641586277, "s:sadam_v6_gb_lgb": -0.014245809297054237, "s:sadam_v4_cat": -0.04249350596858786, "s:gb_bag5": -0.018690888456457076, "s:megayak_shallow_tree": -0.035485976486572604, "wide": -0.03894421088390168, "b:ebm_all_bag": 0.03677181337958047, "b:ebm_dig": 0.02417920459799613, "b:ebm_dig_bag": 0.018801516920721822, "b:ebm_dig_r7": -0.04218759710577042, "b:ebm_diglead_bag": 0.010635010029495755, "b:ebm_diglead_i25": 0.04957738213731999, "b:ebm_diglead_r11": 0.026793560344778228, "b:ebm_diglead_r13": -0.018667093320712, "b:ebm_i25_bag": -0.049075610409961654, "b:ebm_i25_bag4": -0.03384809312970651, "b:ebm_lead4_bag": -0.015195704080086938, "b:gb_bag10": -0.03510906450687058, "b:gb_bag5": 0.045443272119950036, "b:gb_l3": 0.011155345916963714, "b:gb_ohk": -0.018980301466611775, "b:gbc_bag3": -0.012156266174697165, "b:gbe_bag3": 0.01731944865242502, "b:gbn_bag5_w128": -0.06436335154682796, "b:gbn_bag5_w512": 0.019785734139272378, "b:gbr_bag3": 0.029508964640899294, "b:gbx_bag3": 0.07462465072432652, "b:gbx_bag5": -0.03848657908060729, "b:glm_dig": 0.02412632595931039, "b:glm_dl": -0.03997507254154347, "b:glm_dl_bag": -0.041032343432782496, "b:glm_dl_s50": -0.009150197988898694, "b:glm_dl_s50_bag": -0.025061916274043687, "b:glm_ohk": -0.03897688288809032, "b:glm_ohk2": -0.023966298606199914, "b:glm_ohk2_bag": -0.022134715832265495, "b:glm_ohk_c01": -0.021532163362983796, "b:glm_ohkl": 0.06424804151643952, "b:glm_ohkl_bag": 0.04655505223495522, "b:glm_sb": 0.013799322854010239, "b:glm_te": -0.00013493658915205456, "b:lgb_dig_l4": -0.045692432389347906, "b:lgb_dig_l7": -0.02324972627301555, "b:lgb_dig_s100": 0.04486202594296171, "b:lgb_dig_s100_bag": -0.06543446351610092, "b:lgb_dig_s100_bag5": 0.017326167041775087, "b:lgb_dig_s100_l3": -0.039522775606475706, "b:lgb_dig_s100_l3_bag": 0.07698351772220281, "b:lgb_diglead_bag": -0.0918785984210468, "b:lgb_diglead_bag5": 0.10136280283757386, "b:lgb_diglead_l2": 0.026979966112458703, "b:lgb_diglead_l3": -0.08544471791540253, "b:lgb_diglead_l3_bag": 0.03991895600315462, "b:lgb_diglead_l4": 0.039131568140953664, "b:lgb_dk_l4": 0.01398481072426439, "b:lgb_raw_l31": 0.008530204922566535, "b:lgb_raw_l4": 0.009975270088348644, "b:lgb_raw_l4_cnt": -0.011305223356159364, "b:lgb_rawdig_l4": -0.004428883454854229, "b:lgb_te_l4": 0.03588414217788614, "b:lgb_te_l4_s100": 0.004402819123102933, "b:lgb_te_l4_s300": -0.033946661892970076, "b:lgb_te_l4_s5": -0.020379548644496457, "b:realmlp_allcat": -0.021847945261691686, "b:realmlp_cnt": -0.005115497538302371, "b:realmlp_dig": 0.009231675087523289, "b:realmlp_dig_s100": -0.021105687058130693, "b:realmlp_dig_s2": 0.026386424981860487, "b:realmlp_rawdig": 0.016088000296832713, "b:realmlp_te": -0.0009418848153852933, "b:realmlp_te_cnt": 0.002193161082475847, "b:realmlp_te_s2": 0.020374264481207105, "b:realmlp_v1": 0.01134837014926146, "b:xgb_dig_d2": 0.008104427329574886, "b:xgb_raw_d3": -0.0071573342074869525, "b:xgb_te_d2": -0.0021432682839698687}, "step5": {"s:megayak_5f_lm": 0.06897650738977462, "s:masaya_gpt2": -0.009015859365062587, "s:lm7_5fold": 0.0009215367354582742, "s:gpt2_seed2026": -0.010251325714452565, "s:tabpfn_slimypunk_tp1": 0.0141561368318242, "s:sadam_tabpfn_100k_tp3": 0.003507144176651289, "s:yekenot_realmlp": 0.000554288971888248, "s:alex_glm_gen": -0.01799965802469446, "s:daoviet_gpdvae": -0.00757453253363324, "s:gbn_bag5_w512": 0.11668761153993083, "s:sameer_glm10f": -0.039776089842193205, "s:sadam_v6_gb_xgb": 0.08655821449241188, "s:sadam_v6_gb_lgb": -0.06304259940783537, "s:sadam_v4_cat": -0.033421792690255456, "s:gb_bag5": -0.04673261552510822, "s:megayak_shallow_tree": -0.1253885827427653, "wide": -0.013401948752305786, "b:ebm_all_bag": 0.16535217569905436, "b:ebm_dig": -0.06466504774415102, "b:ebm_dig_bag": 0.10939458360630332, "b:ebm_dig_r7": -0.10904054532975026, "b:ebm_diglead_bag": 0.15358016925677073, "b:ebm_diglead_i25": 0.03882931804560387, "b:ebm_diglead_r11": -0.10797963213618902, "b:ebm_diglead_r13": -0.141475691530386, "b:ebm_i25_bag": -0.03330261397321142, "b:ebm_i25_bag4": -0.056408685897294995, "b:ebm_lead4_bag": 0.0464173963220906, "b:gb_bag10": 0.03194951889707664, "b:gb_bag5": 0.02047301464717305, "b:gb_l3": 0.05570343463674305, "b:gb_ohk": -0.013256060365589234, "b:gbc_bag3": 0.00530342071496869, "b:gbe_bag3": 0.00273180802397477, "b:gbn_bag5_w128": -0.023383619716230583, "b:gbn_bag5_w512": -0.07096680647838312, "b:gbr_bag3": 0.009083751637504819, "b:gbx_bag3": 0.1020836624830321, "b:gbx_bag5": -0.10372444459039258, "b:glm_dig": 0.013474305968518285, "b:glm_dl": -0.032059274008832454, "b:glm_dl_bag": -0.021473603925046127, "b:glm_dl_s50": 0.020583262666812933, "b:glm_dl_s50_bag": -0.0039183901110275395, "b:glm_ohk": -0.03765485928672982, "b:glm_ohk2": -0.08255453451274214, "b:glm_ohk2_bag": 0.04653641203627543, "b:glm_ohk_c01": 0.013055097738295897, "b:glm_ohkl": 0.06292326173214656, "b:glm_ohkl_bag": 0.021615313062059504, "b:glm_sb": 0.009721586222763782, "b:glm_te": -0.006725737063415938, "b:lgb_dig_l4": -0.055837477082973985, "b:lgb_dig_l7": -0.016350818439952525, "b:lgb_dig_s100": 0.058211729634193396, "b:lgb_dig_s100_bag": -0.06111204912040655, "b:lgb_dig_s100_bag5": -0.0073488267115739215, "b:lgb_dig_s100_l3": -0.06170331263545208, "b:lgb_dig_s100_l3_bag": 0.07503784806078012, "b:lgb_diglead_bag": -0.1028286243985103, "b:lgb_diglead_bag5": 0.13560974793062627, "b:lgb_diglead_l2": 0.03318288998272117, "b:lgb_diglead_l3": -0.07449187136326936, "b:lgb_diglead_l3_bag": 0.03916999201821208, "b:lgb_diglead_l4": 0.02465532784154723, "b:lgb_dk_l4": 0.007398935534430129, "b:lgb_raw_l31": 0.0065574200392839775, "b:lgb_raw_l4": 0.0029355975964231408, "b:lgb_raw_l4_cnt": 0.013767449345681268, "b:lgb_rawdig_l4": -0.004514260114381885, "b:lgb_te_l4": 0.01348779111629848, "b:lgb_te_l4_s100": 0.01580164481063912, "b:lgb_te_l4_s300": -0.036475794446526356, "b:lgb_te_l4_s5": 0.0010410112240718519, "b:realmlp_allcat": -0.011665638181812679, "b:realmlp_cnt": -0.013742954748969567, "b:realmlp_dig": 0.008894012891697738, "b:realmlp_dig_s100": -0.02251635410884371, "b:realmlp_dig_s2": 0.00834843745840769, "b:realmlp_rawdig": 0.01743081519880163, "b:realmlp_te": 0.0034764493021211245, "b:realmlp_te_cnt": 0.0026581217023631744, "b:realmlp_te_s2": 0.015409114581696826, "b:realmlp_v1": 0.005522739951764543, "b:xgb_dig_d2": 0.028929272398427484, "b:xgb_raw_d3": -0.016849732900939916, "b:xgb_te_d2": 0.005379140652384981}, "step6": {"s:megayak_5f_lm": -0.11167117542101894, "s:masaya_gpt2": 0.008580731570136326, "s:lm7_5fold": -0.0009400463836016774, "s:gpt2_seed2026": 0.027885869579209244, "s:tabpfn_slimypunk_tp1": 0.0016924449935887891, "s:sadam_tabpfn_100k_tp3": 0.01432901271297176, "s:yekenot_realmlp": 0.022390415632839642, "s:alex_glm_gen": -0.0005758371133733123, "s:daoviet_gpdvae": 0.01311917773336681, "s:gbn_bag5_w512": -0.009545946811432997, "s:sameer_glm10f": 0.05630282825450275, "s:sadam_v6_gb_xgb": -0.08321586634169159, "s:sadam_v6_gb_lgb": 0.10171308794729168, "s:sadam_v4_cat": 0.03041043931485113, "s:gb_bag5": -0.013425667860656836, "s:megayak_shallow_tree": 0.18113423530486364, "wide": 0.03224417977782197, "b:ebm_all_bag": -0.03928670360117938, "b:ebm_dig": 0.015432582908453892, "b:ebm_dig_bag": -0.019730929338460986, "b:ebm_dig_r7": 0.021228118735024122, "b:ebm_diglead_bag": -0.019405585095929167, "b:ebm_diglead_i25": -0.016076869397068764, "b:ebm_diglead_r11": 0.019304661106694525, "b:ebm_diglead_r13": 0.025249635776052333, "b:ebm_i25_bag": -0.016462397253955638, "b:ebm_i25_bag4": 0.04030328340809125, "b:ebm_lead4_bag": -0.046473301351245, "b:gb_bag10": -0.004465127493976626, "b:gb_bag5": 0.005295157073829479, "b:gb_l3": -0.07872430096932825, "b:gb_ohk": 0.007827567154987038, "b:gbc_bag3": -0.03280747819608797, "b:gbe_bag3": 0.01175576219083039, "b:gbn_bag5_w128": 0.018717264487127714, "b:gbn_bag5_w512": -0.0011920109259677498, "b:gbr_bag3": 0.004896649909764378, "b:gbx_bag3": -0.011667422511690605, "b:gbx_bag5": 0.015530360868756142, "b:glm_dig": 0.004036061702293574, "b:glm_dl": 0.04219485215779119, "b:glm_dl_bag": 0.010864644604441424, "b:glm_dl_s50": -0.008276003887968196, "b:glm_dl_s50_bag": -0.05826467282773935, "b:glm_ohk": 0.043882937000895586, "b:glm_ohk2": 0.02848919574777289, "b:glm_ohk2_bag": -0.009287167708046554, "b:glm_ohk_c01": -0.03407033309595358, "b:glm_ohkl": -0.013764349514068144, "b:glm_ohkl_bag": -0.03603962101306747, "b:glm_sb": -0.0023173698639354762, "b:glm_te": -0.0009163310748722242, "b:lgb_dig_l4": 0.06490818309252397, "b:lgb_dig_l7": 0.005332908943789934, "b:lgb_dig_s100": -0.039099926221213735, "b:lgb_dig_s100_bag": 0.021086002265371388, "b:lgb_dig_s100_bag5": 0.018067651487240086, "b:lgb_dig_s100_l3": 0.050308694696048394, "b:lgb_dig_s100_l3_bag": -0.021781737915879996, "b:lgb_diglead_bag": 0.025863505660522864, "b:lgb_diglead_bag5": -0.09141971792291925, "b:lgb_diglead_l2": -0.03604788078810194, "b:lgb_diglead_l3": 0.039415925471774575, "b:lgb_diglead_l3_bag": -0.023375313510179387, "b:lgb_diglead_l4": 0.005674349495556749, "b:lgb_dk_l4": 0.0017674212665991548, "b:lgb_raw_l31": -0.004471455199590837, "b:lgb_raw_l4": -0.0036951771362210197, "b:lgb_raw_l4_cnt": -0.053748077290576096, "b:lgb_rawdig_l4": 0.005242284896971225, "b:lgb_te_l4": -0.021624575044207693, "b:lgb_te_l4_s100": -0.01718882590212164, "b:lgb_te_l4_s300": 0.03017029260151744, "b:lgb_te_l4_s5": -0.02436183442707204, "b:realmlp_allcat": 0.013600667820710632, "b:realmlp_cnt": 0.019958712814710327, "b:realmlp_dig": -0.0076563752581303365, "b:realmlp_dig_s100": 0.023659717425888894, "b:realmlp_dig_s2": -0.0029996770123733116, "b:realmlp_rawdig": -0.013394120120286176, "b:realmlp_te": -0.013346668023044723, "b:realmlp_te_cnt": -0.0017512931976057325, "b:realmlp_te_s2": -0.0175153416436808, "b:realmlp_v1": -0.0004147861382334795, "b:xgb_dig_d2": -0.04178865951042272, "b:xgb_raw_d3": 0.004541591163347932, "b:xgb_te_d2": -0.0006877082372638679}, "step7": {"s:megayak_5f_lm": -0.01532889506612532, "s:masaya_gpt2": -0.013261391079182689, "s:lm7_5fold": 0.004874633623866584, "s:gpt2_seed2026": -0.011048328252712398, "s:tabpfn_slimypunk_tp1": 0.019479195136036475, "s:sadam_tabpfn_100k_tp3": 0.03379897098415831, "s:yekenot_realmlp": 0.029448884712266525, "s:alex_glm_gen": -0.04076182290815198, "s:daoviet_gpdvae": 0.03885285756599925, "s:gbn_bag5_w512": 0.016705008805907157, "s:sameer_glm10f": -0.033671768825366336, "s:sadam_v6_gb_xgb": 0.005793147308091748, "s:sadam_v6_gb_lgb": 0.035878360062331686, "s:sadam_v4_cat": -0.014177348042369899, "s:gb_bag5": -0.002983076133072762, "s:megayak_shallow_tree": 0.0429774041829996, "wide": -0.0012445682470603396, "b:ebm_all_bag": 0.011416914599689673, "b:ebm_dig": 0.02294984166507186, "b:ebm_dig_bag": 0.028877548053109183, "b:ebm_dig_r7": -0.04276285768277372, "b:ebm_diglead_bag": -0.008037767147017849, "b:ebm_diglead_i25": 0.019575714007213182, "b:ebm_diglead_r11": 0.005822450818036772, "b:ebm_diglead_r13": -0.03646698657915979, "b:ebm_i25_bag": -0.045582127670408, "b:ebm_i25_bag4": -0.023571440063628404, "b:ebm_lead4_bag": 0.0024724934563454997, "b:gb_bag10": 0.010602709995019643, "b:gb_bag5": -0.00020948904926665563, "b:gb_l3": -0.005008883794821487, "b:gb_ohk": 0.012462403114245438, "b:gbc_bag3": -0.04491596636131862, "b:gbe_bag3": 0.021281358413983074, "b:gbn_bag5_w128": 0.03773384363936814, "b:gbn_bag5_w512": -0.042487368245716466, "b:gbr_bag3": 0.02379136220071678, "b:gbx_bag3": 0.0801743069671423, "b:gbx_bag5": -0.06676185994403352, "b:glm_dig": 0.018579401866817363, "b:glm_dl": 0.047057489740079596, "b:glm_dl_bag": 0.012129654721577742, "b:glm_dl_s50": -0.016842606244825434, "b:glm_dl_s50_bag": -0.046154572849136107, "b:glm_ohk": 0.0279298113811889, "b:glm_ohk2": -0.005464711505301273, "b:glm_ohk2_bag": -0.00444217654093287, "b:glm_ohk_c01": -0.02521683001276968, "b:glm_ohkl": 0.00998815286349657, "b:glm_ohkl_bag": 0.011188677652772922, "b:glm_sb": 0.011534339980283794, "b:glm_te": -0.01941353864783445, "b:lgb_dig_l4": 0.012792179508792752, "b:lgb_dig_l7": -0.016536600806802363, "b:lgb_dig_s100": 0.025988292377181515, "b:lgb_dig_s100_bag": -0.02786056607370909, "b:lgb_dig_s100_bag5": 0.004711599818220775, "b:lgb_dig_s100_l3": -0.018316961517069283, "b:lgb_dig_s100_l3_bag": 0.05854090458349093, "b:lgb_diglead_bag": -0.058955263331882175, "b:lgb_diglead_bag5": 0.042922577252749214, "b:lgb_diglead_l2": -0.0006251518955284371, "b:lgb_diglead_l3": -0.04723462060156513, "b:lgb_diglead_l3_bag": 0.005509594253736563, "b:lgb_diglead_l4": 0.025174373862716728, "b:lgb_dk_l4": 0.015083685645311297, "b:lgb_raw_l31": 0.008250997152138546, "b:lgb_raw_l4": -0.02098204469918052, "b:lgb_raw_l4_cnt": -0.04670494103234027, "b:lgb_rawdig_l4": 0.0004929563863006859, "b:lgb_te_l4": -0.02137609804945939, "b:lgb_te_l4_s100": -0.002944265144476053, "b:lgb_te_l4_s300": -0.012556895046753664, "b:lgb_te_l4_s5": -0.030877460829031114, "b:realmlp_allcat": 0.01652923534649207, "b:realmlp_cnt": 0.005679425026097773, "b:realmlp_dig": 0.00040613096397866576, "b:realmlp_dig_s100": 0.0021495226045041803, "b:realmlp_dig_s2": -0.006397105937466286, "b:realmlp_rawdig": 0.005301293537676727, "b:realmlp_te": -0.018820769127796483, "b:realmlp_te_cnt": 0.007281638359193587, "b:realmlp_te_s2": -0.013005975189400712, "b:realmlp_v1": 0.007037322452335817, "b:xgb_dig_d2": 0.015738726611111017, "b:xgb_raw_d3": -0.031273627583305666, "b:xgb_te_d2": 0.01834167087631238}}')                                  # ten steps; each moves the file it starts from towards the same 87 members
FROM30 = {'joint': 'areeb', 'cv_led': 'areeb', 'step1': 'areeb', 'step2_half': 'step1', 'step2': 'step1', 'step3': 'step2', 'step4': 'step3', 'step5': 'step4', 'step6': 'step4', 'step7': 'step6'}
PUB30 = {'areeb': 0.95388, 'joint': 0.95388, 'cv_led': 0.95387, 'step1': 0.95390, 'step2_half': 0.95393, 'step2': 0.95396, 'step3': 0.95404, 'step4': 0.95408, 'step5': 0.95407, 'step6': 0.95412, 'step7': 0.95417}
NEST30 = {'joint': 3.3, 'cv_led': 7.3, 'step1': -1.1, 'step2_half': -8.6, 'step2': -18.4, 'step3': -29.2, 'step4': -18.6, 'step5': -33.3, 'step6': 16.0, 'step7': -19.0}   # goodpjw2008's nested CV change, from their table
if not _ok29: print('v30 skipped: the v29 inputs are not mounted')
if _ok29:
    SUMO30 = {r: np.zeros(len(X)) for r in R30}; SUMT30 = {r: 0 for r in R30}
    for k in R29['bolder']:                                # the same 87 members as v29
        if k.startswith('s:'): o_, t_ = _L29(Zb['oof_' + k[2:]]), _L29(Zb['test_' + k[2:]])
        elif k.startswith('b:'): o_, t_ = _L29(np.load(_bank29 / f'oof_{k[2:]}.npy')), _L29(np.load(_bank29 / f'test_{k[2:]}.npy'))
        else: o_, t_ = WO29, WT29
        for r in R30: SUMO30[r] = SUMO30[r] + R30[r][k] * o_; SUMT30[r] = SUMT30[r] + R30[r][k] * t_
    ST30, SO30 = {'areeb': b388}, {'areeb': bo29}         # test side (what gets submitted) and labelled side (the replay), both from v29
    for r in R30:
        w_ = sum(R30[r].values())
        ST30[r] = (1 - w_) * _L29(ST30[FROM30[r]]) + SUMT30[r]; SO30[r] = (1 - w_) * _L29(SO30[FROM30[r]]) + SUMO30[r]
    rows30 = []
    for r in ['areeb'] + list(R30):
        f_, a_ = FROM30.get(r), _fauc(yv28, SO30[r])
        rows30.append([r, f_ or '', PUB30[r], (PUB30[r] - PUB30[f_]) * 1e5 if f_ else np.nan, a_, (a_ - _fauc(yv28, SO30[f_])) * 1e5 if f_ else np.nan,
                       _ups28(SO30[r], SO30[f_]) if f_ else np.nan, NEST30.get(r, np.nan), (a_ - a29) * 1e5, _ups28(SO30[r], F29)])
    A30 = pd.DataFrame(rows30, columns=['step', 'starts from', 'public LB', 'public change', 'AUC (in-sample)', 'labelled change', 'folds up', 'nested (goodpjw2008)', 'pts vs our held-out stack', 'folds up vs ours']).set_index('step')
    print('the ten steps on the board and on labelled rows (1 point = 0.00001; labelled change: in-sample replay, from the file the step starts from):')
    print(A30.round({'AUC (in-sample)': 6, 'labelled change': 1, 'pts vs our held-out stack': 1}).to_string())
    d30 = A30.dropna(subset=['public change'])
    print(f"correlation of public change and labelled change over the ten steps: {np.corrcoef(d30['public change'], d30['labelled change'])[0, 1]:+.2f} (with goodpjw2008's nested numbers {np.corrcoef(d30['public change'], d30['nested (goodpjw2008)'])[0, 1]:+.2f})")
    print(f"the chain 0.95388 -> step 7: public {(PUB30['step7'] - PUB30['areeb']) * 1e5:+.0f} points, labelled rows {(_fauc(yv28, SO30['step7']) - _fauc(yv28, bo29)) * 1e5:+.1f} points")
    rng30 = np.random.default_rng(30)
    for r in ('step7', 'cv_led'):
        g_ = []
        for _ in range(100):
            i_ = rng30.choice(len(yv28), n_pub28, replace=False); g_.append((_fauc(yv28[i_], SO30[r][i_]) - _fauc(yv28[i_], bo29[i_])) * 1e5)
        g_ = np.array(g_); pc_ = (PUB30[r] - PUB30['areeb']) * 1e5
        print(f'{r} - the 0.95388 file on {n_pub28} random labelled rows: {g_.mean():+.1f} +- {g_.std():.1f} points; at or above its public change ({pc_:+.0f}) in {np.mean(g_ >= pc_):.0%} of 100 draws')
    for n_, r in [('submission_s417_wide10.csv', 'step7'), ('submission_cvled_wide10.csv', 'cv_led')]:
        v_ = 0.9 * _L29(ST30[r]) + 0.1 * _lr(t_w); o_ = 0.9 * _L29(SO30[r]) + 0.1 * F29
        T[n_] = v_; pd.DataFrame({'id': test['id'], TARGET: rankdata(v_) / len(v_)}).to_csv(OUT / n_, index=False)
        print('wrote', n_, f'| on labelled rows {(_fauc(yv28, o_) - _fauc(yv28, SO30[r])) * 1e5:+.1f} points over the file ({_ups28(o_, SO30[r])} of 5 folds) | distance from the file {_dist28(v_, ST30[r]):.1f}e-4, from our wide stack {_dist28(v_, t_w):.1f}e-4')
    T['the 0.95417 file (rebuilt)'] = ST30['step7']
    # every board file of v28-v30 whose labelled-row change we measured: (public change, labelled change), each against the file it was compared with
    OURS30 = {'v28: 0.95370 file + 10% ours': (-3, 13.4), 'v28: + 25% ours': (-11, 30.9), 'v29: 0.95388 file + 10% ours': (-2, 5.1), 'v29: + 50% ours': (-29, 17.1), 'v29: our stack + bold move': (18, -1.5)}
    fig, (c1, c2) = plt.subplots(1, 2, figsize=(14, 4.8))
    c1.axhline(0, color='#999', lw=1); c1.axvline(0, color='#999', lw=1)
    c1.scatter(d30['labelled change'], d30['public change'], color='#c98b2b', s=40, zorder=3, label="goodpjw2008's ten steps (in-sample replay)")
    for k, r_ in d30.iterrows(): c1.annotate(k.replace('_', ' '), (r_['labelled change'], r_['public change']), textcoords='offset points', xytext=(5, 3), fontsize=8)
    c1.scatter([v[1] for v in OURS30.values()], [v[0] for v in OURS30.values()], color='#4a6fa5', s=40, marker='s', zorder=3, label='our board files, v28-v29')
    for k, v in OURS30.items(): c1.annotate(k.split(': ')[1], (v[1], v[0]), textcoords='offset points', xytext=(5, -10), fontsize=7, color='#4a6fa5')
    c1.set_xlabel('change on labelled rows, points'); c1.set_ylabel('change on the public board, points'); c1.legend(fontsize=8, loc='lower left')
    c1.set_title('two referees: the public 20% and the labelled rows (15 files)', fontsize=10)
    ch_ = ['areeb', 'step1', 'step2', 'step3', 'step4', 'step6', 'step7']
    c2.plot(range(len(ch_)), [(PUB30[r] - PUB30['areeb']) * 1e5 for r in ch_], 's-', color='#b0413e', label='public LB')
    c2.plot(range(len(ch_)), [(_fauc(yv28, SO30[r]) - _fauc(yv28, bo29)) * 1e5 for r in ch_], 'o-', color='#c98b2b', label='labelled rows (in-sample replay)')
    c2.axhline((a29 - _fauc(yv28, bo29)) * 1e5, color='#4a6fa5', lw=2, ls='--', label='our held-out wide stack')
    c2.set_xticks(range(len(ch_))); c2.set_xticklabels(['0.95388'] + [r.replace('step', 'step ') for r in ch_[1:]], fontsize=8)
    c2.set_ylabel('points from the 0.95388 file'); c2.set_title('the climb to 0.95417: up on the board, down on labelled rows', fontsize=10); c2.legend(fontsize=8, loc='lower left')
    plt.tight_layout(); plt.show()


## Part C -- pushing past 0.95674: newest public artifacts + a consensus blend

Deep-research findings applied here (public work published up to mid-November 2026):

* The reference stack tops out at public LB ~0.9539 (its own `submission.csv`) and ~0.95386-0.95417 only by leaning on
  *other authors'* public files (areebbb 0.95388, goodpjw2008 10-step forecast ladder to 0.95417, Aman Atar 0.95370,
  Moran Z Strike 9/10, knightfork 0.95357, daoviet 0.95354, muhammadshoaibaltaf 0.95382).
* New prediction banks published after the reference froze its lists are now folded in generically:
  `mathsonchain/s6e11-fraud-oof-test-17-single-models` (17 single models, CV to 0.95103, fold ids shipped),
  `goodpjw2008/s6e11-fraud-series-artifacts` v3 (updated 2026-11-15: RealMLP v1/te/dig, EBM dig/b4, GLM one-hot-key,
  4-leaf LightGBM bags + full refits), and **any** `oof/test` npy pair or `submission*.csv` found under `/kaggle/input`.
* The blend strategy that has actually moved the public board: heavy rank-logit mix of the strongest public file(s)
  with a small share of our CV-backed wide stack (the 90/10 recipe scored 0.95386 on the 0.95388 file). Part C
  automates that recipe for whatever the strongest mounted file is, and also writes a wide-pool refit that includes
  every newly discovered member.

In [ ]:
# --- Part C.1: discover every public prediction artifact mounted under /kaggle/input ---
# Two kinds of assets: (a) test-side submission files -> PUB_FILES, (b) (oof, test) pairs -> pool.
import glob as _gC, os as _osC

PUB_FILES = {}          # name -> test-side score vector, in test.csv row order
DISCOVERED = {}         # name -> (oof, test) added to the wide pool
_skipped = []

def _vec_from_csv(path):
    try:
        df = pd.read_csv(path)
        if 'id' not in df.columns:
            return None
        target_cols = [c for c in df.columns if c.lower() in ('is_fraud', 'prediction', 'score', 'target')]
        c = target_cols[-1] if target_cols else df.columns[-1]
        s = df.set_index('id')[c]
        v = s.reindex(test['id'].to_numpy()).to_numpy(dtype=float)
        return v if np.isfinite(v).all() and np.unique(v).size > 1 else None
    except Exception:
        return None

def _add_pub_file(name, path):
    try:
        if path.lower().endswith('.csv'):
            v = _vec_from_csv(path)
            if v is None:
                return
        else:
            v = np.load(path, allow_pickle=False).ravel().astype(float)
            if len(v) != len(XT) or not np.isfinite(v).all() or np.unique(v).size <= 1:
                return
        PUB_FILES[name] = v
        print('  public file:', name)
    except Exception as e:
        _skipped.append((name, str(e)[:80]))

if KAGGLE:
    # every csv that looks like a submission (has id + a prediction-like column, right length)
    for pth in _gC.glob('/kaggle/input/**/*.csv', recursive=True):
        b = _osC.path.basename(pth).lower()
        if 'sample' in b or b in ('train.csv', 'test.csv', 'folds.csv'):
            continue
        if 'submission' in b or 'pred' in b:
            _add_pub_file('file:' + pth.split('/kaggle/input/')[-1], pth)
    # npy prediction pairs:  X_oof.npy + X_test.npy | X_full.npy | X_refit*.npy ; oof_X.npy + test_X.npy
    npys = list(_gC.glob('/kaggle/input/**/*.npy', recursive=True))
    by_dir = {}
    for pth in npys:
        by_dir.setdefault(_osC.path.dirname(pth), []).append(pth)

    def _load_pair(po, pt):
        try:
            o = np.load(po, allow_pickle=False).ravel().astype(float)
            t = np.load(pt, allow_pickle=False).ravel().astype(float)
            return (o, t) if len(o) == len(X) and len(t) == len(XT) and np.isfinite(o).all() and np.isfinite(t).all() else None
        except Exception:
            return None

    for d, files in by_dir.items():
        names = {_osC.path.basename(f): f for f in files}
        for bn, fp in names.items():
            if 'oof' not in bn.lower():
                continue
            stem = bn.lower().replace('_oof', '').replace('oof_', '').replace('.npy', '')
            cands = []
            for suf in ('test', 'full', 'refit', 'test_f0'):
                for cand_bn in (f'{suf}_{stem}.npy', f'{stem}_{suf}.npy', f'{stem}_test.npy', f'{stem}_full.npy'):
                    if cand_bn in names and cand_bn != bn:
                        cands.append(names[cand_bn])
            # alexchilton cache style: oof_HASH.npy -> test_HASH.npy
            if bn.startswith('oof_'):
                h = bn[4:]
                for cb in (f'test_{h}',):
                    if cb in names:
                        cands.append(names[cb])
            for cp in cands[:2]:
                pair = _load_pair(fp, cp)
                if pair is not None:
                    key = f'{_osC.path.basename(d)}/{_osC.path.basename(fp)}+{_osC.path.basename(cp)}'
                    DISCOVERED[key] = pair
                    break
    for pth in _gC.glob('/kaggle/input/**/*.npz', recursive=True):
        try:
            z = np.load(pth, allow_pickle=False)
            ks = list(z.keys())
            ok = [k for k in ks if 'oof' in k.lower()]
            tk = [k for k in ks if ('test' in k.lower() or 'full' in k.lower()) and 'oof' not in k.lower()]
            if ok and tk:
                o = z[ok[0]].ravel().astype(float); t = z[tk[0]].ravel().astype(float)
                if len(o) == len(X) and len(t) == len(XT) and np.isfinite(o).all() and np.isfinite(t).all():
                    DISCOVERED[f'npz:{pth.split("/kaggle/input/")[-1]}'] = (o, t)
        except Exception:
            pass
    for k, (o, t) in DISCOVERED.items():
        _pool_add('disc ' + k, o, t)

print(f'public test files found: {len(PUB_FILES)} | oof/test pairs found: {len(DISCOVERED)} | skipped: {len(_skipped)}')

In [ ]:
# --- Part C.2: explicit loaders for the newest published banks (redundant with C.1 when mounted) ---
# mathsonchain: 17 single models; fold_ids.npy ships the partition -> verify it is ours first.
d = _dir('fold_ids.npy', '17-single-models') or _dir('fold_ids.npy', 'oof-test-17')
if d:
    try:
        fids = np.load(f'{d}/fold_ids.npy')
        same = len(fids) == len(X) and all(len(set(fids[fold42 == k])) == 1 for k in range(5))
        print('mathsonchain fold_ids match our partition:', same)
        if same:
            for pth in sorted(_gC.glob(f'{d}/oof_*.npy')) + sorted(_gC.glob(f'{d}/*_oof.npy')):
                bn = _osC.path.basename(pth); stem = bn.replace('oof_', '').replace('_oof', '').replace('.npy', '')
                for tp in (f'{d}/test_{stem}.npy', f'{d}/{stem}_test.npy', f'{d}/{stem}_full.npy'):
                    if _osC.path.exists(tp):
                        try: _pool_add(f'mathsonchain {stem}', np.load(pth), np.load(tp))
                        except Exception as e: print('  skipped mathsonchain', stem, type(e).__name__)
                        break
    except Exception as e:
        print('mathsonchain loader skipped:', type(e).__name__, str(e)[:80])
# goodpjw2008 fraud-series-artifacts members (RealMLP/EBM/GLM/LightGBM bags + full refits)
d = _dir('realmlp_v1_oof.npy', 'fraud-series')
if d:
    for pth in sorted(_gC.glob(f'{d}/*_oof.npy')):
        try:
            bn = _osC.path.basename(pth); stem = bn[:-8]                      # strip _oof.npy
            tp = f'{d}/{stem}_test.npy'
            fp = f'{d}/{stem}_full.npy'
            t = np.load(tp) if _osC.path.exists(tp) else (np.load(fp) if _osC.path.exists(fp) else None)
            if t is not None:
                _pool_add(f'artifacts {stem}', np.load(pth), t)
        except Exception as e:
            print('  skipped artifacts', _osC.path.basename(pth), type(e).__name__)
print('pool size now:', len(pool))

In [ ]:
# --- Part C.3: refit the wide stacker on the enlarged pool ---
# Rebuild the keep-filter and the C=0.01 logistic stacker with every discovered member included.
pauc2 = {k: _fauc(yv28, v[0]) for k, v in pool.items()}
own_top2 = max((_fauc(yv28, oof[k]) for k in names), default=0.0)
top2 = max(max(pauc2.values(), default=0.0), own_top2)
S2 = np.stack([_std(oof[k]) for k in names]); kept2 = []
for k in sorted(pool, key=pauc2.get, reverse=True):
    if pauc2[k] < top2 - 0.008:
        continue
    s = _std(pool[k][0])
    if (S2 @ s / len(s)).max() < 0.99995:
        kept2.append(k); S2 = np.vstack([S2, s])
del S2
print(f'wide pool: {len(names)} own/bank members + {len(kept2)} public members (was {len(kept)})')
ZW2 = np.column_stack([Z] + [lr_(pool[k][0]) for k in kept2]); ZWT2 = np.column_stack([ZT] + [lr_(pool[k][1]) for k in kept2])
wnames2 = names + kept2
wide2_runs, wide2_hold = [], {}
for seed in (7, 8, 9):
    s = np.zeros(len(ZW2))
    for va_i, dv in Parallel(n_jobs=-1)(delayed(_fold)(ZW2, y.values, tr_i, va_i) for tr_i, va_i in StratifiedKFold(5, shuffle=True, random_state=seed).split(ZW2, y)):
        s[va_i] = dv
    wide2_runs.append(roc_auc_score(y, s)); wide2_hold[seed] = s
print(f'wide stacker v2 held-out AUC over splits 7/8/9: {np.mean(wide2_runs):.5f} {np.round(wide2_runs, 5)}  (v1 was {np.mean(wide_runs):.5f})')
wstk2 = LogisticRegression(C=0.01, max_iter=3000, tol=1e-6).fit(ZW2, y)
t_w2 = wstk2.decision_function(ZWT2)
if np.mean(wide2_runs) > np.mean(wide_runs):
    t_w = t_w2; wcoef = pd.Series(wstk2.coef_[0], index=wnames2)
    print('wide stacker v2 adopted as the own-stack side of the final blend')
else:
    print('wide stacker v1 kept (v2 did not improve held-out AUC)')

In [ ]:
# --- Part C.4: the final consensus blend ---
# Claimed public LB of every file we can name (from the research); unknown files rank below the known ladder.
CLAIMED = {
    'areebbb ladder, rebuilt (LB 0.95347)': 0.95347, 'the 0.95388 file (rebuilt)': 0.95388,
    'the 0.95417 file (rebuilt)': 0.95417, 'amanatar 0.95370 file': 0.95370,
    'mix 0.65 Part 12 + 0.35 our wide': 0.95305, 'submission_p12_ours.csv': 0.95305,
    'submission_p12_ours_lm12.csv': 0.95305, 'Moran Z Strike 9 (LB 0.95330)': 0.95330,
    'amanatar Strike 10 (LB 0.95340)': 0.95340, 'goodpjw2008 Part 12 wide': 0.95285,
}
def _claim(name):
    for k, v in CLAIMED.items():
        if k in name:
            return v
    return 0.0

# gather every public test-side file we have: rebuilt ladder files + discovered csvs
CAND = {}
for k, v in T.items():
    CAND[k] = v
for k, v in PUB_FILES.items():
    CAND['pub ' + k] = v
ranked = sorted(CAND, key=lambda k: (_claim(k), k))
print('public-side candidates by claimed LB:')
for k in ranked[-12:]:
    print(f'  {_claim(k):.5f}  {k}')

claimed = [k for k in ranked if _claim(k) > 0]
best_pub = claimed[-1] if claimed else (ranked[-1] if ranked else None)
fallback_only = best_pub is not None and _claim(best_pub) == 0
if best_pub is not None and not fallback_only:
    F_star = _lr(CAND[best_pub])
    own = _lr(t_w)
    # the recipe family that produced 0.95386/0.95305: heavy public file + small CV-backed stack share.
    for w_own, tag in [(0.05, 'w05'), (0.10, 'w10'), (0.20, 'w20')]:
        v_ = (1 - w_own) * F_star + w_own * own
        safe = ''.join(ch if ch.isalnum() else '_' for ch in best_pub)[:40]
        pd.DataFrame({'id': test['id'], TARGET: rankdata(v_) / len(v_)}).to_csv(OUT / f'submission_{tag}_{safe}.csv', index=False)
    # top-3 public files rank-averaged + own stack: diversity across the strongest public files
    top3 = [_lr(CAND[k]) for k in ranked[-3:]]
    v_mix = 0.80 * np.mean(top3, axis=0) + 0.20 * own
    pd.DataFrame({'id': test['id'], TARGET: rankdata(v_mix) / len(v_mix)}).to_csv(OUT / 'submission_consensus_top3.csv', index=False)
    # final submission: the aggressive end of the proven recipe (90% strongest public file + 10% own wide stack)
    v_final = 0.90 * F_star + 0.10 * own
    pd.DataFrame({'id': test['id'], TARGET: rankdata(v_final) / len(v_final)}).to_csv(OUT / 'submission.csv', index=False)
    print(f'submission.csv = 0.90 x {best_pub} (claimed {_claim(best_pub):.5f}) + 0.10 x own wide stack')
    print('also wrote submission_consensus_top3.csv and the w05/w10/w20 variants for the board')
elif best_pub is not None:
    # no file with a known LB: blend the discovered public files conservatively with our stack
    disc = [k for k in ranked if k.startswith('pub file:')]
    use = disc[-5:] if disc else ranked[-5:]
    v_fb = 0.5 * np.mean([_lr(CAND[k]) for k in use], axis=0) + 0.5 * _lr(t_w)
    pd.DataFrame({'id': test['id'], TARGET: rankdata(v_fb) / len(v_fb)}).to_csv(OUT / 'submission.csv', index=False)
    print(f'no claimed-LB file mounted: submission.csv = mean of {len(use)} discovered public files + 0.5 own wide stack')
else:
    print('no public files mounted: submission.csv stays the wide/narrow stacker choice from Part B')

### Optional: conservative pseudo-label refinement (off by default)

The v22 idea kept as a probe: treat the most confident test predictions as pseudo-labels, retrain on
train + down-weighted pseudo rows, blend back. It cannot be validated on the leaderboard, so it stays
disabled; flip `RUN_PSEUDO_LABELING` to write `submission_pseudo.csv`.

In [ ]:
# --- Optional pseudo-label refinement (v22 idea), OFF by default ---
RUN_PSEUDO_LABELING = False
PSEUDO_LOW_Q, PSEUDO_HIGH_Q = 0.005, 0.995     # rank-fraction tails of the wide stack's test score
PSEUDO_W, PSEUDO_BLEND = 0.25, 0.25

if RUN_PSEUDO_LABELING:
    score = t_w if 't_w' in dir() else blend
    r = rankdata(score) / len(score)
    pm = (r < PSEUDO_LOW_Q) | (r > PSEUDO_HIGH_Q)
    pidx = np.flatnonzero(pm)
    plab = (r[pidx] > 0.5).astype(int)
    print(f'pseudo candidates: {len(pidx):,} ({int(plab.sum()):,} positive)')
    X_aug = pd.concat([X.reset_index(drop=True), XT.iloc[pidx].reset_index(drop=True)], ignore_index=True)
    y_aug = pd.concat([y.reset_index(drop=True), pd.Series(plab)], ignore_index=True)
    w_aug = np.concatenate([np.ones(len(y)), np.full(len(pidx), PSEUDO_W)])
    m = lgb.LGBMClassifier(n_estimators=4000, learning_rate=0.05, num_leaves=4, min_child_samples=100,
                           subsample=0.8, subsample_freq=1, colsample_bytree=0.5, reg_lambda=1.0,
                           n_jobs=THREADS, verbose=-1)
    m.fit(X_aug, y_aug, sample_weight=w_aug)
    p_pseudo = m.predict_proba(XT)[:, 1]
    v_ = (1 - PSEUDO_BLEND) * _lr(t_w) + PSEUDO_BLEND * _lr(p_pseudo)
    pd.DataFrame({'id': test['id'], TARGET: rankdata(v_) / len(v_)}).to_csv(OUT / 'submission_pseudo.csv', index=False)
    print('wrote submission_pseudo.csv')
else:
    print('pseudo-labeling disabled; submission.csv unchanged')

### Reading the blend: three lessons that carry to other tabular competitions

**1. A good model can get zero weight.** In v5, RealMLP scored 0.94704 on its own, about as well as CatBoost (0.94721),
and the blend gave both weight 0. A blend does not reward a member for being good; it rewards it for being *right where the others are wrong*.
Look at the rank-correlation matrix above: members that correlate above ~0.99 with a stronger one add almost nothing.
The reverse also happens: in v6, TabM scored only 0.94708 alone, one of the weakest members, and the stacker gave it the third-largest weight (0.123),
because a neural net's errors differ from the trees'. So judge a new model by the blend AUC with and without it, never by its own score.

**2. Smaller trees won here.** 4-leaf LightGBM scored 0.94838, the same LightGBM with 63 leaves 0.94701, on the same features.
Deep trees model interactions between many columns at once. When the target is close to a sum of per-column effects
(the original data looks like a logistic formula), those interactions are mostly noise, and the deep model fits it.
Check this on your own data: one run with `num_leaves=4` costs minutes and tells you which regime you are in.

**3. Measure the combiner on rows it did not fit.** The hill climb's own number is computed on the same OOF predictions it picked weights on,
so it always looks a little better than it is. Here both combiners are refitted inside a second 5-fold split.
In v5 they tied (hill climb 0.94899, logistic stacker 0.94894): with members this similar, the choice of combiner does not matter much.


## 4. How this notebook got here: what helped, what did not

Every version was submitted from this notebook. The public LB scores only part of the test rows, so a difference of a few 0.0001 can be noise.

| version | change | why | public LB |
|---|---|---|---|
| v2 | LightGBM + XGBoost + CatBoost + RealMLP, 5 folds, rank blend | a plain, honest baseline: four model families, out-of-fold weights | 0.94720 |
| v3 | target encoding inside each fold, decimal digits, the original data as lookups | trees split on ranges; an encoding gives them each value's fraud rate directly | 0.94841 |
| v4 | GPU, coarser rounded keys, more members, logistic stacker | more views of the same data, and a combiner the s6e8/s6e9 winners used | 0.94865 |
| v5 | 4-leaf / depth-2 trees, every column encoded per value, spline GLM | the target is close to additive, so shallow trees generalise better (lesson 2) | **0.94960** |
| v6 | more shallow views (depth 1, depth 3 CatBoost, raw inputs), TabM | v5's blend kept only shallow members; give it more of what it uses | **0.94982** |
| v8 | prediction cache (members load from the dataset), Optuna-tuned LightGBM, tutorial figures | tuning finds 2-leaf trees: same OOF as the hand-picked 4-leaf member, 62 min | 0.94981 |
| v9 | tuned XGBoost / CatBoost, count + triple-cross features, kNN, seed bags | v8's "what next" list, each judged by what it adds to the stack; the triple-cross members took the largest weights (0.29, 0.21), stacker held-out 0.94917 | 0.95004 |
| v10 | digit keys (cents, truncated integer levels, leading digits), target-encoded in fold, on 4 new members | the generator writes numbers digit by digit (goodpjw2008, ektarr); each digit tree member alone scores OOF 0.9495, above every earlier member, stacker held-out 0.95019 | **0.95095** |
| v11 | text only: history, lessons | the same predictions as v10 | (not submitted) |
| v12 | digit members without the 45 pair encodings, GLM on the encodings | no-pair members 0.94936 vs 0.94951 (the pairs are not the gap); the GLM took the largest stack weight (0.39) and stacker held-out rose to 0.95053, but the board did not move | 0.95093 |
| v13 | goodpjw2008's Part 3 / 5 recipe reproduced exactly (m-estimate encoder on every column and key, raw inputs, 3 and 4 leaves) | find out why our digit members trail the public ones by 0.0009 | members 0.95036 to 0.95052 alone (ours 0.9495); stacker held-out 0.95019 -> 0.95086, board only +0.00005: **0.95100** |
| v14 | the Part 5 member refitted with five encoder seeds, averaged | areebbb: seed-bagging the encoder lifted one member 0.95037 -> 0.95077 | member 0.95052 -> 0.95066, stacker held-out 0.95086 -> 0.95091: **0.95101** |
| v15 | three members whose test predictions come from one fit on all rows: a 5-seed depth-2 XGBoost bag, a joint GLM (one-hots of every value and digit key), a depth-2 XGBoost boosted from that GLM | v10-v14 raised the stacker 0.95019 -> 0.95091 and the board only 0.95095 -> 0.95101; four public notebooks gained on the board from a full-data refit | members 0.95054 / 0.95011 / 0.95042 alone, stacker held-out 0.95091 -> 0.95099 (largest weight: the GLM booster, 0.293): **0.95146** |
| v16 | 12 public prediction banks as stack members (lazyyeou, nilanjansaha123, yekenot; OOF + full-refit test) | offline: our 35 members CV 0.95092, + all banks +0.00020; areebbb's pooled stack and nina2025's h-blend lead the public board this way | stacker held-out 0.95099 -> 0.95113: **0.95176** |
| v17 | four more banks: goodpjw2008 Part 6, megayak, masayakawamata (GPT-2 keys stack), ektarr | offline on v16's 47 members: +0.00097 together (masayakawamata +0.00060 alone); stacker held-out 0.95113 -> 0.95212: **0.95211** |
| v18 | sameershaikh27's GLM booster trained on ten folds (encodings from 90% of the rows) as a bank member | offline on v17's 58 members: +0.00015, the only one of 11 new public banks above +0.00005 | goodpjw2008 renamed Part 6 to Part 7, so its two banks were not mounted: 57 members, stacker held-out 0.95212 -> 0.95225, board down: **0.95195** |
| v19 | v18 with goodpjw2008's two banks mounted again (same files, new kernel name) | separates v18's two changes: was the board drop the new member or the two lost ones? | goodpjw2008 renamed the kernel again (Part 8) before v19 ran: the banks were again not mounted, the same 57 members as v18; not submitted |
| v20 | the two goodpjw2008 banks found by the series prefix `s6e11-part-`, so a rename no longer drops them | the v19 question again: 59 members, as measured offline (stacker held-out 0.95223) | all 59 mounted, stacker held-out 0.95226; board below v17 again: **0.95202** (see "What the board said" in the v18 section) |
| v21 | queyrusi's digit CNN (reads the digit strings, no target encoding) as a bank member; goodpjw2008's kernel source moved to its Part 10 name | offline on v20's 59 members: +0.00010 / +0.00005 on two splits, highest rank correlation with a member 0.9795 | 60 members, stacker held-out 0.95226 -> 0.95235 (CNN weight 0.237); board down again: **0.95190** |
| v22 | the masayakawamata bank read from the author's archive dataset when the notebook output is missing; a count of expected vs loaded banks; megayak's four banks frozen in our dataset; goodpjw2008's kernel source moved to Part 11; a wide pool of 106 more public members, stacker C 0.01 | the source notebook's newest version has no output files: without this, our largest stack weight (0.736) drops out silently; the wide pool is what the 0.9528 public notebooks have and we did not: offline +0.00017 / +0.00013 on v21's 60 members | stacker held-out 0.95235 -> wide stacker 0.95251 (166 members): **0.95239**, the first gain on the board since v17 |
| v23 | megayak's target-first GPT-2 member (out-of-fold, on our folds) in the wide pool; goodpjw2008's kernel source moved to its Part 12 name | goodpjw2008 Part 12: +0.00006 nested CV, public 0.95285 -> 0.95298 on their stack; rank correlation 0.98 with our members | 0.95260 |
| v24 | text only: section 5 (what next) and the credits brought to v23's state | the same predictions as v23 | (not submitted) |
| v25 | TabICL measured as a member (null, section 4); the 0.95300 public recipe (65/35 rank-logit mix of a wide and a narrow stack) measured on our own two stackers' held-out rows, plus two extra files for the board | three public notebooks reached 0.95300 with a mix that lowers the author's own CV; does it hold on our OOF? | held-out wide 0.95257 / 0.95259 / 0.95255, 65/35 mix 0.95255 / 0.95259 / 0.95255 (null); submission.csv unchanged (v23's ranking, 0.95260); board probes: 0.65 our wide + 0.35 Part 9 0.95259, 0.65 our wide + 0.35 our narrow 0.95244 |
| v26 | the 0.95330 public file (Moran Z, Strike 9) measured on the test side; the 0.95300 recipe with our wide stack in Part 9's place as a board file; the noise of a public-board step, measured on held-out rows | the public best rose by ten public-board steps of 0.00001 to 0.00005, chosen by their public score: are they larger than the noise? | `submission.csv` unchanged (0.95260); `submission_p12_ours.csv` (0.65 Part 12 + 0.35 our wide) **0.95305**; noise of a public difference for files 33e-4 apart: 2.8e-4 |
| v27 | the public ladder (Strike 10 0.95340, LM Ladder 0.95346, Three Ladders 0.95347) replayed on held-out rows: steps picked on a public-sized subset, scored on a private-sized one; the 0.95347 file rebuilt and measured on the test side; the ladder's LM step from our own files as two board files | six public-board steps of 0.00001 to 0.00010 moved the public best; what is a step picked on the board worth on rows it was not picked on? | replay: public +2.2e-4, private -1.6e-4 over 100 draws; board: the LM step changed neither file at five decimals, `submission_p12_ours_lm12.csv` **0.95305** (without the step 0.95305), `submission_wide_lm12.csv` 0.95260 (0.95260) |
| v28 | the public 0.95370 file (Aman Atar) taken apart into 16 public models and replayed on out-of-fold rows against our held-out stack; two more language-model seeds (goodpjw2008) in the wide pool; a fold-signature and a test-against-OOF check on every pool member | the public best is now one file with no out-of-fold score: what is in it, and what does its mix score on labelled rows? | the file's mix on labelled rows 0.95186, our held-out stack 0.95257 (-70 points for the file; on the public board +110); the same 16 models with weights fitted on labels 0.95211; the two seeds +1.2 to +2.1 points on three splits, below the +5 bar, `submission.csv` unchanged (0.95260); board: `submission_a370_wide10.csv` **0.95367** (the title), `submission_a370_wide25.csv` 0.95359, `submission_wide_seeds.csv` 0.95260 |
| v29 | the public 0.95388 file (areebbb, one step past goodpjw2008's 0.95386 bold mix) rebuilt byte for byte from its published weights and replayed on labelled rows; the same 87-member move applied to our held-out stack; v28's board results; two v28 numbers corrected | the first public ladder file that reports a gain on out-of-fold rows in all five folds: is it a gain over a label-fitted stack, or over the 0.95370 file? | the 0.95388 mix on labelled rows 0.95246 in-sample, our held-out stack 0.95257 (-11 points; the 0.95370 file was -70); the move on our stack: careful +4.6 (3 of 5 folds), bold -1.3, bolder -6.3 (1 of 5); `submission.csv` unchanged (0.95260); board: `submission_b388_wide10.csv` **0.95386** (the title), `submission_b388_wide50.csv` 0.95359, `submission_wide_bold.csv` 0.95278 |
| v30 | goodpjw2008's ten public-forecast steps from the 0.95388 file to the public 0.95417 file rebuilt from their published weights, each scored on labelled rows; v29's board results; board change against labelled-row change for 15 files | the public best is now a file its author says is tuned on the public rows: what does a public climb look like from the other 80%? | the chain: public +29 points, labelled rows -42 (in-sample); step 7 52.7 points below our held-out stack; public change and labelled change correlate at -0.28 over the ten steps; the CV-led step +1.6 over our stack (public 0.95387); `submission.csv` unchanged (0.95260); board files `submission_s417_wide10.csv`, `submission_cvled_wide10.csv`: see the next version |

**What did not work, or did not help enough to keep**

- **All 276 pairwise crosses.** The usual playground move. On three earlier playgrounds it measured -0.0036, -0.0004 and +0.0002;
  with 24 columns most pairs are noise and each one is a chance to overfit. Here only pairs among the ten strongest columns are encoded.
- **Deep trees as the main model.** 63-leaf LightGBM 0.94701 vs 4-leaf 0.94838 (lesson 2).
- **A model trained only on the original 20k rows** (LightGBM started from its predictions, v4/v5): 0.94459 alone and weight 0 in the blend.
  The original data is used as lookup features instead, which keeps what it knows without its smaller, slightly different distribution.
- **RealMLP**: strong alone (0.94704), weight 0 in the blend (lesson 1). It stays in the notebook as a reference for how to run it.
- **A plain logistic regression and a small extra-trees LightGBM** (v4): replaced in v5 by the spline GLM and the shallow trees, which give the blend the same kind of view (linear, low-interaction) in a stronger form.
- **kNN fraud rate** (v9): the fraud rate of each row's 100 nearest neighbours, on GPU. OOF 0.89930 alone and stacker weight -0.018: trees on encoded columns already know what a neighbourhood knows.
- **Tuning XGBoost and CatBoost with Optuna** (v9): both searches picked the shallowest option (XGBoost depth 1, CatBoost depth 3) and the tuned members took stacker weights 0.001 and 0.082. Tuning confirmed lesson 2 and added almost nothing.
- **Seed bags of the depth-1/2 XGBoost** (v9): OOF 0.94855 / 0.94844 against 0.94837 / 0.94831 single, weights 0.097 / 0.031. A small gain, not worth three times the training.
- **A better stacker that the board does not see** (v12-v14): the stacker's held-out AUC rose 0.95019 -> 0.95091 over three versions (the GLM on the encodings, the Part 3 recipe, the encoder-seed bag), and the public LB moved 0.95095 -> 0.95101.
  The v14 bag alone: member +0.00014 OOF, board +0.00001. More OOF accuracy from the same 80%-of-rows test predictions stopped paying; v15 changes the test side instead.
- **More public banks that the stack already knew** (measured offline on v17's 58 members, logistic stacker C 0.05, one 5-fold split; bar +0.00005):
  goodpjw2008's 5-seed GLM boost +0.00004, its XGBoost GLM boost +0.00002, its residual MLPs (an MLP added to the GLM logit) +0.00003 and +0.00001,
  its joint-GLM bag 0.00000, its EBM bag -0.00003; sameershaikh27's 5-fold, 3-seed GLM boost +0.00002;
  sadamtorres' TabPFN-3.5 (OOF 0.94883, a different family, rank correlation 0.983) -0.00002 and tuned CatBoost +0.00001; takayuki0127's GLM -0.00002.
  Our TabICL member (v24's kernel s6e11-tabicl-member: 4 estimators, 60k-row context, 5 folds; OOF 0.94503, rank correlation 0.975 with its nearest member)
  on v19's 59 members, splits 7 and 11: +0.00002 / -0.00004. A foundation model ranks rows differently and still adds nothing the stack lacks.
  Earlier (2026-11-03): masayakawamata's GPT-2 target-first classifier -0.00010, seven more banks between -0.00010 and +0.00002,
  and GPT-2 / GPT-4 / BERT / T5 token keys -0.00002 to 0.00000. A strong model alone is not enough: the stack has to lack what it knows.
- **The four ideas of v10's "what next" list** (measured offline for v20 on v19's 59 members, logistic stacker C 0.05, splits 7 and 11; bar +0.00005 on both).
  Alone, three of them beat the digit member they extend (0.94951): digit keys + triple crosses in one member 0.94974, digit keys of the integer and count columns 0.94960,
  learning rate 0.01 0.94959; interaction constraints (a tree splits within one source column's group) 0.94934.
  On top of the stack: +0.00001 / -0.00006, +0.00003 / +0.00001, +0.00002 / -0.00003, +0.00003 / -0.00001. Rank correlation with an existing member 0.991-0.998:
  better versions of a member the stack already has add nothing. Code: `tools/whatnext_member.py`, `tools/whatnext_addon.py` in github.com/alexchilton/kaggley (s6e11-cc-fraud).
- **queyrusi's GLM + shallow-tree blend** (measured offline for v21 on v20's 59 members, splits 7 and 11): OOF 0.95107 alone, rank correlation 0.9938 with a member already in the stack, +0.00001 / +0.00002.
  The same author's digit CNN, weaker alone (0.94974) and less correlated (0.9795), passed the bar and is in v21.
- **A better stacker that the board does not see, again (v18-v21).** Every version since v17 raised our stacker's held-out AUC and lowered the public LB:
  0.95212 / **0.95211** (v17), 0.95225 / 0.95195 (v18), 0.95226 / 0.95202 (v20), 0.95235 / 0.95190 (v21). Two other authors measured the same thing on their own stacks.
  [SIDHAARTH SHREE](https://www.kaggle.com/code/sidhaarthshree/s6e11-100-self-trained-stack-lb-0-95154) (section 6), every model self-trained: a plain 7-member stack CV 0.95084, **LB 0.95154**;
  adding GLM-boost and residual-MLP members (anything that starts from a GLM's logit) CV 0.95133-0.95136, LB 0.95141; GLM-boost alone CV 0.95128, LB 0.95097.
  [Sameer Shaikh](https://www.kaggle.com/code/sameershaikh27/s6e11-fraud-pooled-stack-of-public-oof-banks): 89 -> 93 pooled members, nested CV +0.00004, LB 0.95191 -> 0.95189.
  The public score covers 20% of the test rows and moves about ±0.0003 between random draws, so each pair is noise-sized; what is not noise-sized is that three independent stacks moved the same way.
  HYPOTHESIS (unverified): the GLM-started members, which hold large stack weights (in v21: ten-fold GLM boost 0.421, goodpjw2008 and megayak GLM boost 0.234 each, megayak GLM boost XGBoost 0.180),
  fit something in the 700k training rows that the public 20% does not reward. The private board will say which.
- **Blending with a public stack's file** (measured offline for v22, splits 7 and 11): a rank blend of 0.7 goodpjw2008's Part 11 wide stack (nested OOF 0.95250) and 0.3 our wide stack
  0.95252 / 0.95251 against our wide stack alone 0.95250 / 0.95245, +0.00002 / +0.00006. goodpjw2008's wide stack as one more member of v21's 60: 0.95241 / 0.95240, below the wide pool's 0.95250 / 0.95245.
  A blend of two stacks built from the same public members adds little out of fold; its public-board gain (+0.00018 for goodpjw2008) is the size of the public noise.
- **The 0.95300 two-stack recipe on our own stacks** (v25, held-out splits 7 / 8 / 9): wide stack alone 0.95257 / 0.95259 / 0.95255, 0.65 wide + 0.35 our 60-member stack 0.95255 / 0.95259 / 0.95255.
  On the board: 0.65 our wide + 0.35 goodpjw2008 Part 9 0.95259, 0.65 our wide + 0.35 our 60-member stack 0.95244, against 0.95260 for the wide stack alone.
  The narrow stack holds no member the wide stack lacks, so a mix can only reweight what the wide stacker already fitted; on held-out rows and on the board it did not help.
- **A ladder step from a fitted stack** (v27, all 700k held-out rows of the wide stacker, split 7): + 0.12 (megayak LM - stack) -0.51e-4, + 0.12 (masayakawamata GPT-2 - stack) -0.75e-4,
  a step of 0.12 away from the MLP bag -0.63e-4, away from the GB bag -0.55e-4. Six such steps picked on a public-sized subset: +2.2e-4 there, -1.6e-4 on the private-sized rest (100 draws, 8% positive).
  The stacker already holds each member at the weight the labelled rows support; a step picked on 60k rows moves it off that weight.
  On the board (v27, 2026-11-12) the LM step changed neither of our files at five decimals: 0.95305 with and without it, 0.95260 with and without it.
- **A second seed of each language model, as two more pool members** (v28, held-out splits 7 / 8 / 9): +2.1 / +1.2 / +1.9 points (1 point = 0.00001), every split up, below the +5 bar for the submitted stack.
  Seed 7 of megayak's LM alone +0.6, seed 2026 of masayakawamata's GPT-2 alone +1.5 (split 7). Real and small: on a public-sized subset the same change is +1.3 ± 7.6 points.
- **The best public file's weights, on labelled rows** (v28): the 0.95370 file's mix of 16 public models scores 0.95186 out of fold; the same 16 models with weights fitted on labels 0.95211; our wide stack 0.95257.
  We do not submit the file as ours. Two mixes of it with our stack are written for the board, because the board score of a mix next to its labelled-row score is the measurement.
  On the board (2026-11-13): 90% file + 10% our stack 0.95367, 75 / 25 0.95359; on labelled rows the same two mixes are +13 and +31 points above the file. The two referees rank them in opposite order.
- **The public ladder's move on our own stack** (v29, held-out split 7; the move's weights were chosen on these out-of-fold rows, so a gain would be optimistic): goodpjw2008's careful mix +4.6 points (3 of 5 folds), balanced +2.3 (3 of 5), bold -1.3 (1 of 5), areebbb's bolder -6.3 (1 of 5), two steps past bold -34.6.
  On the replayed 0.95370 file the same moves gain +19 to +41 points. The move corrects that file's weights towards what labels choose (megayak's shallow trees from -0.19 back up, sadamtorres' XGBoost down); a stack fitted on labels has no such error to correct.
  On the board (2026-11-15) the bold move on our stack scored 0.95278, +18 points over our stack alone, while labelled rows say -1.5. 10% and 50% of our stack in the 0.95388 file: -2 and -29 on the board, +5 and +17 on labelled rows.
- **Climbing the public board with a forecast of the public score** (v30, goodpjw2008's ten steps from the 0.95388 file to the 0.95417 file, replayed on labelled rows, in-sample): public +29 points, labelled rows -42; the 0.95417 file is 53 points below our held-out stack.
  On 100 public-sized random subsets of labelled rows the same change is -44 ± 22 points and never reaches +29. Public and labelled change correlate at -0.28 over the ten steps. We do not use this method for `submission.csv`.
- **A stronger stacker penalty alone** (v22, splits 7 and 11): C 0.01 instead of 0.05 on v21's 60 members, 0.95231 / 0.95230 against 0.95233 / 0.95232. It only helps once the pool is wide.
- **A bug worth knowing about.** The first nested hill climb summed fold weights without normalising them, so each fold's predictions
  were on a different scale and the pooled AUC read 0.895. When an honest re-score is far below the in-sample one, suspect the code before the model.


In [ ]:

hist = pd.DataFrame({'version': ['v2', 'v3', 'v4', 'v5', 'v6', 'v8', 'v9', 'v10', 'v12', 'v13', 'v14', 'v15', 'v16', 'v17', 'v18', 'v20', 'v21', 'v22', 'v23', 'v25', 'v26', 'v27', 'v28', 'v29'], 'LB': [0.94720, 0.94841, 0.94865, 0.94960, 0.94982, 0.94981, 0.95004, 0.95095, 0.95093, 0.95100, 0.95101, 0.95146, 0.95176, 0.95211, 0.95195, 0.95202, 0.95190, 0.95239, 0.95260, 0.95259, 0.95305, 0.95305, 0.95367, 0.95386],
                     'change': ['4-family baseline', 'in-fold target encoding', 'more members, stacker', 'shallow trees', 'more shallow views, TabM', 'cache, tuned LightGBM', 'triple crosses', 'digit keys', 'no-pair digits, GLM', 'Part 3 recipe', '5 encoder seeds', 'full refit, joint GLM', 'public banks', 'four more banks', 'ten-fold bank', 'banks re-mounted', 'digit CNN', 'wide pool', 'second LM', 'mix + Part 9 (probe)', 'Part 12 + ours (probe)', '+ LM step (probe)', '0.95370 file + ours (probe)', '0.95388 file + ours (probe)']})
fig, ax = plt.subplots(figsize=(8, 3.2)); ax.plot(hist['version'], hist['LB'], 'o-', color='#4a6fa5')
for _, r in hist.iterrows(): ax.annotate(r['change'], (r['version'], r['LB']), textcoords='offset points', xytext=(0, 8), ha='center', fontsize=8)
ax.set_ylabel('public LB AUC'); ax.set_title('Public LB per version of this notebook'); ax.set_ylim(0.9465, 0.9545); plt.show()


In [ ]:
# Final submission guard: submission.csv must be present, aligned to test['id'], finite, and non-constant.
def _valid_submission(path):
    try:
        s = pd.read_csv(path)
        if 'id' not in s.columns or TARGET not in s.columns or len(s) != len(test):
            return False
        v = s.set_index('id')[TARGET].reindex(test['id'].to_numpy()).to_numpy(dtype=float)
        return (np.isfinite(v).all() and np.unique(v).size > 1 and not s['id'].duplicated().any())
    except Exception:
        return False

if not _valid_submission(OUT / 'submission.csv'):
    fallback = None
    for v in [globals().get('t_w'), globals().get('blend')] + list(pred.values()):
        v = np.asarray(v, dtype=float).ravel()
        if len(v) == len(XT) and np.isfinite(v).all() and np.unique(v).size > 1:
            fallback = v; break
    if fallback is None:
        fallback = rankdata(np.arange(len(XT), dtype=float))
    pd.DataFrame({'id': test['id'], TARGET: rankdata(fallback) / len(fallback)}).to_csv(OUT / 'submission.csv', index=False)
    print('submission.csv was missing or invalid; rewrote it from the best available test vector')
else:
    print('submission.csv validated:', len(test), 'rows, finite, non-constant')

subs = sorted(OUT.glob('submission*.csv'))
bad = [p.name for p in subs if not _valid_submission(p)]
print(f'submission files: {len(subs)} written, {len(bad)} invalid -> {bad if bad else "none"}')

## 5. What next

v10's digit keys moved the leaderboard more than any change since v5 (+0.0009). The lesson: in a synthetic playground, ask how the generator wrote the numbers, not only what they mean.

Status at v23 (public 0.95260); TabICL measured for v25, a null (section 4); the 0.95300 two-stack recipe measured for v25, a null on held-out rows and on the board (section 3, v25); the 0.95330 file and the noise of a public-board step measured for v26 (section 3, v26); the public ladder replayed on held-out rows for v27 (section 3, v27); the 0.95370 file taken apart, two more language-model seeds and the pool checks for v28 (section 3, v28); the 0.95388 file rebuilt and its move applied to our stack for v29 (section 3, v29: the ladder's move corrects the 0.95370 file towards label-fitted weights and does not help a label-fitted stack); the 0.95417 file's ten steps scored on labelled rows for v30 (section 3, v30: +29 on the board, -42 on labelled rows). Our own tree members are saturated: the four ideas this list held since v10 were all measured for v20, each made a slightly better member and none moved the stack.
The gains since then came from members of another family: v22's wide pool of public members (its GPT-2 member took the largest weight) and v23's target-first GPT-2 (megayak). That is the rule this list now follows: a new member must rank rows differently, below ~0.99 rank correlation with every member, or it will not add.
- **Our own target-first GPT-2 seed on our folds**: a second seed added +0.00014 on fold 0 for goodpjw2008. v28 measured goodpjw2008's two published seeds in our pool: +0.00002 on held-out rows. More seeds of the same two models will add less each time; a third kind of language model (another architecture or row format) is the open question.
- **Forensic and density features**: how the generator wrote each row, as in S6E8.
- **The test side**: full-data refits for our strongest members, as v15 did for three of them.

Companion notebook: [Four months, same screens: what held](https://www.kaggle.com/code/alexchilton/four-months-same-screens-what-held-s6e11), how to measure the noise floor an idea must beat, and how the same feature screens changed verdict over four playgrounds.

## Credits

- Competition and synthetic data: Kaggle Playground Series S6E11; original data by Udit Jain.
- RealMLP: D. Holzmüller, L. Grinsztajn, I. Steinwart, *Better by Default: Strong Pre-Tuned MLPs and Boosted Trees on Tabular Data* (NeurIPS 2024), pytabkit.
- TabM: Y. Gorishniy, A. Kotelnikov, A. Babenko, *TabM: Advancing Tabular Deep Learning with Parameter-Efficient Ensembling* (ICLR 2025), via pytabkit.
- Target-first language-model members: C. Deotte's GPT-2 recipe ([kaggle-playground-solutions](https://github.com/cdeotte/kaggle-playground-solutions), Apache 2.0), as trained and published by
  [megayak](https://www.kaggle.com/datasets/megayak/s6e11-target-first-lm) and [masayakawamata](https://www.kaggle.com/datasets/masayakawamata/s6e11-gpt2-target-first-oof).
- Public members and prediction banks in the stack and the wide pool (each listed in the notebook's sources): goodpjw2008 (OOF bank, wide stack Part 12),
  queyrusi (digit CNN), sadamtorres (TabPFN and GBDT pairs), sameershaikh27 (ten-fold GLM boost, digit-sequence CNN), masayakawamata (public stack, tabpack),
  amanatar (Strike 10 file, the 0.95370 file and its bundle of 16 models), Moran Z (Strike 9 file), areebbb (the ladder recipes and their probes),
  goodpjw2008 (seed 7 of the compact LM, seed 2026 of the GPT-2, the replay of a file's weights on out-of-fold rows, the public-forecast mixes behind the 0.95386 file), areebbb (the 0.95388 bolder step), goodpjw2008 (the ten public-forecast steps behind the 0.95417 file), kagankoral (the final-pick check), dariushafshar (the pool checks), asteriospanoras (the noise of a public gap),
  malyshevdanil (pooled stack), wguesdon (tree ensembles), lazyyeou (digit-trie target encoding), nilanjansaha123 (digit trie with GPT-2), ektarr (RealMLP stacking).
- Ideas taken from public S6E11/S6E10 notebooks, measured here: shallow trees, per-value encoding and the spline GLM from
  [sidhaarthshree](https://www.kaggle.com/code/sidhaarthshree/s6e11-fraud-per-value-te-shallow-trees-stack) and
  [goodpjw2008](https://www.kaggle.com/code/goodpjw2008/s6e11-part-2-two-view-tree-blend-lb-0-94945); RealMLP settings from
  [yekenot](https://www.kaggle.com/code/yekenot/ps-s6-e10-realmlp-pytabkit) via [ravi20076](https://www.kaggle.com/code/ravi20076/playgrounds6e11-public-baseline-v1).
